# Tahap 4B — Sensitivitas pasca-hasil atas evaluasi final (K-6, K-7)

Notebook ini **tidak mengganti** hasil Tahap 4 (protokol terkunci 16 Okt 2026). Semua angka 4B dilaporkan sebagai analisis sensitivitas yang ditentukan **setelah** hasil Tahap 4 dilihat.

Sistem yang dievaluasi identik dengan Tahap 4: M0 `last.pt` beku, gate post-hoc tanpa z (960 fitur, LogReg C = 0,001, target `soft_small`), B2 (N = 200, ρ = 0,5), satu engine TensorRT FP16 dengan ONNX `If`. Kodenya `tahap4b.py`, dibangkitkan dari `tahap4_final.py` oleh `buat_tahap4b.py` dengan patch teks kecil; selain dua tambahan di bawah, isinya sama.

**K-6 — latensi ujung-ke-ujung.** Di Tahap 4, latensi router dirakit: aktivasi × T(rute A paksa) + (1 − aktivasi) × T(rute B paksa) + overhead host B2 yang diukur terpisah. Di sini latensi diukur langsung per gambar: host menghitung threshold B2 dari 200 skor sebelumnya, engine A3 berjalan dengan threshold itu, lalu y, keputusan dan skor disalin ke host. Seluruh test diproses dalam urutan aliran yang sama dengan evaluasi akurasi, sehingga keputusan identik dengan Tahap 4 (diperiksa: `e2e_parity`). Vanilla, P2-Static, rute A paksa dan rute B paksa diukur di pass yang sama, diselang-seling per gambar. H6 dihitung ulang dengan fraksi campuran dari latensi ujung-ke-ujung (`H6_e2e_status`).

**K-7 — baseline `last.pt`.** Di Tahap 4, Vanilla dan P2-Static memakai `best.pt` (epoch terbaik di val), sedangkan M0 memakai `last.pt`. Jalankan notebook ini dua kali: `BASELINE_CKPT = "best.pt"` (memisahkan efek K-6 saja) dan `BASELINE_CKPT = "last.pt"` (K-6 + K-7). Keluaran masuk ke folder berbeda (`/kaggle/working/tahap4b_best`, `/kaggle/working/tahap4b_last`).

**Perkiraan waktu (T4):** sama dengan Tahap 4 ditambah ±1–2 menit per build untuk pass ujung-ke-ujung, yaitu ±25–30 menit per model per pilihan checkpoint.

**Yang dikirim balik:** `tahap4b_ringkasan_<best|last>.zip` dan `tahap4b_<model>.zip` per model.


In [ ]:
# [0] YAML jembatan VisDrone (path asli dataset di Kaggle Input)
base_input_path = '/kaggle/input/datasets/agungmaugi/visdrone/VisDrone/kaggle/working/datasets/VisDrone_1'
yaml_content = f"""path: {base_input_path}
train: images/train
val: images/val
test: images/test

# Classes
names:
  0: Person
  1: Vehicle
"""
with open('/kaggle/working/visdrone.yaml', 'w') as f:
    f.write(yaml_content)
print("✅ /kaggle/working/visdrone.yaml")

In [ ]:
# [1] KONFIGURASI
import os, sys, json, time, shutil, subprocess, hashlib, platform, datetime
from pathlib import Path

KITTI_YAML = "/kaggle/input/datasets/agungmaugi/kitti-2class/kitti_2class.yaml"
VIS_YAML = "/kaggle/working/visdrone.yaml"
BASELINE_CKPT = "last.pt"   # <- [4B] "best.pt" (hanya K-6) | "last.pt" (K-6 + K-7)
KITTI_VAN = f"/kaggle/input/models/agungmaugi/yolov8n-kitti/pytorch/default/1/Skenario1_YOLOv8n_Standar/weights/{BASELINE_CKPT}"
KITTI_P2S = f"/kaggle/input/models/agungmaugi/yolov8n-p2-statis-kitti/pytorch/default/1/Skenario2_YOLOv8n_P2_Statis/weights/{BASELINE_CKPT}"
VIS_VAN = (f"/kaggle/input/models/agungmaugi/yolov8n-visdrone/pytorch/default/1/content/thesis-Difficulty-Aware-Router/"
           f"Tesis_VisDrone/Skenario1_YOLOv8n_Standar/weights/{BASELINE_CKPT}")      # path = Tahap 2c VisDrone
VIS_P2S = f"/kaggle/input/models/agungmaugi/yolov8n-p2-visdrone/pytorch/default/1/Skenario2_YOLOv8n_P2_Statis/weights/{BASELINE_CKPT}"

RUN_MODEL = "kitti_s42"     # <- kitti_s42 | kitti_s7 | kitti_s2026 | visdrone_s42 | None (hanya ringkasan) | list, mis.
                           #    ["kitti_s42", "kitti_s7"] dengan reuse_dir berisi *_intermediate s42 (s42 hanya dianalisis ulang)
MERGE_GLOB = None          # mis. "/kaggle/input/hasil-tahap4b/**/tahap4b_*.zip" (zip hasil, bukan _intermediate) untuk ringkasan gabungan
CFG = dict(
    repo_url="https://github.com/maugi1547/thesis-Difficulty-Aware-Router.git",
    repo_dir="/kaggle/working/thesis-Difficulty-Aware-Router",
    code_commit="5416543",
    model_yaml="ultralytics/cfg/models/v8/yolov8n-p2-router.yaml",
    models=[
        dict(name="kitti_s42", data_yaml=KITTI_YAML, seed=42, archive="/kaggle/input/datasets/arungmaugi/archive-m0-lam002-seed42",
             max_det=300, coco_max_dets=[1, 10, 100], vanilla_ckpt=KITTI_VAN, p2static_ckpt=KITTI_P2S),
        dict(name="kitti_s7", data_yaml=KITTI_YAML, seed=7, archive="/kaggle/input/datasets/agungmaugi/archive-m0-lam002-seed7",
             max_det=300, coco_max_dets=[1, 10, 100], vanilla_ckpt=KITTI_VAN, p2static_ckpt=KITTI_P2S),
        dict(name="kitti_s2026", data_yaml=KITTI_YAML, seed=2026, archive="/kaggle/input/datasets/arungmaugi/archive-m0-lam002-seed2026",
             max_det=300, coco_max_dets=[1, 10, 100], vanilla_ckpt=KITTI_VAN, p2static_ckpt=KITTI_P2S),
        dict(name="visdrone_s42", data_yaml=VIS_YAML, seed=42, archive="/kaggle/input/datasets/agungmaugi/archive-m0-lam002-visdrone-seed42",
             max_det=500, coco_max_dets=[1, 10, 500], vanilla_ckpt=VIS_VAN, p2static_ckpt=VIS_P2S),
    ],
    ckpt_name="last.pt", out_dir=f"/kaggle/working/tahap4b_{BASELINE_CKPT[:-3]}", baseline_ckpt=BASELINE_CKPT,
    imgsz=640, conf=0.001, iou=0.7, rho=0.5, gate_C=0.001,      # terkunci (= 2c / 3a / D2)
    b2_window=200, b2_min_hist=20, b2_orders=5,                 # B2 terkunci (3a): N = 200, ρ = 0.5
    h5_ms=0.15, h7_dev=0.05,                                    # kriteria (rencana 13 Okt)
    n_boot=1000, n_draws=5, n_draws_sekunder=3, boot_seed=20261016, boot_jobs=4,   # boot_seed SAMA untuk semua model (pooling KITTI)
    cascade=True, fp32_check=True,
    precision="fp16", backend="trt", n_builds=2, n_prof_images=300, n_warmup=30, n_reps=3, job_workers=2,
    n_onnx_check=4, trt_timeout_s=10800, sh_timeout_s=1800, n_train=None, run_trt=True,
    reuse_dir=None,          # mis. "/kaggle/input/tahap4-intermediate": dicari **/<model>/stageA.pkl; tahap A / job yang ada dilewati
    zip_intermediate=True,
)
CFG.update(globals().get("CFG_OVERRIDE", {}))
assert CFG["repo_dir"] == "/kaggle/working/thesis-Difficulty-Aware-Router" or "CFG_OVERRIDE" in globals()
ALL_MODELS = CFG["models"]
if "CFG_OVERRIDE" not in globals():
    _run = list(RUN_MODEL) if isinstance(RUN_MODEL, (list, tuple)) else ([RUN_MODEL] if RUN_MODEL else [])
    assert all(r in [m["name"] for m in ALL_MODELS] for r in _run), f"RUN_MODEL tidak dikenal: {RUN_MODEL}"
    CFG["models"] = [m for m in ALL_MODELS if m["name"] in _run]
REPO = Path(CFG["repo_dir"]); OUT = Path(CFG["out_dir"]); OUT.mkdir(parents=True, exist_ok=True)
PROV = {"started_utc": datetime.datetime.utcnow().isoformat(timespec="seconds"), "cfg": CFG}
T_START = time.perf_counter()

def stamp(*a):
    print(f"[{datetime.datetime.now().strftime('%H:%M:%S')} | {(time.perf_counter() - T_START) / 60:6.1f} mnt]", *a, flush=True)

def sh(cmd, cwd=None, secret=None, timeout=None):
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0", PIP_NO_INPUT="1")   # tidak pernah menunggu input -> gagal cepat
    shown = " ".join(c if c != secret else "***" for c in map(str, cmd))
    shown = shown.replace(secret, "***") if secret else shown
    stamp("$", shown[:160])
    try:
        r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, env=env, stdin=subprocess.DEVNULL,
                           timeout=timeout or CFG["sh_timeout_s"])
    except subprocess.TimeoutExpired:
        raise RuntimeError(f"TIMEOUT {timeout or CFG['sh_timeout_s']} s: {shown}")
    out = (r.stdout + r.stderr).strip()
    if secret:
        out = out.replace(secret, "***")
    if r.returncode != 0:
        raise RuntimeError(f"{shown}\n{out}")
    return out

print(json.dumps(CFG, indent=1, default=str))
stamp("RUN_MODEL =", RUN_MODEL if "CFG_OVERRIDE" not in globals() else "(override)", "| GPU:", shutil.which("nvidia-smi") is not None)

In [ ]:
# [2] CLONE dari GitHub + checkout commit terkunci (clone baru setiap sesi -> tidak ada sisa state lama)
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass                                              # repo publik: token tidak perlu
url = CFG["repo_url"]
auth_url = url.replace("https://", f"https://x-access-token:{token}@") if token else url

if REPO.exists():
    shutil.rmtree(REPO)
sh(["git", "clone", "--quiet", auth_url, str(REPO)], secret=token)
sh(["git", "remote", "set-url", "origin", url], cwd=REPO)          # token tidak tersimpan di .git/config
try:
    sh(["git", "checkout", "--quiet", CFG["code_commit"]], cwd=REPO)
except RuntimeError as e:
    raise RuntimeError(f"Commit {CFG['code_commit']} tidak ada di GitHub. Sudah di-push? "
                       f"HEAD remote = {sh(['git', 'log', '-1', '--format=%h %s', 'origin/HEAD'], cwd=REPO)}") from e
PROV["git_commit"] = sh(["git", "rev-parse", "HEAD"], cwd=REPO)
PROV["git_commit_msg"] = sh(["git", "log", "-1", "--format=%s (%cd)"], cwd=REPO)
PROV["git_branches_containing"] = sh(["git", "branch", "-r", "--contains", "HEAD"], cwd=REPO).split()
print("✅ commit:", PROV["git_commit"][:10], "|", PROV["git_commit_msg"], "| branch:", PROV["git_branches_containing"],
      "| token:", "ya" if token else "tidak")

In [ ]:
# [3] INSTALL editable dari repo (ultralytics pip bawaan dihapus agar tidak tertukar)
print(sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "ultralytics"]) or "uninstall: ok")
print(sh([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])[-500:] or "install: ok")

In [ ]:
# [3b] Paket evaluasi
print(sh([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxscript", "onnxruntime", "pycocotools", "scikit-learn"]
         + (["tensorrt", "pycuda"] if CFG["run_trt"] else []))[-500:] or "install: ok")

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/router_train_utils.py
# [4] versi sama dengan Tahap 1 (hapus sel ini setelah file di-commit ke repo)
"""
router_train_utils.py — utilitas murni untuk training & sanity check Router-P2 Dual-Branch.

Dipakai bersama oleh script_training.py (run utama) dan notebook smoke test, supaya yang
diuji di Tahap 0 adalah kode yang sama dengan yang dipakai run 100 epoch.
Isi: scheduler berparameter, penerap ROUTER_CFG, logger CSV router (per-run, 11 kolom,
label epoch diperbaiki), logger loss + penjaga NaN, timer epoch, pemeriksa CSV/loss,
estimasi anggaran, uji kalibrasi tersimpan, uji konsistensi gate.
"""
import os, sys, csv, json, time, shutil
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import torch

CSV_COLS = ["Epoch", "Batch", "Lambda", "P2_Prob_Real", "Rel", "Diff_W", "Final_L3",
            "GV_Target", "GV_Loss", "GV_Weight", "GV_Offset"]
LOSS_NAMES = ["box_A", "cls_A", "dfl_A", "router", "proxy", "box_B", "cls_B", "dfl_B"]


def find_router(model):
    """Kembalikan modul router pertama (nama kelas mengandung 'DifficultyAwareRouter')."""
    for m in model.modules():
        if "DifficultyAwareRouter" in m.__class__.__name__:
            return m
    raise RuntimeError("Router tidak ditemukan di model.")


def _ramp(epoch, start, length, final):
    """0 sampai epoch == start, lalu naik linear ke `final` selama `length` epoch, lalu tetap."""
    if epoch <= start:
        return 0.0
    return final * min((epoch - start) / max(length, 1), 1.0)


def make_curriculum_scheduler(router_warmup_end, gv_start, gv_ramp, final_gv,
                              lam_start, lam_ramp, final_lambda, history=None):
    """
    Versi berparameter dari ideal_curriculum_scheduler (script_training.py).

    Konfigurasi run utama yang IDENTIK dengan kode lama:
        make_curriculum_scheduler(router_warmup_end=10, gv_start=10, gv_ramp=20, final_gv=0.5,
                                  lam_start=50, lam_ramp=30, final_lambda=<nilai Anda>)
    Perbedaan perilaku dgn versi lama: _is_warmup di-set dua arah (lama hanya mematikan).
    Untuk run baru hasilnya sama; untuk resume, versi ini lebih benar.

    history: dict opsional, diisi {epoch: {"lambda", "gv", "warmup"}} untuk diverifikasi.
    """
    def scheduler(trainer):
        epoch = trainer.epoch + 1
        lam = _ramp(epoch, lam_start, lam_ramp, final_lambda)
        gv = _ramp(epoch, gv_start, gv_ramp, final_gv)
        warm = epoch <= router_warmup_end
        targets = [trainer.model]
        if getattr(trainer, "ema", None):
            targets.append(trainer.ema.ema)
        for mdl in targets:
            mdl.router_penalty_lambda = lam
            mdl.gate_value_loss_weight = gv
            mdl.total_batches = len(trainer.train_loader)
            find_router(mdl)._is_warmup = warm
        if history is not None:
            history[epoch] = {"lambda": lam, "gv": gv, "warmup": warm}
        print(f"📈 [SCHEDULER] epoch {epoch} | warmup={warm} | lambda={lam:.4f} | gate_value_w={gv:.4f}")
    return scheduler


def make_router_cfg_applier(router_cfg, dump_path=None):
    """
    Callback on_pretrain_routine_end: tulis hyperparameter router secara EKSPLISIT ke model
    (bukan mengandalkan default getattr yang tersebar di loss.py & model_yolov8_router.py).
    Harus jalan sebelum batch pertama, karena DualBranchDetectionLoss membaca
    branch_b_loss_weight & gate_offset_momentum saat criterion dibuat (lazy, batch pertama).
    """
    def apply(trainer):
        for mdl in [trainer.model] + ([trainer.ema.ema] if getattr(trainer, "ema", None) else []):
            for k, v in router_cfg.items():
                setattr(mdl, k, v)
        if dump_path is not None:
            Path(dump_path).write_text(json.dumps(router_cfg, indent=1))
        print("🔧 [ROUTER CFG]", router_cfg)
    return apply


def make_router_csv_logger(csv_path, allow_existing=False):
    """
    Pengganti on_train_epoch_end di script_training.py:
      - path per-run (bukan /kaggle/working/... global yang di-append lintas run),
      - menolak file yang sudah ada (cegah data run lama tercampur),
      - memverifikasi tiap baris berisi 11 kolom,
      - kolom Epoch diganti epoch ASLI (label di loss.py = debug_counter//total_batches + 4,
        salah setiap kali warmup tidak berakhir di epoch 4). Tetap 11 kolom.
    """
    csv_path = Path(csv_path)
    if csv_path.exists() and not allow_existing:      # allow_existing=True hanya untuk resume run yang sama
        raise FileExistsError(f"{csv_path} sudah ada — hapus/ganti nama dulu.")

    def on_train_epoch_end(trainer):
        buf = getattr(trainer.model, "router_buffer", None)
        if not buf:
            return
        epoch = trainer.epoch + 1
        new = not csv_path.exists()
        with open(csv_path, "a", newline="") as f:
            w = csv.writer(f)
            if new:
                w.writerow(CSV_COLS)
            for row in buf:
                if len(row) != len(CSV_COLS):
                    raise ValueError(f"router_buffer berisi {len(row)} kolom, header {len(CSV_COLS)}: {row}")
                w.writerow([epoch] + list(row[1:]))
        buf.clear()
    return on_train_epoch_end


def make_loss_items_logger(store, csv_path=None):
    """
    on_train_batch_end: simpan 8 komponen loss per batch + GAGAL CEPAT bila NaN/Inf.
    (Ultralytics 8.3.240 punya NaN recovery yang diam-diam memuat ulang checkpoint.)
    Bila csv_path diberikan, `store` ditulis ke file & dikosongkan lewat .flush(trainer)
    (dipasang di on_train_epoch_end) supaya run 100 epoch tidak menumpuk di RAM.
    """
    state = {"header": csv_path is not None and not Path(csv_path).exists()}

    def on_train_batch_end(trainer):
        li = trainer.loss_items.detach().float().cpu()
        epoch = trainer.epoch + 1
        store.append([epoch] + li.tolist())
        if not torch.isfinite(li).all() or not torch.isfinite(trainer.loss.detach()).all():
            raise FloatingPointError(f"Loss non-finite di epoch {epoch}, batch #{len(store)}: "
                                     f"{dict(zip(LOSS_NAMES, li.tolist()))}")

    def flush(trainer):
        if csv_path is None or not store:
            return
        pd.DataFrame(store, columns=["epoch"] + LOSS_NAMES).to_csv(
            csv_path, mode="a", header=state["header"], index=False)
        state["header"] = False
        store.clear()

    on_train_batch_end.flush = flush
    return on_train_batch_end


class EpochTimer:
    """
    Ukur waktu per epoch, dipisah: train (loop batch), val_total (validasi + save + callback),
    dan valB (validasi Branch B, diukur lewat wrapper `timed`).
    cuda.synchronize() supaya kernel async tidak 'bocor' ke fase berikutnya.
    """
    def __init__(self):
        self.rows, self._t, self._valB = [], {}, 0.0

    @staticmethod
    def now():
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        return time.perf_counter()

    def on_train_epoch_start(self, trainer):
        self._t["t0"] = self.now()
        self._valB = 0.0

    def on_train_epoch_end(self, trainer):
        self._t["t1"] = self.now()

    def on_fit_epoch_end(self, trainer):
        # final_eval() Ultralytics memanggil on_fit_epoch_end SEKALI LAGI setelah loop
        # (validasi best.pt) -> abaikan panggilan yang tidak didahului on_train_epoch_end.
        if self._t.get("t1") is None:
            return
        t2 = self.now()
        self.rows.append(dict(epoch=trainer.epoch + 1,
                              n_batches=len(trainer.train_loader),
                              train_s=self._t["t1"] - self._t["t0"],
                              val_total_s=t2 - self._t["t1"],
                              valB_s=self._valB))
        self._t["t1"] = None

    def timed(self, cb):
        def wrapper(validator):
            t0 = self.now()
            cb(validator)
            self._valB += self.now() - t0
        return wrapper


# ---------------------------- pemeriksa ----------------------------
class Report:
    def __init__(self):
        self.items = []

    def add(self, name, status, detail=""):
        icon = {"PASS": "✅", "WARN": "⚠️ ", "FAIL": "❌"}[status]
        print(f"{icon} [{status}] {name}" + (f" — {detail}" if detail else ""))
        self.items.append({"check": name, "status": status, "detail": str(detail)})

    def check(self, name, ok, detail="", warn_only=False):
        self.add(name, "PASS" if ok else ("WARN" if warn_only else "FAIL"), detail)

    @property
    def n_fail(self):
        return sum(i["status"] == "FAIL" for i in self.items)


def check_router_csv(df, n_batches, epochs, router_warmup_end, sched_history,
                     offset_clip=1.0, report=None):
    """
    Semua pemeriksaan pada router_loss_components.csv. `df` dibaca dgn pd.read_csv.

    Artefak yang DIKETAHUI dan diperhitungkan:
      - Sejak AUDIT-FIX #6 kolom GV_* berasal dari step yang SAMA (tidak lagi tertinggal 1 step),
        jadi pemeriksaan di sini ketat untuk semua baris. Dengan kode lama, baris pertama
        pasca-warmup akan gagal di cek GV_Weight (nilai warmup = 0).
      - GV_Offset adalah offset yang DIPAKAI step ini (nilai EMA sebelum di-update). Baris
        pertama pasca-warmup = 0 (EMA belum terinisialisasi), ini normal.
      - GV_Offset dicetak 4 desimal -> perubahan EMA < 5e-5 tampak 'tidak berubah'.
      - Mode 'small': batch tanpa gambar valid (tidak ada objek kecil di KEDUA branch)
        -> GV_Loss = GV_Target = 0 dan offset tidak di-update.
    """
    r = report or Report()
    r.check("CSV: header 11 kolom dgn urutan benar", list(df.columns) == CSV_COLS, list(df.columns))
    if list(df.columns) != CSV_COLS:
        return r

    exp_rows = (epochs - router_warmup_end) * n_batches
    r.check("CSV: jumlah baris = epoch pasca-warmup × batch/epoch", len(df) == exp_rows,
            f"{len(df)} vs {exp_rows}")
    r.check("CSV: tidak ada baris pada epoch warmup", int((df.Epoch <= router_warmup_end).sum()) == 0)

    num = df[CSV_COLS[2:]].astype(float)
    r.check("CSV: semua nilai finite (tidak NaN/Inf)", bool(np.isfinite(num.values).all()))

    # --- jadwal tercatat sesuai scheduler ---
    for e, g in df.groupby("Epoch"):
        h = sched_history.get(int(e))
        if h is None:
            r.add(f"E{e}: riwayat scheduler", "FAIL", "epoch tidak ada di riwayat scheduler")
            continue
        r.check(f"E{e}: kolom Lambda = jadwal ({h['lambda']:.4f})",
                np.allclose(g.Lambda.astype(float), h["lambda"], atol=1e-4))
        r.check(f"E{e}: kolom GV_Weight = jadwal ({h['gv']:.4f})",
                np.allclose(g.GV_Weight.astype(float), h["gv"], atol=1e-4))
        if h["lambda"] == 0:
            r.check(f"E{e}: Final_L3 = 0 saat lambda = 0", bool((g.Final_L3.astype(float) == 0).all()))
        else:
            r.check(f"E{e}: Final_L3 > 0 saat lambda > 0", bool((g.Final_L3.astype(float) > 0).all()),
                    f"min={g.Final_L3.min():.6f}")

    # --- rentang yang dijamin oleh kode ---
    r.check("P2_Prob_Real di [0, 1]", bool(df.P2_Prob_Real.between(0, 1).all()))
    r.check("Diff_W di [0.1, 2.0] (clamp di compute_router_loss)", bool(df.Diff_W.between(0.1 - 1e-6, 2.0 + 1e-6).all()),
            f"[{df.Diff_W.min():.4f}, {df.Diff_W.max():.4f}]")
    r.check("Rel ≥ 0", bool((df.Rel >= 0).all()))
    r.check("|GV_Loss| ≤ clip advantage (mean(p·adv), |adv| ≤ clip)",
            bool((df.GV_Loss.abs() <= offset_clip + 1e-6).all()), f"max |GV_Loss| = {df.GV_Loss.abs().max():.4f}")

    # --- kolom GV berubah tiap batch (baris pertama: offset EMA belum ada) ---
    g = df.iloc[1:].reset_index(drop=True)
    empty = ((g.GV_Loss == 0) & (g.GV_Target == 0)).mean()
    r.check("Batch tanpa sampel valid (GV_Loss = GV_Target = 0) < 5%", empty < 0.05, f"{empty:.1%}", warn_only=True)
    for col, thr in [("GV_Target", 0.95), ("GV_Loss", 0.95), ("GV_Offset", 0.80)]:
        frac = float((g[col].diff().iloc[1:] != 0).mean()) if len(g) > 1 else 0.0
        r.check(f"{col} berubah antar-batch (≥ {thr:.0%} batch)", frac >= thr, f"{frac:.1%}")

    # --- GV_Offset tidak menempel di ±clip ---
    at_clip = float((df.GV_Offset.abs() >= offset_clip - 1e-3).mean())
    r.check(f"GV_Offset tidak menempel di ±{offset_clip}", at_clip == 0, f"{at_clip:.1%} baris di batas")
    r.check("GV_Offset jauh dari batas (|offset| < 0.5·clip)", bool(df.GV_Offset.abs().max() < 0.5 * offset_clip),
            f"rentang [{df.GV_Offset.min():.4f}, {df.GV_Offset.max():.4f}]", warn_only=True)
    tail = df.GV_Offset.iloc[-max(len(df) // 4, 2):]
    r.check("GV_Offset tidak monoton naik/turun terus di 25% akhir (indikasi drift)",
            not (tail.diff().dropna() > 0).all() and not (tail.diff().dropna() < 0).all(),
            f"awal {tail.iloc[0]:.4f} → akhir {tail.iloc[-1]:.4f}", warn_only=True)

    # --- aktivasi tidak kolaps (indikator awal saja, 1 epoch tidak konklusif) ---
    last_e = df.Epoch.max()
    p_last = float(df[df.Epoch == last_e].P2_Prob_Real.mean())
    r.check(f"E{last_e}: rata-rata P2_Prob_Real di [0.05, 0.95] (bukan kolaps)", 0.05 <= p_last <= 0.95,
            f"{p_last:.3f}", warn_only=True)
    return r


def check_loss_items(li_df, router_warmup_end, sched_history, report=None):
    """Pemeriksaan 8 komponen loss per batch (loss_items.csv)."""
    r = report or Report()
    vals = li_df[LOSS_NAMES].astype(float)
    r.check("loss_items: 8 komponen finite di semua batch", bool(np.isfinite(vals.values).all()))
    for e, g in li_df.groupby("epoch"):
        lam = sched_history.get(int(e), {}).get("lambda", 0.0)
        warm = int(e) <= router_warmup_end
        rmean = float(g.router.mean())
        if warm or lam == 0:
            r.check(f"E{e}: komponen 'router' = 0 (warmup / lambda 0)", bool((g.router == 0).all()), f"mean={rmean:.6f}")
        else:
            r.check(f"E{e}: komponen 'router' > 0 (lambda > 0)", rmean > 0, f"mean={rmean:.6f}")
    # informatif (bukan kriteria lulus): 3 epoch terlalu pendek untuk menilai konvergensi
    print(li_df.groupby("epoch")[LOSS_NAMES].mean().round(4).to_string())
    return r


def budget_table(timing_df, plan, branch_b_interval=10, overhead=0.10, session_limit_h=12.0):
    """
    Estimasi jam GPU per run dari waktu epoch 'steady' (epoch ≥ 2; epoch 1 memuat cache,
    autotune cuDNN, check_amp, dsb).

    jam = (E·(t_train + t_valA) + floor(E/interval_B)·t_valB) · (1 + overhead) / 3600
    t_valA = val_total − valB (validasi Branch A + simpan checkpoint + callback lain).
    """
    st = timing_df[timing_df.epoch >= 2]
    if st.empty:
        st = timing_df
    t_train = st.train_s.mean()
    t_valB = st.valB_s.mean()
    t_valA = (st.val_total_s - st.valB_s).mean()
    rows = []
    for name, (E, n_runs) in plan.items():
        h = (E * (t_train + t_valA) + (E // branch_b_interval) * t_valB) * (1 + overhead) / 3600
        rows.append(dict(run=name, epochs=E, n_runs=n_runs, jam_per_run=round(h, 2),
                         jam_total=round(h * n_runs, 2),
                         perlu_resume=h > session_limit_h - 0.5))
    summary = dict(t_train_s=round(t_train, 1), t_valA_s=round(t_valA, 1), t_valB_s=round(t_valB, 1),
                   epoch1_train_s=round(float(timing_df.iloc[0].train_s), 1))
    return pd.DataFrame(rows), summary


CALIB_T_TEST = 1.2345678901234567   # nilai "tidak bulat" -> pembulatan fp16/fp32 pasti ketahuan
CALIB_B_TEST = -0.3456789012345678


def _calib(router):
    return float(getattr(router, "calib_T", 1.0)), float(getattr(router, "calib_b", 0.0))


def test_calibration_persistence(model, build_fresh_model, tmp_dir, report=None):
    """
    Uji 0.4. Set calib_T/calib_b, simpan, muat ulang lewat 3 jalur yang benar-benar dipakai:
      J1 pickle checkpoint Ultralytics : torch.save({'ema': deepcopy(model).half()}) -> torch.load
         (= trainer.save_model + attempt_load_one_weight)
      J2 state_dict -> model baru       : load_state_dict(strict=False)
         (= load_dualbranch() di notebook uji-akurasi-latensi)
      J3 Ultralytics model.load()       : intersect_dicts + load_state_dict
         (= DualBranchTrainer.get_model(weights=...), mis. fine-tune)

    Sejak AUDIT-FIX #9 kalibrasi disimpan sebagai buffer int64 dalam mikro-unit, jadi nilai
    yang tersimpan = round(v·1e6)/1e6. "Sama persis" di sini berarti: nilai yang DIBACA
    sebelum simpan == nilai setelah muat ulang (==, bukan allclose), dan galat kuantisasi
    set→baca ≤ 5e-7.
    """
    r = report or Report()
    tmp_dir = Path(tmp_dir); tmp_dir.mkdir(parents=True, exist_ok=True)
    rt = find_router(model)
    rt.calib_T, rt.calib_b = CALIB_T_TEST, CALIB_B_TEST
    ref = (rt.calib_T, rt.calib_b)
    r.check("Kalibrasi: galat kuantisasi set→baca ≤ 5e-7",
            abs(ref[0] - CALIB_T_TEST) <= 5e-7 and abs(ref[1] - CALIB_B_TEST) <= 5e-7,
            f"set=({CALIB_T_TEST}, {CALIB_B_TEST}) baca={ref}")
    keys = [k for k in model.state_dict() if "calib" in k]
    r.check("Kalibrasi ikut state_dict (buffer)", len(keys) == 2, keys)

    # J1
    p = tmp_dir / "calib_pickle.pt"
    torch.save({"ema": deepcopy(model).half()}, p)
    m1 = torch.load(p, map_location="cpu", weights_only=False)["ema"].float()
    rt1 = find_router(m1)
    r.check("Kalibrasi J1 (pickle checkpoint Ultralytics, lewat .half()) identik", _calib(rt1) == ref, f"{_calib(rt1)}")
    r.check("Buffer kalibrasi tetap int64 setelah .half()/.float()",
            all(b.dtype == torch.int64 for n, b in rt1.named_buffers() if "calib" in n))

    # J2
    sd = {k: (v.float() if v.is_floating_point() else v) for k, v in model.state_dict().items()}
    torch.save(sd, tmp_dir / "calib_state_dict.pt")
    sd2 = torch.load(tmp_dir / "calib_state_dict.pt", map_location="cpu")
    m2 = build_fresh_model()
    missing, unexpected = m2.load_state_dict(sd2, strict=False)
    r.check("Kalibrasi J2 (state_dict -> model baru) identik", _calib(find_router(m2)) == ref,
            f"{_calib(find_router(m2))} | missing={len(missing)} unexpected={len(unexpected)}")

    # J3
    m3 = build_fresh_model()
    m3.load(m1, verbose=False)
    r.check("Kalibrasi J3 (Ultralytics model.load) identik", _calib(find_router(m3)) == ref, f"{_calib(find_router(m3))}",
            warn_only=True)   # jalur fine-tune: ikut mewarisi kalibrasi -> sadari saat fine-tune
    return r, (m1, m2, m3)


@torch.no_grad()
def gate_decisions_both_paths(model, img):
    """
    Satu forward eval penuh (model(img) -> _predict_once -> router.forward eval), tangkap input
    router via forward-pre-hook, lalu panggil compute_gate_only pada input yang SAMA.
    Returns: (dec_forward atau None, n_active_forward, dec_gate_only, prob_gate_only)
    """
    rt = find_router(model)
    cap = {}
    h = rt.register_forward_pre_hook(lambda m, inp: cap.update(p3=inp[0][0], p2=inp[0][1]))
    try:
        model(img)
    finally:
        h.remove()
    B = img.shape[0]
    dec_fwd = getattr(rt, "_last_gate_decision", None)        # butuh patch P2
    dec_fwd = dec_fwd.float().flatten().clone() if dec_fwd is not None else None
    n_fwd = float(rt.current_activation_prob) * B
    dec_only = rt.compute_gate_only(cap["p3"], cap["p2"]).float().flatten()
    # probabilitas (untuk margin ke threshold) — rumus sama dgn compute_gate_only
    z = rt._topk_spatial_pool(rt.squeeze_p3(cap["p3"]) + rt.squeeze_p2(cap["p2"]),
                              k=getattr(rt, "gate_topk", 10), alpha=getattr(rt, "gate_topk_alpha", 0.7))
    l = 5.0 * torch.tanh(rt.classifier(z).view(B, 2).float() / 5.0)
    prob = torch.sigmoid((l[:, 1] - l[:, 0]) * getattr(rt, "calib_T", 1.0) + getattr(rt, "calib_b", 0.0))
    return dec_fwd, n_fwd, dec_only, prob


def test_gate_consistency(model, batches, calibs, report=None):
    """
    Uji 0.5. Untuk tiap kalibrasi & tiap batch: keputusan forward(eval) == compute_gate_only.
    `calibs`: list (nama, T, b). Model harus .eval(); presisi mengikuti dtype model.
    """
    r = report or Report()
    rt = find_router(model)
    saved = _calib(rt)
    model.eval()
    try:
        for name, T, b in calibs:
            rt.calib_T, rt.calib_b = T, b
            n_img = n_mis = 0
            n_fwd_tot = n_only_tot = 0.0
            min_margin = 1.0
            weak = False
            for img in batches:
                dec_fwd, n_fwd, dec_only, prob = gate_decisions_both_paths(model, img)
                n_img += img.shape[0]
                n_fwd_tot += n_fwd
                n_only_tot += float(dec_only.sum())
                min_margin = min(min_margin, float((prob - 0.5).abs().min()))
                if dec_fwd is None:
                    weak = True
                else:
                    n_mis += int((dec_fwd.cpu() != dec_only.cpu()).sum())
            if weak:
                r.check(f"Gate [{name}] jumlah aktif forward == compute_gate_only (VERSI LEMAH, patch P2 belum)",
                        abs(n_fwd_tot - n_only_tot) < 0.5,
                        f"{n_fwd_tot:.0f} vs {n_only_tot:.0f} dari {n_img}", warn_only=True)
            else:
                r.check(f"Gate [{name}] keputusan per-gambar identik", n_mis == 0,
                        f"{n_mis}/{n_img} beda | aktif {n_only_tot:.0f}/{n_img} "
                        f"({n_only_tot / max(n_img, 1):.1%}) | margin min |p−0.5| = {min_margin:.2e}")
    finally:
        rt.calib_T, rt.calib_b = saved
    return r


def percentile_bias_for_rate(model, batches, target_rate, T=1.0):
    """b sehingga fraksi (logit·T + b > 0) ≈ target_rate pada `batches` (gaya DynamicDet).
    Dipakai HANYA untuk membuat kalibrasi uji yang menaruh banyak sampel dekat threshold."""
    rt = find_router(model)
    saved = _calib(rt)
    rt.calib_T, rt.calib_b = 1.0, 0.0
    logits = []
    for img in batches:
        _, _, _, prob = gate_decisions_both_paths(model, img)
        logits.append(torch.logit(prob.clamp(1e-7, 1 - 1e-7)).cpu())
    rt.calib_T, rt.calib_b = saved
    lg = torch.cat(logits) * T
    return float(-torch.quantile(lg, 1.0 - target_rate))




# =============================================================================
# PATCH SUMBER (EOL-aware, idempotent) — dipakai notebook sebelum import ultralytics
# =============================================================================
def apply_source_patch(path, old, new, done_marker, name):
    """
    Ganti `old` -> `new` tepat satu kali di file `path`, mempertahankan EOL file (LF/CRLF).
    Idempotent: bila `done_marker` sudah ada, file tidak diubah. Returns status string.
    """
    path = Path(path)
    raw = path.read_bytes().decode("utf-8")
    if done_marker in raw:
        return f"ℹ️ {name}: sudah ada — tidak di-patch."
    if "\r\n" in raw:
        old, new = old.replace("\n", "\r\n"), new.replace("\n", "\r\n")
    n = raw.count(old)
    if n != 1:
        raise RuntimeError(f"{name}: pola ditemukan {n}x (harus 1) di {path} — versi file berbeda, cek manual.")
    path.write_bytes(raw.replace(old, new).encode("utf-8"))
    return f"✅ {name}: di-patch."


PATCH_P2 = dict(
    name="P2 keputusan & logit gate per-gambar (block.py, eval)",
    rel="ultralytics/nn/modules/block.py",
    done_marker="self._last_gate_logit_raw = raw_logit_diff.detach()",
    old=("            if not torch.jit.is_tracing():\n"
         "                self.current_activation_prob = gate_mask.mean().detach()\n\n"
         "            f_c2f = self.compute_expert(f_p3, f_p2_back)\n"),
    new=("            if not torch.jit.is_tracing():\n"
         "                self.current_activation_prob = gate_mask.mean().detach()\n"
         "                # [TAHAP0] per-gambar: keputusan (uji konsistensi vs compute_gate_only) dan logit\n"
         "                # mentah sebelum kalibrasi (sebaran logit di log validasi)\n"
         "                self._last_gate_decision = gate_mask.view(B).detach()\n"
         "                self._last_gate_logit_raw = raw_logit_diff.detach()\n\n"
         "            f_c2f = self.compute_expert(f_p3, f_p2_back)\n"),
)

PATCH_CONF_SIGN = dict(
    name="Tanda conf di difficulty score (loss.py): - menjadi +",
    rel="ultralytics/utils/loss.py",
    done_marker="diff_score = entropy_f + var_f + conf_f",
    old=("                # TODO: verifikasi -- router.last_conf adalah UNCERTAINTY (1 - prob, tinggi =\n"
         "                # tidak yakin), sama arah dengan entropy dan var. Mengurangkannya di sini\n"
         "                # berarti ketidakpastian kelas yang lebih tinggi MENURUNKAN skor kesulitan.\n"
         "                # Apakah tanda minus ini disengaja (mis. dulu conf berarti keyakinan)?\n"
         "                diff_score = entropy_f + var_f - conf_f\n"),
    new=("                # [FIX 2026-10-09] router.last_conf adalah UNCERTAINTY (1 - prob, tinggi = tidak\n"
         "                # yakin), searah dengan entropy dan var. Tanda minus versi lama adalah salah ketik\n"
         "                # (dikonfirmasi penulis): ketiga sinyal ketidakpastian kini DIJUMLAHKAN.\n"
         "                # Run sebelum fix ini memakai entropy + var - conf (tidak sebanding langsung).\n"
         "                diff_score = entropy_f + var_f + conf_f\n"),
)


# =============================================================================
# GO/NO-GO KUALITAS RANKING GATE (mis. checkpoint epoch 50, sebelum lambda naik)
# =============================================================================
def _raw_gate_logit(rt, p3, p2):
    """Logit gate mentah (sebelum kalibrasi) — rumus identik dengan compute_gate_only."""
    z = rt._topk_spatial_pool(rt.squeeze_p3(p3) + rt.squeeze_p2(p2),
                              k=getattr(rt, "gate_topk", 10), alpha=getattr(rt, "gate_topk_alpha", 0.7))
    l = 5.0 * torch.tanh(rt.classifier(z).view(p3.shape[0], 2).float() / 5.0)
    return l[:, 1] - l[:, 0]


def _auc_with_ci(labels, scores, n_boot=1000, seed=0):
    """AUC (Mann-Whitney, ties = 0.5) + CI 95% bootstrap persentil. labels: 1 = Branch A lebih baik."""
    from scipy.stats import rankdata
    labels = np.asarray(labels, bool); scores = np.asarray(scores, float)

    def auc(lb, sc):
        n1, n0 = lb.sum(), (~lb).sum()
        if n1 == 0 or n0 == 0:
            return np.nan
        r = rankdata(sc)
        return (r[lb].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

    a = auc(labels, scores)
    rng = np.random.default_rng(seed); n = len(labels); boots = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        boots.append(auc(labels[i], scores[i]))
    boots = np.array([b for b in boots if np.isfinite(b)])
    lo, hi = (np.percentile(boots, [2.5, 97.5]) if len(boots) else (np.nan, np.nan))
    return float(a), float(lo), float(hi)


@torch.no_grad()
def gate_ranking_report(model, loader, device, max_batches=None, signal_mode="small",
                        min_std=0.05, n_boot=1000):
    """
    Apakah gate sudah MERANKING gambar sesuai sinyal yang melatihnya? (model beku, FP32, eval)

    Per gambar val: logit gate mentah + loss_A/loss_B per-sample (total & small, dari criterion
    yang sama dengan training). Oracle = 1 bila loss_A < loss_B. Untuk mode 'small' hanya
    gambar dengan objek kecil ter-assign di KEDUA branch yang dipakai (sama dgn valid_mask training).

    GO bila: std logit > min_std DAN batas bawah CI 95% AUC (mode sinyal training) > 0.5.
    Catatan: oracle memakai sinyal loss yang noisy — AUC menjawab "apakah gate mempelajari
    sinyal yang diberikan", bukan "apakah keputusan itu benar secara semantik".
    """
    from ultralytics.cfg import get_cfg
    from ultralytics.utils import DEFAULT_CFG
    model = deepcopy(model).float().eval().to(device)
    if isinstance(getattr(model, "args", None), dict) or not hasattr(model, "args"):
        a = getattr(model, "args", {}) or {}
        model.args = get_cfg(DEFAULT_CFG, overrides={k: a[k] for k in ("box", "cls", "dfl") if k in a})
    rt = find_router(model)
    crit = model.init_criterion()
    feats, rows = {}, {k: [] for k in ("raw", "la", "lb", "las", "lbs", "nsa", "nsb")}
    h = rt.register_forward_pre_hook(lambda m, inp: feats.update(p3=inp[0][0], p2=inp[0][1]))
    try:
        for bi, batch in enumerate(loader):
            if max_batches is not None and bi >= max_batches:
                break
            img = batch["img"].to(device).float() / 255.0
            b = {"img": img, **{k: batch[k].to(device) for k in ("cls", "bboxes", "batch_idx")}}
            det_A, det_B = model._predict_once_dual(img)
            crit.loss_A(det_A, b); crit.loss_B(det_B, b)
            rows["raw"].append(_raw_gate_logit(rt, feats["p3"], feats["p2"]).cpu())
            rows["la"].append(crit.loss_A._last_per_sample_loss.float().cpu())
            rows["lb"].append(crit.loss_B._last_per_sample_loss.float().cpu())
            rows["las"].append(crit.loss_A._last_per_sample_loss_small.float().cpu())
            rows["lbs"].append(crit.loss_B._last_per_sample_loss_small.float().cpu())
            rows["nsa"].append(crit.loss_A._last_small_obj_count.float().cpu())
            rows["nsb"].append(crit.loss_B._last_small_obj_count.float().cpu())
    finally:
        h.remove()
    R = {k: torch.cat(v).numpy() for k, v in rows.items()}
    raw = R["raw"]
    prob = 1.0 / (1.0 + np.exp(-raw))
    out = dict(n_images=int(len(raw)), logit_mean=float(raw.mean()), logit_std=float(raw.std()),
               prob_std_uncalibrated=float(prob.std()),
               logit_p10=float(np.percentile(raw, 10)), logit_p90=float(np.percentile(raw, 90)),
               frac_saturated=float((np.abs(raw) > 9.5).mean()),
               activation_uncalibrated=float((raw > 0).mean()))
    a, lo, hi = _auc_with_ci(R["la"] < R["lb"], raw, n_boot)
    out.update(total=dict(n=int(len(raw)), oracle_rate=float((R["la"] < R["lb"]).mean()), auc=a, auc_ci=[lo, hi]))
    v = (R["nsa"] > 0) & (R["nsb"] > 0)
    if v.sum() >= 20:
        a, lo, hi = _auc_with_ci(R["las"][v] < R["lbs"][v], raw[v], n_boot)
        out.update(small=dict(n=int(v.sum()), oracle_rate=float((R["las"][v] < R["lbs"][v]).mean()),
                              auc=a, auc_ci=[lo, hi]))
    key = signal_mode if signal_mode in out else "total"
    out["decision_signal"] = key
    out["go"] = bool(out["logit_std"] > min_std and out[key]["auc_ci"][0] > 0.5)
    out["reason"] = (f"std={out['logit_std']:.4f} (> {min_std}?), AUC[{key}]={out[key]['auc']:.3f} "
                     f"CI95=[{out[key]['auc_ci'][0]:.3f}, {out[key]['auc_ci'][1]:.3f}] (batas bawah > 0.5?)")
    return out


def stop_training(trainer, reason):
    """
    Hentikan training dengan rapi setelah epoch ini. final_eval() -> strip_optimizer() membuat
    last.pt TIDAK bisa di-resume (epoch=-1, optimizer dibuang), jadi salinan utuh disimpan SEKARANG
    (save_model sudah jalan untuk epoch ini). Returns path salinan.
    """
    epoch = trainer.epoch + 1
    keep = Path(trainer.wdir) / f"last_resumable_epoch{epoch}.pt"
    # final_eval() memicu on_fit_epoch_end sekali lagi SETELAH last.pt di-strip -> jangan timpa salinan
    if getattr(trainer, "stop", False) or (Path(trainer.save_dir) / "STOPPED.txt").exists():
        return str(keep)
    if Path(trainer.last).exists():
        shutil.copy(trainer.last, keep)
    (Path(trainer.save_dir) / "STOPPED.txt").write_text(f"epoch {epoch}: {reason}\nresume: {keep}\n")
    print(f"⛔ Training dihentikan di epoch {epoch}: {reason}. Untuk lanjut: resume dari {keep}")
    trainer.stop = True
    return str(keep)


def make_activation_stop_rule(gate_val_csv, after_epoch=60, low=0.05, high=0.95, patience=5):
    """
    Aturan berhenti Tahap 1: aktivasi val (EMA, tanpa kalibrasi, dari gate_val_stats.csv) < low atau
    > high selama `patience` epoch berturut-turut SETELAH `after_epoch`. Dipasang SESUDAH callback yang
    menulis gate_val_stats.csv (on_val_end -> sudah tertulis saat on_fit_epoch_end).
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        if getattr(trainer, "stop", False) or epoch <= after_epoch or not Path(gate_val_csv).exists():
            return
        gv = pd.read_csv(gate_val_csv).drop_duplicates("epoch", keep="first").set_index("epoch")
        win = [e for e in range(epoch - patience + 1, epoch + 1)]
        if all(e in gv.index and e > after_epoch for e in win):
            a = gv.loc[win, "act_uncalibrated"]
            if (a < low).all() or (a > high).all():
                stop_training(trainer, f"aktivasi val {a.round(3).tolist()} di luar [{low}, {high}] "
                                       f"selama {patience} epoch (epoch {win[0]}–{win[-1]})")
    return cb


def make_logit_health_stop_rule(gate_val_csv, after_epoch=10, min_std=0.05, max_sat=0.20, patience=2):
    """
    Aturan berhenti Tahap 1 versi alpha = 0 (revisi 2026-10-09, menggantikan aturan aktivasi).
    Dengan alpha = 0 aktivasi deployment dipilih post-hoc lewat persentil, jadi yang harus dijaga
    adalah PERINGKAT logit: berhenti bila std logit val < min_std ATAU fraksi logit tersaturasi
    (|z| > 9.5, batas tanh +-10) > max_sat selama `patience` epoch berturut-turut setelah after_epoch
    (setelah warmup router: selama warmup logit memang hampir konstan).
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        if getattr(trainer, "stop", False) or epoch <= after_epoch or not Path(gate_val_csv).exists():
            return
        gv = pd.read_csv(gate_val_csv).drop_duplicates("epoch", keep="last").set_index("epoch")
        if "logit_std" not in gv:
            return
        win = list(range(epoch - patience + 1, epoch + 1))
        if all(e in gv.index and e > after_epoch for e in win):
            w = gv.loc[win]
            bad = (w.logit_std < min_std) | (w.frac_saturated > max_sat)
            if bad.all():
                stop_training(trainer, f"kesehatan logit gagal epoch {win[0]}–{win[-1]}: std={w.logit_std.round(4).tolist()} "
                                       f"(min {min_std}), saturasi={w.frac_saturated.round(3).tolist()} (maks {max_sat})")
    return cb


def make_auc_stop_rule(trainer_ref, out_csv, every=10, after_epoch=30, patience=2, signal_mode="total",
                       max_batches=None):
    """
    Pengaman Tahap 1 revisi ke-2 (2026-10-09, setelah run M0 seed 42 λ=0.04: AUC epoch 100 = 0.417,
    terbalik). Tiap `every` epoch: gate_ranking_report pada model EMA di val -> baris ke out_csv
    (epoch, AUC, CI, std logit, aktivasi). Berhenti bila batas bawah CI 95% AUC <= 0.5 pada
    `patience` pemeriksaan berturut-turut dengan epoch > after_epoch. Pemeriksaan <= after_epoch
    tetap dicatat (informatif). Biaya: +-1 menit per pemeriksaan di T4.
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        # Catatan: JANGAN pakai trainer.stop sebagai guard — Ultralytics men-set stop=True di epoch terakhir
        # sebelum on_fit_epoch_end, sehingga pemeriksaan epoch 100 terlewat (bug di run M0_lam002_seed42).
        if epoch % every != 0 or (Path(trainer.save_dir) / "STOPPED.txt").exists():
            return
        hist = pd.read_csv(out_csv) if Path(out_csv).exists() else pd.DataFrame()
        if not hist.empty and epoch in set(hist.epoch):
            return                                   # final_eval memicu callback lagi -> jangan dobel
        device = next(trainer.model.parameters()).device
        rep = gate_ranking_report(trainer.ema.ema, trainer.test_loader, device, max_batches=max_batches,
                                  signal_mode=signal_mode)
        key = rep["decision_signal"]
        row = dict(epoch=epoch, auc=rep[key]["auc"], auc_lo=rep[key]["auc_ci"][0], auc_hi=rep[key]["auc_ci"][1],
                   logit_std=rep["logit_std"], logit_mean=rep["logit_mean"],
                   act_uncalibrated=rep["activation_uncalibrated"],
                   auc_small=rep.get("small", {}).get("auc", np.nan))
        hist = pd.concat([hist, pd.DataFrame([row])], ignore_index=True)
        hist.to_csv(out_csv, index=False)
        print(f"\n📊 [AUC CHECK epoch {epoch}] AUC[{key}]={row['auc']:.3f} CI95=[{row['auc_lo']:.3f}, {row['auc_hi']:.3f}] "
              f"| std logit={row['logit_std']:.3f} | aktivasi={row['act_uncalibrated']:.3f}\n")
        late = hist[hist.epoch > after_epoch].sort_values("epoch").tail(patience)
        if len(late) == patience and (late.auc_lo <= 0.5).all():
            stop_training(trainer, f"ranking gate gagal: batas bawah CI AUC <= 0.5 pada epoch {late.epoch.tolist()} "
                                   f"(AUC {late.auc.round(3).tolist()})")
    return cb


def make_criterion_cfg_applier(criterion_cfg, dump_path=None):
    """
    on_pretrain_routine_end (SETELAH router cfg): buat criterion sekarang dan set atribut yang dibaca
    v8DetectionLoss dari DIRINYA SENDIRI (mis. normalize_bg_anchors), bukan dari model.
    Model.loss() hanya membuat criterion bila belum ada, jadi criterion ini yang dipakai training.
    """
    def apply(trainer):
        m = trainer.model
        m.criterion = m.init_criterion()
        for k, v in criterion_cfg.items():
            setattr(m.criterion.loss_A, k, v); setattr(m.criterion.loss_B, k, v)
        if dump_path is not None:
            Path(dump_path).write_text(json.dumps(criterion_cfg, indent=1))
        print("🔧 [CRITERION CFG]", criterion_cfg)
    return apply


def make_gate_ranking_check(trainer_ref, at_epoch, out_path, signal_mode="small", stop_on_nogo=True,
                            max_batches=None):
    """
    on_fit_epoch_end: di akhir epoch `at_epoch`, jalankan gate_ranking_report pada model EMA
    (val loader trainer). NO-GO + stop_on_nogo -> trainer.stop = True (training berhenti rapi,
    last.pt tersimpan, bisa dilanjutkan dengan resume=True bila diputuskan lanjut).
    """
    def cb(trainer):
        epoch = trainer.epoch + 1
        if epoch != at_epoch or Path(out_path).exists():
            return
        device = next(trainer.model.parameters()).device
        rep = gate_ranking_report(trainer.ema.ema, trainer.test_loader, device, max_batches=max_batches,
                                  signal_mode=signal_mode)
        rep.update(epoch=epoch, stop_on_nogo=stop_on_nogo)
        Path(out_path).write_text(json.dumps(rep, indent=1))
        print(f"\n🚦 [GATE CHECK epoch {epoch}] {'GO' if rep['go'] else 'NO-GO'} — {rep['reason']}\n")
        if not rep["go"] and stop_on_nogo:
            rep["resumable_checkpoint"] = stop_training(trainer, "gate check NO-GO")
            Path(out_path).write_text(json.dumps(rep, indent=1))
    return cb

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/script_training.py
# [5] versi sama dengan Tahap 1 (hapus sel ini setelah file di-commit ke repo)
"""
script_training.py — Training Router-P2 YOLOv8n Dual-Branch (Skenario 3), versi importable.

Perubahan dibanding kode_berkomentar/script_training.py (logika model/loss TIDAK berubah):
  - Section 1 lama (patch loss.py di disk lewat penggantian string) DIHAPUS: guard
    _compute_router_penalty dan default lambda sudah ada di loss.py, sehingga patch itu
    tidak pernah mengubah apa pun, dan menulis file library saat runtime berisiko.
  - Eksekusi dibungkus build_trainer() + main(); import file ini tidak memulai training.
  - ideal_curriculum_scheduler diganti make_curriculum_scheduler (router_train_utils),
    yang dengan FULL_SCHED identik dengan jadwal lama (diuji epoch 1–100) kecuali lambda
    final = 0.04 (keputusan 2026-10-09).
  - Hyperparameter router ditulis eksplisit (ROUTER_CFG); router_hybrid_alpha = 0.
  - Semua log ke save_dir run: router_loss_components.csv (11 kolom, label epoch benar),
    loss_items.csv, epoch_timing.csv, branch_b_metrics.csv, router_cfg.json, sched_history.json.
  - Penjaga NaN per batch (gagal cepat, bukan NaN recovery diam-diam Ultralytics).
  - Sebaran logit gate per validasi (gate_val_stats.csv) dan go/no-go ranking gate di akhir
    epoch lam_start (50), tepat sebelum lambda naik (gate_check.json; NO-GO -> berhenti).
  - Butuh patch sumber (notebook sel 4): P2 logit/keputusan gate per-gambar, tanda conf (+).
"""
import os
import csv
import json
from copy import copy
from pathlib import Path

import pandas as pd
import torch
from ultralytics import settings
from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator
from ultralytics.model_yolov8_router import DualBranchDetectionModel

from router_train_utils import (EpochTimer, make_activation_stop_rule, make_criterion_cfg_applier,
                                make_logit_health_stop_rule, make_auc_stop_rule,
                                make_curriculum_scheduler, make_gate_ranking_check, make_loss_items_logger,
                                make_router_cfg_applier, make_router_csv_logger)

# ==============================================================================
# KONFIGURASI RUN UTAMA
# ==============================================================================
BASE_OVERRIDES = dict(
    model="ultralytics/cfg/models/v8/yolov8n-p2-router.yaml",
    data="/kaggle/input/datasets/agungmaugi/kitti-2class/kitti_2class.yaml",
    batch=16, imgsz=640, optimizer="AdamW", lr0=0.002, cos_lr=True, weight_decay=0.0005,
    device=0, workers=4, seed=42, amp=True, verbose=True, save=True,
)
# save_period=10: epoch50.pt (go/no-go) + titik resume bila sesi Kaggle putus.
FULL_OVERRIDES = {**BASE_OVERRIDES, "epochs": 100, "project": "/kaggle/working/runs_tahap1",
                  "name": "M0_lam002_seed42", "save_period": 10}

# Fase 1 (1–50): lambda 0; router warmup 1–10; gate_value_weight 0 -> 0.5 di epoch 11–30.
# Fase 2 (51–80): lambda 0 -> 0.02 linear. Fase 3 (81–100): lambda 0.02.
# Fase 2 (51–80): lambda 0 -> final linear. Fase 3 (81–100): lambda final.
FULL_SCHED = dict(router_warmup_end=10, gv_start=10, gv_ramp=20, final_gv=0.5,
                  lam_start=50, lam_ramp=30, final_lambda=0.02)
# λ akhir 0.02 (keputusan 2026-10-09 malam). Run pertama λ=0.04 (seed 42) berakhir dengan ranking gate
# terbalik (AUC 0.417 di epoch 100) dan kini dilaporkan sebagai ablasi.

# Keputusan 2026-10-09: alpha = 0 -> router_target_activation tidak berpengaruh; sparsity bekerja
# sebagai harga P2 (c = lambda * Diff_W / gate_value_weight); activation rate deployment dipilih
# post-hoc (operating_point_sweep.py). Nilai target tetap ditulis supaya konfigurasi eksplisit.
# gate_signal_mode='total' (rencana, konfigurasi terkunci): 'small' membuang gambar yang objek kecilnya
# hanya tertangkap P2 (bias seleksi); 'small' diuji sebagai ablasi A5.
ROUTER_CFG = dict(
    gate_signal_mode="total",
    router_hybrid_alpha=0.0,
    router_target_activation=0.35,
    router_activation_tolerance=0.10,
    router_min_activation_floor=0.05,
    branch_b_loss_weight=0.7,
    gate_offset_momentum=0.99,
)
# Pengaman revisi ke-2 (2026-10-09): AUC ranking gate di val tiap 10 epoch; berhenti bila batas bawah
# CI <= 0.5 dua pemeriksaan berturut-turut setelah epoch 30. Menggantikan aturan aktivasi (rencana awal)
# dan aturan std/saturasi logit (revisi ke-1), yang tidak menangkap ranking terbalik di run λ=0.04.
AUC_STOP_RULE = dict(every=10, after_epoch=30, patience=2)

# Atribut yang dibaca v8DetectionLoss dari dirinya sendiri (bukan dari model) -> diset ke criterion.
CRITERION_CFG = dict(normalize_bg_anchors=True)   # ablasi A6: False


# ==============================================================================
# TRAINER & VALIDATOR
# ==============================================================================
class DualBranchTrainer(DetectionTrainer):
    """DetectionTrainer yang membangun DualBranchDetectionModel dan 8 nama komponen loss."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = DualBranchDetectionModel(cfg=cfg or self.args.model, ch=3, nc=self.data["nc"], verbose=verbose)
        if weights:
            model.load(weights)
        return model

    def get_validator(self):
        # Urutan = combined_items di DualBranchDetectionLoss: 5 item loss_A + 3 item loss_B.
        self.loss_names = ("box_A", "cls_A", "dfl_A", "router", "proxy", "box_B", "cls_B", "dfl_B")
        return DetectionValidator(self.test_loader, save_dir=self.save_dir, args=copy(self.args),
                                  _callbacks=self.callbacks)


class DualBranchValidatorB(DetectionValidator):
    """Validator yang mengevaluasi Branch B: _predict_once diganti sementara, selalu dipulihkan."""

    def __call__(self, trainer=None, model=None):
        if model is not None:
            target_model = model
        elif trainer is not None:
            target_model = trainer.ema.ema if (hasattr(trainer, "ema") and trainer.ema) else trainer.model
        else:
            raise ValueError("DualBranchValidatorB butuh `trainer` atau `model`.")
        assert hasattr(target_model, "_predict_once_dual"), "Model bukan DualBranchDetectionModel."

        original = target_model._predict_once

        def branch_b_predict_once(x, profile=False, visualize=False, embed=None):
            _, det_B = target_model._predict_once_dual(x, profile, visualize, embed)
            return det_B

        target_model._predict_once = branch_b_predict_once
        try:
            return super().__call__(trainer=trainer, model=model)
        finally:
            target_model._predict_once = original


# ==============================================================================
# CALLBACK VALIDASI
# ==============================================================================
def make_router_stat_checker(trainer_ref, csv_path=None):
    """
    Aktivasi P2 + sebaran logit gate mentah di validasi, per epoch -> csv_path.
    Angka ini TANPA kalibrasi, FP16, dataloader rect -> bukan angka deployment; gunanya memantau
    apakah logit gate tersebar antar-gambar (syarat kalibrasi persentil) sepanjang training.
    Butuh patch P2 (router._last_gate_logit_raw); tanpa patch hanya aktivasi yang tercatat.
    """
    stats = {"total": 0.0, "n": 0, "logits": []}

    def on_val_start(validator):
        stats["total"], stats["n"], stats["logits"] = 0.0, 0, []

    def on_val_batch_end(validator):
        m_ = trainer_ref.ema.ema if (hasattr(trainer_ref, "ema") and trainer_ref.ema) else trainer_ref.model
        if m_ is None:
            return
        for m in m_.modules():
            if "DifficultyAwareRouter" in m.__class__.__name__:
                v = getattr(m, "current_activation_prob", None)
                if v is not None:
                    stats["total"] += v.item() if hasattr(v, "item") else float(v)
                    stats["n"] += 1
                lg = getattr(m, "_last_gate_logit_raw", None)
                if lg is not None:
                    stats["logits"].append(lg.detach().float().flatten().cpu())
                break

    def on_val_end(validator):
        if not stats["n"]:
            return
        act = stats["total"] / stats["n"]
        msg = f"P2 AKTIF: {act * 100:.2f}%"
        row = {"epoch": trainer_ref.epoch + 1, "act_uncalibrated": act}
        if stats["logits"]:
            lg = torch.cat(stats["logits"])
            row.update(n=int(lg.numel()), logit_mean=float(lg.mean()), logit_std=float(lg.std()),
                       logit_p10=float(lg.quantile(0.1)), logit_p90=float(lg.quantile(0.9)),
                       frac_saturated=float((lg.abs() > 9.5).float().mean()))
            msg += f" | logit mean={row['logit_mean']:.3f} std={row['logit_std']:.4f}"
        print(f"\n📉 [EVALUASI ROUTER] {msg}\n")
        if csv_path is not None:
            pd.DataFrame([row]).to_csv(csv_path, mode="a", header=not Path(csv_path).exists(), index=False)

    return on_val_start, on_val_batch_end, on_val_end


def make_branch_b_checker(trainer_ref, interval=10, csv_path=None):
    """Evaluasi Branch B tiap `interval` epoch; hasil ke csv_path (default: save_dir/branch_b_metrics.csv)."""
    def on_val_end_check_branch_b(validator):
        epoch = trainer_ref.epoch + 1
        if epoch % interval != 0:
            return
        path = Path(csv_path or Path(trainer_ref.save_dir) / "branch_b_metrics.csv")
        vb = DualBranchValidatorB(dataloader=trainer_ref.test_loader, save_dir=trainer_ref.save_dir,
                                  args=copy(trainer_ref.args))
        try:
            vb(trainer=trainer_ref)
            bm = vb.metrics.box
            print(f"🔍 [BRANCH B] Epoch {epoch} | mAP50: {bm.map50:.4f} | mAP50-95: {bm.map:.4f} | "
                  f"P: {bm.mp:.4f} | R: {bm.mr:.4f}")
            new = not path.exists()
            with open(path, "a", newline="") as f:
                w = csv.writer(f)
                if new:
                    w.writerow(["epoch", "mAP50", "mAP50-95", "precision", "recall"])
                w.writerow([epoch, bm.map50, bm.map, bm.mp, bm.mr])
        except Exception as e:  # jangan hentikan training karena evaluasi tambahan
            import traceback
            print(f"⚠️  [BRANCH B CHECK GAGAL] {type(e).__name__}: {e}")
            traceback.print_exc()

    return on_val_end_check_branch_b


# ==============================================================================
# BANGUN TRAINER (dipakai run utama DAN smoke test)
# ==============================================================================
def build_trainer(overrides, sched, router_cfg=ROUTER_CFG, branch_b_interval=10,
                  gate_check_epoch="auto", stop_on_nogo=True, criterion_cfg=CRITERION_CFG,
                  activation_stop_rule=None, health_stop_rule=None, auc_stop_rule=None):
    """
    Returns (trainer, ctx). ctx berisi objek log yang bisa diperiksa setelah train():
      timer, sched_history, loss_items_cb, paths (semua di trainer.save_dir).
    Catatan: save_dir sudah ditentukan di konstruktor trainer, jadi path log aman dipakai.

    gate_check_epoch: epoch go/no-go ranking gate ("auto" = sched["lam_start"], tepat sebelum
    lambda naik; None = mati). NO-GO + stop_on_nogo -> training berhenti setelah epoch itu.
    activation_stop_rule: dict untuk make_activation_stop_rule (aturan rencana lama, untuk alpha > 0), None = mati.
    health_stop_rule: dict untuk make_logit_health_stop_rule (revisi ke-1, terbukti terlalu longgar), None = mati.
    auc_stop_rule: dict untuk make_auc_stop_rule (revisi ke-2: AUC ranking tiap N epoch), None = mati.
    """
    settings.update({"raytune": False})
    resuming = bool(overrides.get("resume"))
    trainer = DualBranchTrainer(overrides=overrides)
    sd = Path(trainer.save_dir); sd.mkdir(parents=True, exist_ok=True)
    paths = dict(router_csv=sd / "router_loss_components.csv", loss_items=sd / "loss_items.csv",
                 timing=sd / ("epoch_timing_resume.csv" if resuming else "epoch_timing.csv"), branch_b=sd / "branch_b_metrics.csv",
                 router_cfg=sd / "router_cfg.json", sched=sd / "sched_history.json",
                 gate_val=sd / "gate_val_stats.csv", gate_check=sd / "gate_check.json",
                 criterion_cfg=sd / "criterion_cfg.json", auc_monitor=sd / "auc_monitor.csv")

    sched_history, timer = {}, EpochTimer()
    li_store = []
    li_cb = make_loss_items_logger(li_store, paths["loss_items"])
    r_start, r_batch, r_end = make_router_stat_checker(trainer, paths["gate_val"])
    if gate_check_epoch == "auto":
        gate_check_epoch = sched["lam_start"]

    def save_epoch_logs(tr):
        pd.DataFrame(timer.rows).to_csv(paths["timing"], index=False)
        paths["sched"].write_text(json.dumps({str(k): v for k, v in sched_history.items()}, indent=1))

    trainer.add_callback("on_pretrain_routine_end", make_router_cfg_applier(router_cfg, paths["router_cfg"]))
    trainer.add_callback("on_pretrain_routine_end", make_criterion_cfg_applier(criterion_cfg, paths["criterion_cfg"]))
    trainer.add_callback("on_train_epoch_start", make_curriculum_scheduler(**sched, history=sched_history))
    trainer.add_callback("on_train_epoch_start", timer.on_train_epoch_start)
    trainer.add_callback("on_train_batch_end", li_cb)
    trainer.add_callback("on_train_epoch_end", timer.on_train_epoch_end)          # sebelum flush CSV
    trainer.add_callback("on_train_epoch_end", make_router_csv_logger(paths["router_csv"], allow_existing=resuming))
    trainer.add_callback("on_train_epoch_end", li_cb.flush)
    trainer.add_callback("on_val_start", r_start)
    trainer.add_callback("on_val_batch_end", r_batch)
    trainer.add_callback("on_val_end", r_end)
    trainer.add_callback("on_val_end", timer.timed(make_branch_b_checker(trainer, branch_b_interval, paths["branch_b"])))
    trainer.add_callback("on_fit_epoch_end", timer.on_fit_epoch_end)
    trainer.add_callback("on_fit_epoch_end", save_epoch_logs)
    if gate_check_epoch:
        trainer.add_callback("on_fit_epoch_end", make_gate_ranking_check(
            trainer, gate_check_epoch, paths["gate_check"], signal_mode=router_cfg.get("gate_signal_mode", "small"),
            stop_on_nogo=stop_on_nogo))
    if activation_stop_rule:
        trainer.add_callback("on_fit_epoch_end", make_activation_stop_rule(paths["gate_val"], **activation_stop_rule))
    if health_stop_rule:
        trainer.add_callback("on_fit_epoch_end", make_logit_health_stop_rule(paths["gate_val"], **health_stop_rule))
    if auc_stop_rule:
        trainer.add_callback("on_fit_epoch_end", make_auc_stop_rule(
            trainer, paths["auc_monitor"], signal_mode=router_cfg.get("gate_signal_mode", "total"), **auc_stop_rule))

    ctx = dict(timer=timer, sched_history=sched_history, loss_items_cb=li_cb, paths=paths,
               sched=sched, router_cfg=router_cfg, criterion_cfg=criterion_cfg, overrides=overrides,
               gate_check_epoch=gate_check_epoch)
    return trainer, ctx


def main():
    print("🔥 SKENARIO 3: Router-P2 YOLOv8n Dual-Branch (Difficulty-Aware Router)")
    trainer, ctx = build_trainer(FULL_OVERRIDES, FULL_SCHED, ROUTER_CFG, branch_b_interval=10,
                                 auc_stop_rule=AUC_STOP_RULE)
    trainer.train()
    return trainer, ctx


if __name__ == "__main__":
    main()

In [ ]:
# [6] PATCH SUMBER (idempotent, EOL-aware) — hapus sel ini setelah kedua patch di-commit ke repo
#   P2   block.py : simpan keputusan & logit gate mentah per-gambar di forward eval (uji 0.5 + log validasi).
#                   Di dalam blok `not torch.jit.is_tracing()` -> graf export tidak berubah.
#   CONF loss.py  : difficulty score = entropy + var + conf (tanda minus lama = salah ketik, 2026-10-09).
sys.path.insert(0, str(REPO))
import router_train_utils as U
for P_ in (U.PATCH_P2, U.PATCH_CONF_SIGN):
    print(U.apply_source_patch(REPO / P_["rel"], P_["old"], P_["new"], P_["done_marker"], P_["name"]))
PROV["git_diff_sha256"] = hashlib.sha256(sh(["git", "diff"], cwd=REPO).encode()).hexdigest()
PROV["git_status"] = sh(["git", "status", "--short"], cwd=REPO)
print("git status:\n" + PROV["git_status"])

In [ ]:
# [7] VERIFIKASI VERSI KODE — berhenti di sini bila sesi ini tidak memakai kode yang dimaksud
import inspect
sys.path.insert(0, str(REPO)); os.chdir(REPO)
import torch, ultralytics
import ultralytics.nn.modules.block as _blk
import ultralytics.model_yolov8_router as _mr
import ultralytics.utils.loss as _ls
_src_r = inspect.getsource(_blk.UltraLightWeightDifficultyAwareRouter)
CODE_FEATURES = {
    "ultralytics di-import dari repo (bukan pip)": Path(ultralytics.__file__).resolve().is_relative_to(REPO.resolve()),
    "AUDIT-FIX #9 kalibrasi = buffer int64": "_calib_T_micro" in _src_r,
    "GroupNorm di classifier gate": "GroupNorm(1, hidden_dim // 2)" in _src_r,
    "top-k spatial pool di gate": "_topk_spatial_pool" in _src_r,
    "patch P2 keputusan + logit gate per-gambar": "_last_gate_logit_raw" in _src_r,
    "fix tanda conf (+) di difficulty score": "entropy_f + var_f + conf_f" in inspect.getsource(_ls.v8DetectionLoss),
    "AUDIT-FIX #6 kolom GV tanpa lag": hasattr(_mr.DualBranchDetectionLoss, "_flush_pending_log"),
    "AUDIT-FIX #3 advantage di-clip": "gate_adv_clip" in inspect.getsource(_mr.DualBranchDetectionLoss),
    "AUDIT-FIX #7 normalisasi bg anchor": "normalize_bg_anchors" in inspect.getsource(_ls.v8DetectionLoss),
}
for k, v in CODE_FEATURES.items():
    print(("✅ " if v else "❌ ") + k)
PROV.update(torch=torch.__version__, ultralytics=ultralytics.__version__, python=platform.python_version(),
            cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
            code_features=CODE_FEATURES)
print({k: PROV[k] for k in ("torch", "ultralytics", "cuda", "gpu")})
assert all(CODE_FEATURES.values()), "Versi kode BUKAN yang dimaksud — push/clone ulang, lalu Restart & Run All."
assert torch.cuda.is_available() or "CFG_OVERRIDE" in globals(), "GPU tidak aktif (Settings → Accelerator → GPU T4)."

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/tahap2_eval.py
# [8] modul (hapus sel ini setelah file di-commit ke repo)
"""
tahap2_eval.py — Tahap 2: kalibrasi & protokol evaluasi Router-P2 Dual-Branch (satu checkpoint per run).

Isi (dipanggil dari notebook tahap2_eval.ipynb):
  - Muat checkpoint (state_dict -> model baru, kalibrasi persisten), stage 1/2A/2B (sama dgn deployment)
  - cache_split(): SATU forward per gambar -> logit gate mentah z, output Branch A & B, loss per-sample,
    info GT. Semua mode evaluasi (A paksa, B paksa, ter-route pada threshold apa pun, acak, oracle)
    dirakit dari cache yang sama -> NMS/validator identik, biaya murah.
  - Sweep titik operasi (router vs acak) di val -> r*; kalibrasi persentil (T=1, b=-q).
  - COCO (pycocotools) + bootstrap berpasangan per gambar (AP, AP50, AP_small) dari evalImgs
    tanpa menjalankan ulang COCOeval.evaluate().
  - Perilaku gate (RQ3) dgn z (peringkat), koreksi Holm; diagnostik BatchNorm gate (close_mosaic).
"""
import json
import sys
import time
import copy
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from router_train_utils import find_router, _raw_gate_logit, _auc_with_ci


# =============================================================================
# MODEL & STAGE
# =============================================================================
def model_fingerprint(m, skip_calib=True):
    """Hash state_dict. Buffer kalibrasi dikecualikan (berubah sah saat kalibrasi)."""
    import hashlib
    h = hashlib.sha256()
    for k, v in sorted(m.state_dict().items()):
        if skip_calib and "_calib_" in k:
            continue
        h.update(k.encode()); h.update(v.detach().float().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def set_calibration(router, T, b):
    router.calib_T = float(T); router.calib_b = float(b)


def get_calibration(router):
    return float(router.calib_T), float(router.calib_b)


def load_dualbranch(path, model_yaml, nc, names, device):
    """
    Muat checkpoint Ultralytics (last.pt ter-strip: 'model' = EMA) ke DualBranchDetectionModel baru
    lewat state_dict (strict=False hanya untuk buffer kalibrasi). Returns (model eval, info).
    """
    from ultralytics.model_yolov8_router import DualBranchDetectionModel
    from ultralytics.cfg import get_cfg
    from ultralytics.utils import DEFAULT_CFG
    ck = torch.load(path, map_location="cpu", weights_only=False)
    key = "ema" if ck.get("ema") is not None else "model"
    src = ck[key]
    sd = {k: (v.float() if v.is_floating_point() else v) for k, v in src.state_dict().items()}
    m = DualBranchDetectionModel(cfg=model_yaml, ch=3, nc=nc, verbose=False)
    missing, unexpected = m.load_state_dict(sd, strict=False)
    bad = [k for k in missing if "calib" not in k]
    assert not bad and not unexpected, f"State dict tidak cocok: missing={bad[:5]} unexpected={list(unexpected)[:5]}"
    hyp = ck.get("train_args") or {}
    m.args = get_cfg(DEFAULT_CFG, overrides={k: hyp[k] for k in ("box", "cls", "dfl") if k in hyp})
    m.names = names
    m = m.to(device).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    info = dict(ckpt=str(path), weights_key=key, epoch=ck.get("epoch"), train_args_name=hyp.get("name"),
                train_args_seed=hyp.get("seed"), calib_in_ckpt=get_calibration(find_router(m)))
    return m, info


class Stage1_BackboneNeckGate(nn.Module):
    """Layer 0-15 + gate (compute_gate_only: kalibrasi + threshold)."""
    def __init__(self, full_model):
        super().__init__()
        ml = list(full_model.model)
        self.pre_router_layers = nn.ModuleList(ml[0:16])
        self.router = ml[16]
        assert "DifficultyAwareRouter" in type(self.router).__name__

    def forward(self, x):
        y = []
        for m in self.pre_router_layers:
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y.append(x)
        p3_neck, p2_backbone, p4_neck, p5_sppf = x, y[2], y[12], y[9]
        gate = self.router.compute_gate_only(p3_neck, p2_backbone)
        return gate, p3_neck, p2_backbone, p4_neck, p5_sppf


class Stage2A_WithP2(nn.Module):
    """Layer 16 (expert) + 17-25 + Detect_A -> y (B, 4+nc, N_A)."""
    def __init__(self, full_model):
        super().__init__()
        ml = list(full_model.model)
        self.router, self.branch_a_layers, self.detect_A = ml[16], nn.ModuleList(ml[17:26]), ml[32]

    def forward(self, p3_neck, p2_backbone, p4_neck, p5_sppf):
        p2_out = self.router.compute_expert_only(p3_neck, p2_backbone)
        y = {15: p3_neck, 12: p4_neck, 9: p5_sppf, 16: p2_out}
        x = p2_out
        for i, m in zip(range(17, 26), self.branch_a_layers):
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y[i] = x
        out = self.detect_A([p2_out, y[19], y[22], y[25]])
        return out[0] if isinstance(out, tuple) else out


class Stage2B_NoP2(nn.Module):
    """Layer 26-31 + Detect_B -> y (B, 4+nc, N_B)."""
    def __init__(self, full_model):
        super().__init__()
        ml = list(full_model.model)
        self.branch_b_layers, self.detect_B = nn.ModuleList(ml[26:32]), ml[33]

    def forward(self, p3_neck, p4_neck, p5_sppf):
        y = {15: p3_neck, 12: p4_neck, 9: p5_sppf}
        x = p3_neck
        for i, m in zip(range(26, 32), self.branch_b_layers):
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y[i] = x
        out = self.detect_B([p3_neck, y[28], y[31]])
        return out[0] if isinstance(out, tuple) else out


def make_stages(model):
    s1, s2a, s2b = Stage1_BackboneNeckGate(model).eval(), Stage2A_WithP2(model).eval(), Stage2B_NoP2(model).eval()
    return s1, s2a, s2b


@torch.no_grad()
def check_stage_equivalence(model, stages, loader, device, n=2, tol=1e-5):
    s1, s2a, s2b = stages
    for k, batch in enumerate(loader):
        if k >= n:
            break
        im = batch["img"].to(device).float() / 255.0
        _, p3, p2, p4, p5 = s1(im)
        yA_ref, yB_ref = (o[0] for o in model._predict_once_dual(im))
        dA = ((s2a(p3, p2, p4, p5) - yA_ref).abs().max() / yA_ref.abs().max()).item()
        dB = ((s2b(p3, p4, p5) - yB_ref).abs().max() / yB_ref.abs().max()).item()
        assert dA < tol and dB < tol, f"Stage tidak setara dgn model utuh: {dA:.2e} / {dB:.2e}"
    return True


# =============================================================================
# DATA & VALIDATOR
# =============================================================================
def make_loader(data, split, batch, imgsz=640, workers=2, pin_memory=None):
    """
    Letterbox persegi imgsz, tanpa augmentasi, urutan tetap (sama dengan engine TensorRT fixed-shape).

    SENGAJA DataLoader biasa, BUKAN build_dataloader Ultralytics: InfiniteDataLoader memakai satu iterator
    persisten, sehingga loop yang di-`break` lebih awal (uji kesetaraan, verifikasi ONNX, next(iter(...)))
    menggeser urutan SEMUA iterasi berikutnya (rotasi). Akurasi tetap benar (GT ikut batch), tetapi apa pun
    yang menyejajarkan hasil per gambar dengan dataset.labels / run lain menjadi tidak sejajar.
    """
    from torch.utils.data import DataLoader
    from ultralytics.data import YOLODataset
    ds = YOLODataset(img_path=data[split], data=data, task="detect", imgsz=imgsz,
                     augment=False, rect=False, cache=False, batch_size=batch)
    return DataLoader(ds, batch_size=batch, shuffle=False, num_workers=workers, collate_fn=ds.collate_fn,
                      pin_memory=torch.cuda.is_available() if pin_memory is None else pin_memory, drop_last=False)


def make_validator_cls():
    from ultralytics.models.yolo.detect import DetectionValidator
    from ultralytics.utils.nms import non_max_suppression

    class ListPredValidator(DetectionValidator):
        """Validator resmi Ultralytics yang menerima daftar prediksi per gambar (NMS & threshold identik)."""
        def postprocess(self, preds):
            out = []
            for pred in preds:
                x = non_max_suppression(pred, self.args.conf, self.args.iou, nc=0, multi_label=True,
                                        agnostic=False, max_det=self.args.max_det)[0]
                out.append({"bboxes": x[:, :4], "conf": x[:, 4], "cls": x[:, 5], "extra": x[:, 6:]})
            return out

        def scale_preds(self, predn, pbatch):
            """
            Skala letterbox -> resolusi asli PER SUMBU (hanya untuk JSON/pycocotools; metrik Ultralytics dihitung
            di ruang letterbox dan tidak terpengaruh). ops.scale_boxes bawaan memakai satu gain = rasio TINGGI untuk
            kedua sumbu; pada KITTI (1242x375 -> 640x194, tinggi dibulatkan dari 193.2) rasio tinggi 0.51733 vs
            lebar 0.51530, sehingga koordinat x menyusut 0.39% (±4 px di tepi kanan) -> AP COCO turun, terutama
            objek kecil & IoU tinggi.
            """
            (gh, gw), (px, py) = pbatch["ratio_pad"]
            b = predn["bboxes"].clone()
            b[:, [0, 2]] = (b[:, [0, 2]] - px) / gw
            b[:, [1, 3]] = (b[:, [1, 3]] - py) / gh
            h0, w0 = pbatch["ori_shape"]
            b[:, [0, 2]] = b[:, [0, 2]].clamp(0, w0); b[:, [1, 3]] = b[:, [1, 3]].clamp(0, h0)
            return {**predn, "bboxes": b}
    return ListPredValidator


def run_eval(name, predict_fn, loader, data, cfg, device, save_json=False, verbose=True):
    """predict_fn(img_float_0_1) -> (list[(1,4+nc,N)], list[dict info per gambar])."""
    V = make_validator_cls()
    v = V(args=dict(data=cfg["data_yaml"], conf=cfg["conf"], iou=cfg["iou"], max_det=cfg["max_det"],
                    split=cfg.get("report_split", "test"), imgsz=cfg["imgsz"], batch=1, half=False, plots=False,
                    save_json=save_json, task="detect", single_cls=False, agnostic_nms=False))
    v.data, v.device = data, device
    v.init_metrics(SimpleNamespace(names=data["names"], end2end=False))
    records = []
    with torch.no_grad():
        for batch in loader:
            batch = v.preprocess(batch)
            preds, infos = predict_fn(batch["img"])
            v.update_metrics(v.postprocess(preds), batch)
            for f, info in zip(batch["im_file"], infos):
                records.append({"file": Path(f).name, **info})
    stats = v.get_stats()
    if verbose:
        print(f"--- {name} ---"); v.print_results()
    assert v.seen == len(loader.dataset), f"{name}: {v.seen} gambar dievaluasi, dataset {len(loader.dataset)}"
    return dict(name=name, stats=stats, records=records, jdict=list(v.jdict) if save_json else None, seen=v.seen)


def summarize_acc(r):
    s = r["stats"]
    return {"mAP50": s["metrics/mAP50(B)"], "mAP50-95": s["metrics/mAP50-95(B)"],
            "P": s["metrics/precision(B)"], "R": s["metrics/recall(B)"], "n_images": r["seen"]}


# =============================================================================
# CACHE PER GAMBAR (satu forward; semua mode evaluasi dirakit dari sini)
# =============================================================================
@torch.no_grad()
def cache_split(model, stages, loader, device, keep_outputs=True):
    """
    Per gambar: z (logit gate mentah), output Stage2A & Stage2B (CPU), loss per-sample A/B (total & small),
    jumlah objek kecil ter-assign, info GT (resolusi asli, definisi COCO). Loader harus batch=1, shuffle=False.
    """
    s1, s2a, s2b = stages
    rt = find_router(model)
    crit = model.init_criterion()
    labels = loader.dataset.labels
    out = dict(file=[], z=[], yA=[], yB=[], la=[], lb=[], las=[], lbs=[], nsa=[], nsb=[],
               n_obj=[], n_small=[], small_ratio=[], mean_area=[])
    for i, batch in enumerate(loader):
        assert batch["img"].shape[0] == 1, "cache_split butuh loader batch=1"
        f = batch["im_file"][0]
        assert Path(f).name == Path(labels[i]["im_file"]).name, "Urutan loader != dataset.labels"
        im = batch["img"].to(device).float() / 255.0
        _, p3, p2, p4, p5 = s1(im)
        out["z"].append(float(_raw_gate_logit(rt, p3, p2)[0]))
        if keep_outputs:
            out["yA"].append(s2a(p3, p2, p4, p5).float().cpu())
            out["yB"].append(s2b(p3, p4, p5).float().cpu())
        b = {"img": im, **{k: batch[k].to(device) for k in ("cls", "bboxes", "batch_idx")}}
        det_A, det_B = model._predict_once_dual(im)
        crit.loss_A(det_A, b); crit.loss_B(det_B, b)
        out["la"].append(float(crit.loss_A._last_per_sample_loss[0]))
        out["lb"].append(float(crit.loss_B._last_per_sample_loss[0]))
        out["las"].append(float(crit.loss_A._last_per_sample_loss_small[0]))
        out["lbs"].append(float(crit.loss_B._last_per_sample_loss_small[0]))
        out["nsa"].append(float(crit.loss_A._last_small_obj_count[0]))
        out["nsb"].append(float(crit.loss_B._last_small_obj_count[0]))
        lab = labels[i]
        h, w = lab["shape"]
        bx = lab["bboxes"].reshape(-1, 4)
        area = bx[:, 2] * w * bx[:, 3] * h
        out["file"].append(Path(f).name)
        out["n_obj"].append(int(len(bx)))
        out["n_small"].append(int((area < 32 ** 2).sum()))
        out["small_ratio"].append(float((area < 32 ** 2).mean()) if len(bx) else np.nan)
        out["mean_area"].append(float(area.mean()) if len(bx) else np.nan)
    for k in ("z", "la", "lb", "las", "lbs", "nsa", "nsb", "n_obj", "n_small", "small_ratio", "mean_area"):
        out[k] = np.asarray(out[k], dtype=np.float64)
    out["oracle_A"] = out["la"] < out["lb"]                       # 1 = Branch A lebih baik (sinyal training 'total')
    out["valid_small"] = (out["nsa"] > 0) & (out["nsb"] > 0)
    out["oracle_A_small"] = out["las"] < out["lbs"]
    return out


def predict_from_cache(cache, use_a, device):
    """predict_fn untuk run_eval dari cache; loader harus urutan sama (dicek lewat records)."""
    pos = {"k": 0}

    def fn(im):
        preds, infos = [], []
        for _ in range(im.shape[0]):
            k = pos["k"]; pos["k"] += 1
            a = bool(use_a[k])
            # clone WAJIB: non_max_suppression Ultralytics menulis xywh->xyxy IN-PLACE pada tensor input;
            # tanpa salinan, cache rusak untuk evaluasi berikutnya (di CPU .to() tidak menyalin).
            preds.append((cache["yA"][k] if a else cache["yB"][k]).to(device, copy=True))
            infos.append({"branch": "A" if a else "B", "z": float(cache["z"][k])})
        return preds, infos
    return fn


def eval_assignment(name, cache, use_a, loader, data, cfg, device, save_json=False, verbose=False):
    r = run_eval(name, predict_from_cache(cache, use_a, device), loader, data, cfg, device,
                 save_json=save_json, verbose=verbose)
    assert [x["file"] for x in r["records"]] == cache["file"], "Urutan loader != cache"
    r["activation"] = float(np.mean(use_a))
    r["use_a"] = np.asarray(use_a, bool)
    return r


def random_assignment(n, k, seed):
    rng = np.random.default_rng(seed)
    ua = np.zeros(n, bool)
    ua[rng.choice(n, k, replace=False)] = True
    return ua


# =============================================================================
# TITIK OPERASI & KALIBRASI PERSENTIL
# =============================================================================
def fit_percentile_threshold(raw_logits, target_activation):
    """T = 1, b = -q dgn q = kuantil (1 - rho) logit val -> aktivasi val = rho (selain ties)."""
    assert 0.0 < target_activation < 1.0
    z = torch.as_tensor(raw_logits, dtype=torch.float64)
    q = torch.quantile(z, 1.0 - target_activation).item()
    return dict(calib_T=1.0, calib_b=-q, threshold=q, target_activation=float(target_activation),
                activation_on_calib_split=float((z > q).double().mean()),
                n_unique_logits=int(z.unique().numel()), n=int(z.numel()))


def sweep(cache, loader, data, cfg, device, thr_source, rates, n_random=3, seed=0, verbose=False):
    """
    Router (threshold = kuantil logit thr_source) vs acak (aktivasi sama, n_random undian) per rate.
    thr_source = logit VAL, juga saat mengevaluasi test (threshold tidak pernah dari test).
    """
    z = cache["z"]; n = len(z); rows = []
    for r in rates:
        thr = -np.inf if r >= 1 else (np.inf if r <= 0 else float(np.quantile(thr_source, 1 - r)))
        use_a = z > thr
        k = int(use_a.sum())
        acc = summarize_acc(eval_assignment(f"router r={r:.2f}", cache, use_a, loader, data, cfg, device, verbose=verbose))
        rnd = []
        for j in range(n_random if 0 < k < n else 0):
            ua = random_assignment(n, k, seed + 1000 * j + int(r * 100))
            rnd.append(summarize_acc(eval_assignment(f"acak r={k / n:.2f}", cache, ua, loader, data, cfg, device))["mAP50"])
        rows.append(dict(rate_target=r, rate_realized=k / n, threshold=thr if np.isfinite(thr) else None,
                         mAP50=acc["mAP50"], mAP50_95=acc["mAP50-95"],
                         acak_mAP50=float(np.mean(rnd)) if rnd else acc["mAP50"],
                         acak_sd=float(np.std(rnd)) if rnd else 0.0, n_acak=len(rnd)))
    df = pd.DataFrame(rows)
    df["gain_vs_acak"] = df.mAP50 - df.acak_mAP50
    return df


def choose_r_star(df_val):
    """Aturan terkunci: r* = argmax (mAP50 router - mAP50 acak) pada 0 < rho < 1, di VAL."""
    inner = df_val[(df_val.rate_target > 0) & (df_val.rate_target < 1)]
    return float(inner.loc[inner.gain_vs_acak.idxmax(), "rate_target"])


# =============================================================================
# COCO + BOOTSTRAP BERPASANGAN PER GAMBAR
# =============================================================================
def build_coco_gt(dataset, names, path):
    """GT COCO resolusi asli; category_id = cls+1, image_id = int(stem) (= pred_to_json Ultralytics)."""
    from pycocotools.coco import COCO
    images, anns, aid = [], [], 1
    for lab in dataset.labels:
        stem = Path(lab["im_file"]).stem
        iid = int(stem) if stem.isnumeric() else stem
        h, w = lab["shape"]
        images.append({"id": iid, "file_name": Path(lab["im_file"]).name, "height": int(h), "width": int(w)})
        for c, (cx, cy, bw, bh) in zip(lab["cls"].reshape(-1), lab["bboxes"].reshape(-1, 4)):
            bw_px, bh_px = float(bw * w), float(bh * h)
            anns.append({"id": aid, "image_id": iid, "category_id": int(c) + 1,
                         "bbox": [float(cx * w - bw_px / 2), float(cy * h - bh_px / 2), bw_px, bh_px],
                         "area": bw_px * bh_px, "iscrowd": 0})
            aid += 1
    gt = {"images": images, "annotations": anns, "categories": [{"id": i + 1, "name": str(n)} for i, n in names.items()]}
    Path(path).write_text(json.dumps(gt))
    return COCO(str(path))


COCO_KEYS = ["mAP50-95", "mAP50", "mAP75", "mAP_small", "mAP_medium", "mAP_large",
             "AR1", "AR10", "AR100", "AR_small", "AR_medium", "AR_large"]


# maxDets COCOeval. Default COCO [1, 10, 100] (KITTI ±5 objek/gambar). Untuk dataset padat (VisDrone ±59 objek/gambar,
# sebagian >100) set mis. [1, 10, 500] SEBELUM evaluasi; elemen terakhir dipakai untuk AP di summarize()
# (index 2) maupun di CocoBootstrap, jadi keduanya tetap konsisten. Kolom "AR100" lalu berarti AR@maxDets[2].
COCO_MAX_DETS = [1, 10, 100]


def coco_eval(coco_gt, jdict, quiet=True):
    """COCOeval lengkap (maxDets = COCO_MAX_DETS). Returns (stats dict, objek COCOeval — evalImgs dipakai bootstrap)."""
    assert len(COCO_MAX_DETS) == 3, "summarize() pycocotools mengasumsikan tepat 3 nilai maxDets"
    import contextlib, io
    from pycocotools.cocoeval import COCOeval
    ids = set(coco_gt.getImgIds())
    assert {d["image_id"] for d in jdict} <= ids, "image_id prediksi tidak ada di GT"
    buf = io.StringIO()
    with (contextlib.redirect_stdout(buf) if quiet else contextlib.nullcontext()):
        if len(jdict):
            dt = coco_gt.loadRes(jdict)
        else:                                               # tanpa deteksi sama sekali -> semua AP = 0
            from pycocotools.coco import COCO
            dt = COCO(); dt.dataset = {"images": coco_gt.dataset["images"], "categories": coco_gt.dataset["categories"],
                                      "annotations": []}
            dt.createIndex()
        E = COCOeval(coco_gt, dt, "bbox")
        E.params.imgIds = sorted(ids)
        E.params.maxDets = list(COCO_MAX_DETS)
        E.evaluate(); E.accumulate(); E.summarize()
    # pycocotools 2.0.x: stats[0] (mAP50-95) memakai maxDets=100 TERTANAM (bukan maxDets[2]) -> -1 bila 100 tidak
    # ada di daftar. Hitung ulang dari precision pada maxDets[-1], area 'all' (identik untuk default [1,10,100]).
    prec = E.eval["precision"][:, :, :, 0, -1]
    E.stats[0] = float(prec[prec > -1].mean()) if (prec > -1).any() else -1.0
    return dict(zip(COCO_KEYS, map(float, E.stats))), E


class CocoBootstrap:
    """
    AP COCO untuk subset gambar (boleh berulang, untuk bootstrap) langsung dari evalImgs COCOeval:
    replikasi accumulate() + summarize() (maxDet 100) tanpa menjalankan evaluate() ulang.
    Diverifikasi sama dengan COCOeval.stats pada himpunan penuh (lihat verify()).
    """
    AREAS = {"all": 0, "small": 1, "medium": 2, "large": 3}

    def __init__(self, E):
        p = E.params
        self.T, self.R, self.I = len(p.iouThrs), p.recThrs, len(p.imgIds)
        self.A, self.K, self.maxDet = len(p.areaRng), len(p.catIds), p.maxDets[-1]
        self.iou50 = int(np.where(np.isclose(p.iouThrs, 0.5))[0][0])
        self.stats_full = np.asarray(E.stats, float)
        # pra-ekstrak array per (kategori, area, gambar); None = tidak ada GT & deteksi
        self.cells = {}
        for k in range(self.K):
            for name, a in self.AREAS.items():
                base = k * self.A * self.I + a * self.I
                lst = []
                for i in range(self.I):
                    e = E.evalImgs[base + i]
                    if e is None:
                        lst.append(None); continue
                    lst.append((np.asarray(e["dtScores"][:self.maxDet], float),
                                np.asarray(e["dtMatches"][:, :self.maxDet]) > 0,
                                np.asarray(e["dtIgnore"][:, :self.maxDet], bool),
                                int(np.count_nonzero(np.asarray(e["gtIgnore"]) == 0))))
                self.cells[(k, name)] = lst

    def _precision(self, idx, area):
        """List per kategori: array (T, R) presisi terinterpolasi, atau None (kategori tanpa GT valid)."""
        out = []
        for k in range(self.K):
            cell = self.cells[(k, area)]
            E = [cell[i] for i in idx if cell[i] is not None]
            if not E:
                out.append(None); continue
            npig = sum(e[3] for e in E)
            if npig == 0:
                out.append(None); continue
            scores = np.concatenate([e[0] for e in E])
            order = np.argsort(-scores, kind="mergesort")
            dtm = np.concatenate([e[1] for e in E], axis=1)[:, order]
            dtig = np.concatenate([e[2] for e in E], axis=1)[:, order]
            tp = np.cumsum(dtm & ~dtig, axis=1, dtype=float)
            fp = np.cumsum(~dtm & ~dtig, axis=1, dtype=float)
            nd = tp.shape[1]
            q = np.zeros((self.T, len(self.R)))
            if nd:
                rc = tp / npig
                pr = tp / (fp + tp + np.spacing(1))
                pr = np.maximum.accumulate(pr[:, ::-1], axis=1)[:, ::-1]      # presisi monoton (= loop COCO)
                for t in range(self.T):
                    ii = np.searchsorted(rc[t], self.R, side="left")
                    ok = ii < nd
                    q[t, ok] = pr[t, ii[ok]]
            out.append(q)
        return out

    def metrics(self, idx):
        """dict mAP50-95, mAP50 (area all) & mAP_small untuk subset gambar idx (indeks ke E.params.imgIds)."""
        P_all = [p for p in self._precision(idx, "all") if p is not None]
        P_s = [p for p in self._precision(idx, "small") if p is not None]
        m = {}
        if P_all:
            P = np.stack(P_all); m["mAP50-95"] = float(P.mean()); m["mAP50"] = float(P[:, self.iou50].mean())
        else:
            m["mAP50-95"] = m["mAP50"] = np.nan
        m["mAP_small"] = float(np.stack(P_s).mean()) if P_s else np.nan
        return m

    def verify(self, atol=1e-9):
        full = self.metrics(np.arange(self.I))
        ref = {"mAP50-95": self.stats_full[0], "mAP50": self.stats_full[1], "mAP_small": self.stats_full[3]}
        for k, v in ref.items():
            v = np.nan if v < 0 else v
            assert (np.isnan(v) and np.isnan(full[k])) or abs(full[k] - v) <= atol, f"CocoBootstrap {k}: {full[k]} != COCOeval {v}"
        return full


BOOT_METRICS = ("mAP50-95", "mAP50", "mAP_small")


def paired_bootstrap(boot_a, boots_b, n_boot=1000, seed=0):
    """
    Selisih metrik A − rata-rata(B_list) pada resample gambar YANG SAMA (bootstrap berpasangan per gambar).
    boot_a: CocoBootstrap; boots_b: list CocoBootstrap (mis. 3 undian router acak) — semua dari GT yang sama.
    Returns DataFrame: metric, diff, ci_lo, ci_hi (persentil 95%), p_two_sided (proporsi bootstrap melewati 0;
    batas bawah 1/B — tulis 'p < 1/B', bukan p = 0).
    """
    I = boot_a.I
    assert all(b.I == I for b in boots_b)
    rng = np.random.default_rng(seed)

    def diff(idx):
        a = boot_a.metrics(idx)
        bs = [b.metrics(idx) for b in boots_b]
        return {m: a[m] - np.nanmean([b[m] for b in bs]) for m in BOOT_METRICS}

    point = diff(np.arange(I))
    samples = {m: [] for m in BOOT_METRICS}
    for _ in range(n_boot):
        d = diff(rng.integers(0, I, I))
        for m in BOOT_METRICS:
            samples[m].append(d[m])
    rows = []
    for m in BOOT_METRICS:
        s = np.asarray(samples[m]); s = s[np.isfinite(s)]
        lo, hi = np.percentile(s, [2.5, 97.5])
        p = min(1.0, max(2 * min((s <= 0).mean(), (s >= 0).mean()), 1.0 / len(s)))   # resolusi p = 1/B
        rows.append(dict(metric=m, diff=point[m], ci_lo=lo, ci_hi=hi, p_two_sided=p, n_boot=len(s)))
    return pd.DataFrame(rows)


# =============================================================================
# PERILAKU GATE (RQ3)
# =============================================================================
def holm(pvals):
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj


def gate_behaviour(cache):
    """Spearman z vs kepadatan/ukuran, parsial (proporsi kecil | jumlah objek), AUC vs oracle; Holm."""
    from scipy import stats as st
    df = pd.DataFrame({k: cache[k] for k in ("z", "n_obj", "n_small", "small_ratio", "mean_area")})
    tests = []

    def sp(x, y, label):
        m = df[[x, y]].dropna()
        r, p = st.spearmanr(m[x], m[y]); tests.append(dict(test=label, r=r, p=p, n=len(m)))

    sp("z", "n_obj", "Spearman z vs jumlah objek")
    sp("z", "small_ratio", "Spearman z vs proporsi objek kecil")
    sp("z", "n_small", "Spearman z vs jumlah objek kecil")
    sp("z", "mean_area", "Spearman z vs rata-rata luas objek")
    m = df[["z", "small_ratio", "n_obj"]].dropna()
    rx, ry, rz = st.rankdata(m.z), st.rankdata(m.small_ratio), st.rankdata(m.n_obj)
    res = lambda a, b: a - np.polyval(np.polyfit(b, a, 1), b)
    r_part, p_part = st.pearsonr(res(rx, rz), res(ry, rz))
    tests.append(dict(test="Parsial (Spearman) z vs proporsi kecil | jumlah objek (H3)", r=r_part, p=p_part, n=len(m)))
    T = pd.DataFrame(tests); T["p_holm"] = holm(T.p)
    auc = _auc_with_ci(cache["oracle_A"], cache["z"])
    v = cache["valid_small"]
    auc_s = _auc_with_ci(cache["oracle_A_small"][v], cache["z"][v]) if v.sum() >= 20 else (np.nan, np.nan, np.nan)
    A = pd.DataFrame([dict(target="loss_A < loss_B (total)", n=len(cache["z"]), oracle_rate=float(cache["oracle_A"].mean()),
                           auc=auc[0], ci_lo=auc[1], ci_hi=auc[2]),
                      dict(target="loss_A_small < loss_B_small (gambar valid)", n=int(v.sum()),
                           oracle_rate=float(cache["oracle_A_small"][v].mean()) if v.sum() else np.nan,
                           auc=auc_s[0], ci_lo=auc_s[1], ci_hi=auc_s[2])])
    return T, A


# =============================================================================
# DIAGNOSTIK BATCHNORM GATE (hipotesis close_mosaic)
# =============================================================================
def _squeeze_bns(rt):
    return [m for seq in (rt.squeeze_p3, rt.squeeze_p2) for m in seq.modules() if isinstance(m, nn.BatchNorm2d)]


@torch.no_grad()
def gate_logits_on(model, loader, device, max_images=None):
    """z per gambar lewat forward penuh model (hook input router) — mode BN mengikuti status modul."""
    rt = find_router(model)
    feats, zs, files = {}, [], []
    h = rt.register_forward_pre_hook(lambda m, inp: feats.update(p3=inp[0][0], p2=inp[0][1]))
    try:
        n = 0
        for batch in loader:
            im = batch["img"].to(device).float() / 255.0
            model._predict_once_dual(im)
            zs.append(_raw_gate_logit(rt, feats["p3"], feats["p2"]).cpu().numpy())
            files += [Path(f).name for f in batch["im_file"]]
            n += im.shape[0]
            if max_images and n >= max_images:
                break
    finally:
        h.remove()
    return np.concatenate(zs), files


@torch.no_grad()
def bn_diagnostic(model, val_cache, val_loader_b16, train_loader_b16, device, n_train=1297):
    """
    Bandingkan peringkat gate (AUC vs oracle 'total' val) & sebaran logit di val untuk:
      (a) eval (running stats)             — referensi, harus = cache
      (b) BN squeeze memakai statistik batch val (batch 16) — meniru mode training
      (c) running stats BN squeeze diestimasi ulang di gambar TRAIN tanpa augmentasi (kumulatif)
      (d) gambar TRAIN tanpa augmentasi, mode eval — apakah gate berbeda di data training?
    Semua pada salinan model; model asli tidak berubah.
    """
    rows = []
    oracle = dict(zip(val_cache["file"], val_cache["oracle_A"]))

    def row(name, z, files, labels=None):
        y = np.array([oracle[f] for f in files]) if labels is None else labels
        a = _auc_with_ci(y, z, n_boot=500)
        rows.append(dict(variant=name, n=len(z), logit_mean=float(np.mean(z)), logit_std=float(np.std(z)),
                         act_z_gt_0=float(np.mean(z > 0)), auc=a[0], ci_lo=a[1], ci_hi=a[2]))

    z, f = gate_logits_on(model, val_loader_b16, device); row("(a) val, eval (running stats)", z, f)
    mb = copy.deepcopy(model)
    for bn in _squeeze_bns(find_router(mb)):
        bn.train()
    z, f = gate_logits_on(mb, val_loader_b16, device); row("(b) val, BN squeeze statistik batch", z, f)
    mc = copy.deepcopy(model); rt = find_router(mc)
    for bn in _squeeze_bns(rt):
        bn.reset_running_stats(); bn.momentum = None; bn.train()
    gate_logits_on(mc, train_loader_b16, device, max_images=n_train)       # hanya mengisi running stats
    for bn in _squeeze_bns(rt):
        bn.eval()
    z, f = gate_logits_on(mc, val_loader_b16, device); row("(c) val, running stats BN squeeze diestimasi ulang di train", z, f)
    return pd.DataFrame(rows)


def train_split_gate(model, train_loader_b1, device, max_images=1297):
    """(d) Gate & oracle di gambar train tanpa augmentasi, mode eval (dipotong max_images)."""
    stages = make_stages(model)
    sub = _LimitedLoader(train_loader_b1, max_images)
    c = cache_split(model, stages, sub, device, keep_outputs=False)
    a = _auc_with_ci(c["oracle_A"], c["z"], n_boot=500)
    return dict(variant="(d) train tanpa augmentasi, eval", n=len(c["z"]), logit_mean=float(c["z"].mean()),
                logit_std=float(c["z"].std()), act_z_gt_0=float((c["z"] > 0).mean()), auc=a[0], ci_lo=a[1], ci_hi=a[2])


class _LimitedLoader:
    """Bungkus loader: berhenti setelah n gambar; dataset.labels dipotong agar cache_split tetap sinkron."""
    def __init__(self, loader, n):
        self.loader, self.n = loader, n
        self.dataset = SimpleNamespace(labels=loader.dataset.labels[:n])

    def __iter__(self):
        for i, b in enumerate(self.loader):
            if i >= self.n:
                break
            yield b


# =============================================================================
# BASELINE STATIS (Vanilla, P2-Static)
# =============================================================================
def load_static(path, device):
    from ultralytics import YOLO
    m = YOLO(path).model.float().to(device).eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m


def predict_static_pt(m):
    @torch.no_grad()
    def fn(im):
        preds = []
        for i in range(im.shape[0]):
            out = m(im[i:i + 1])
            preds.append(out[0] if isinstance(out, (tuple, list)) else out)
        return preds, [{} for _ in range(im.shape[0])]
    return fn


# =============================================================================
# ONNX & TENSORRT (FP32, 3 engine router + 2 baseline; sama dgn notebook uji 2026-10-08)
# =============================================================================
class _Stage1Out(nn.Module):
    def __init__(self, s):
        super().__init__(); self.s = s

    def forward(self, x):
        return self.s(x)


class _StaticOut(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m

    def forward(self, x):
        out = self.m(x)
        return out[0] if isinstance(out, (tuple, list)) else out


def export_onnx(module, args, path, in_names, out_names):
    with torch.no_grad():
        torch.onnx.export(module.eval(), args, str(path), input_names=in_names, output_names=out_names,
                          opset_version=17, dynamic_axes=None, dynamo=False)


def cpu_copy(m):
    """
    Salinan model di CPU yang aman untuk ekspor. Detect menyimpan anchors/strides sebagai ATRIBUT biasa (bukan
    buffer) dan hanya menghitung ulang bila shape input berubah; .cpu() tidak memindahkannya, sehingga salinan
    CPU dari model yang pernah jalan di GPU masih memegang anchors cuda:0 ("Expected all tensors to be on the
    same device"). shape=None memaksa anchors dihitung ulang di device yang benar pada forward berikutnya.
    """
    c = copy.deepcopy(m).cpu().eval()
    for mod in c.modules():
        if hasattr(mod, "anchors") and hasattr(mod, "shape"):
            mod.shape = None
            mod.anchors, mod.strides = torch.empty(0), torch.empty(0)
    return c


def export_all_onnx(model, vanilla, p2static, x0, out_dir, tag):
    """x0: (1,3,H,W) float CPU. Returns dict path ONNX + salinan CPU stage (untuk verifikasi)."""
    m_cpu = cpu_copy(model)
    assert get_calibration(find_router(m_cpu)) == get_calibration(find_router(model)), "Kalibrasi tidak ikut tersalin"
    s1c, s2ac, s2bc = make_stages(m_cpu)
    with torch.no_grad():
        _, p3, p2, p4, p5 = s1c(x0)
    P = {k: Path(out_dir) / f"{k}_{tag}.onnx" for k in ("stage1", "stage2a", "stage2b")}
    P["vanilla"], P["p2static"] = Path(out_dir) / "vanilla.onnx", Path(out_dir) / "p2static.onnx"
    export_onnx(_Stage1Out(s1c), (x0,), P["stage1"], ["image"], ["gate", "p3_neck", "p2_backbone", "p4_neck", "p5_sppf"])
    export_onnx(s2ac, (p3, p2, p4, p5), P["stage2a"], ["p3_neck", "p2_backbone", "p4_neck", "p5_sppf"], ["y_A"])
    export_onnx(s2bc, (p3, p4, p5), P["stage2b"], ["p3_neck", "p4_neck", "p5_sppf"], ["y_B"])
    statics = {"vanilla": _StaticOut(cpu_copy(vanilla)).eval(), "p2static": _StaticOut(cpu_copy(p2static)).eval()}
    for k, mod in statics.items():
        export_onnx(mod, (x0,), P[k], ["image"], ["y"])
    return P, (s1c, s2ac, s2bc), statics


def verify_onnx(P, cpu_stages, statics, loader, n=5):
    import onnxruntime as ort
    sess = {k: ort.InferenceSession(str(p), providers=["CPUExecutionProvider"]) for k, p in P.items()}
    run = lambda k, feed: sess[k].run(None, {i.name: feed[i.name] for i in sess[k].get_inputs()})
    rel = lambda a, b: float(np.abs(a - b).max() / (np.abs(b).max() + 1e-12))
    s1c, s2ac, s2bc = cpu_stages
    worst = {}
    with torch.no_grad():
        for k, batch in enumerate(loader):
            if k >= n:
                break
            x = batch["img"].float() / 255.0
            g, p3, p2, p4, p5 = s1c(x)
            o1 = run("stage1", {"image": x.numpy()})
            assert float(o1[0].flatten()[0]) == float(g.flatten()[0]), f"Gate ONNX != PyTorch pada {batch['im_file'][0]}"
            worst["stage1"] = max(worst.get("stage1", 0), *[rel(a, b.numpy()) for a, b in zip(o1[1:], (p3, p2, p4, p5))])
            f = {"p3_neck": p3.numpy(), "p2_backbone": p2.numpy(), "p4_neck": p4.numpy(), "p5_sppf": p5.numpy()}
            worst["stage2a"] = max(worst.get("stage2a", 0), rel(run("stage2a", f)[0], s2ac(p3, p2, p4, p5).numpy()))
            worst["stage2b"] = max(worst.get("stage2b", 0), rel(run("stage2b", f)[0], s2bc(p3, p4, p5).numpy()))
            for nm, mod in statics.items():
                worst[nm] = max(worst.get(nm, 0), rel(run(nm, {"image": x.numpy()})[0], mod(x).numpy()))
    assert all(v < 1e-4 for v in worst.values()), f"Selisih ONNX vs PyTorch > 1e-4: {worst}"
    return worst


def init_trt():
    """Import TensorRT + pycuda (buat context CUDA). Returns namespace atau None bila tidak tersedia."""
    try:
        import tensorrt as trt
        import pycuda.driver as cuda
        import pycuda.autoinit  # noqa: F401
    except Exception as e:
        print("TensorRT/pycuda tidak tersedia:", e)
        return None
    try:
        cuda.Device(0)                                  # GPU dicek lewat pycuda: torch.cuda TIDAK disentuh
    except Exception as e:
        print("GPU tidak tersedia untuk pycuda:", e)
        return None
    NS = SimpleNamespace(trt=trt, cuda=cuda, logger=trt.Logger(trt.Logger.WARNING))

    def build_engine(onnx_path, engine_path):
        builder = trt.Builder(NS.logger)
        network = builder.create_network()
        parser = trt.OnnxParser(network, NS.logger)
        with open(onnx_path, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError("\n".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
        config = builder.create_builder_config()
        config.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)   # tanpa flag presisi = FP32
        eb = builder.build_serialized_network(network, config)
        assert eb is not None, f"Gagal build {onnx_path}"
        Path(engine_path).write_bytes(eb)

    def _io(engine):
        inp, out = [], []
        for i in range(engine.num_io_tensors):
            n = engine.get_tensor_name(i)
            (inp if engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT else out).append(n)
        return inp, out

    class TRTEngineOptimized:
        """Buffer dialokasikan sekali. Vanilla & P2-Static."""
        def __init__(self, path):
            with open(path, "rb") as f, trt.Runtime(NS.logger) as rt:
                self.engine = rt.deserialize_cuda_engine(f.read())
            self.ctx = self.engine.create_execution_context(); self.stream = cuda.Stream()
            self.inp, self.out = _io(self.engine); self.d, self.h = {}, {}
            for n in self.inp:
                shape, dt = tuple(self.engine.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n))
                self.ctx.set_input_shape(n, shape)
                self.d[n] = cuda.mem_alloc(int(np.prod(shape)) * np.dtype(dt).itemsize)
                self.ctx.set_tensor_address(n, int(self.d[n]))
            for n in self.out:
                self.h[n] = cuda.pagelocked_empty(tuple(self.ctx.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n)))
                self.d[n] = cuda.mem_alloc(self.h[n].nbytes)
                self.ctx.set_tensor_address(n, int(self.d[n]))

        def infer(self, image_np):
            cuda.memcpy_htod_async(self.d[self.inp[0]], image_np, self.stream)
            self.ctx.execute_async_v3(stream_handle=self.stream.handle)
            for n in self.out:
                cuda.memcpy_dtoh_async(self.h[n], self.d[n], self.stream)
            self.stream.synchronize()
            return self.h[self.out[0]]

    class TRTEngineChainable:
        """Router: Stage1 dari host; Stage2 menerima pointer device Stage1 (tanpa salinan fitur)."""
        def __init__(self, path):
            with open(path, "rb") as f, trt.Runtime(NS.logger) as rt:
                self.engine = rt.deserialize_cuda_engine(f.read())
            self.ctx = self.engine.create_execution_context(); self.stream = cuda.Stream()
            self.inp, self.out = _io(self.engine); self.d_in, self.d_out, self.h_out = {}, {}, {}
            for n in self.inp:
                shape, dt = tuple(self.engine.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n))
                self.ctx.set_input_shape(n, shape)
                self.d_in[n] = cuda.mem_alloc(int(np.prod(shape)) * np.dtype(dt).itemsize)
            for n in self.out:
                self.h_out[n] = cuda.pagelocked_empty(tuple(self.ctx.get_tensor_shape(n)), trt.nptype(self.engine.get_tensor_dtype(n)))
                self.d_out[n] = cuda.mem_alloc(self.h_out[n].nbytes)
                self.ctx.set_tensor_address(n, int(self.d_out[n]))

        def run_from_host(self, image_np):
            n = self.inp[0]
            cuda.memcpy_htod_async(self.d_in[n], image_np, self.stream)
            self.ctx.set_tensor_address(n, int(self.d_in[n]))
            self.ctx.execute_async_v3(stream_handle=self.stream.handle)

        def read_scalar(self, name):
            cuda.memcpy_dtoh_async(self.h_out[name], self.d_out[name], self.stream)
            self.stream.synchronize()
            return float(self.h_out[name].flatten()[0])

        def run_from_device(self, ptrs):
            for n, p in ptrs.items():
                self.ctx.set_tensor_address(n, int(p))
            self.ctx.execute_async_v3(stream_handle=self.stream.handle)
            for n in self.out:
                cuda.memcpy_dtoh_async(self.h_out[n], self.d_out[n], self.stream)
            self.stream.synchronize()
            return self.h_out[self.out[0]]

    NS.build_engine, NS.Optimized, NS.Chainable = build_engine, TRTEngineOptimized, TRTEngineChainable
    return NS


class TRTSystem:
    """Router 3-engine + 2 baseline. router(image_np, force=None|'A'|'B') -> (y, gate, branch)."""
    A_KEYS, B_KEYS = ("p3_neck", "p2_backbone", "p4_neck", "p5_sppf"), ("p3_neck", "p4_neck", "p5_sppf")

    def __init__(self, NS, engines):
        self.s1, self.s2a, self.s2b = (NS.Chainable(engines[k]) for k in ("stage1", "stage2a", "stage2b"))
        self.vanilla, self.p2static = NS.Optimized(engines["vanilla"]), NS.Optimized(engines["p2static"])

    def router(self, image_np, force=None):
        self.s1.run_from_host(image_np)
        g = self.s1.read_scalar("gate")
        use_a = (g > 0.5) if force is None else (force == "A")
        if use_a:
            y = self.s2a.run_from_device({k: self.s1.d_out[k] for k in self.A_KEYS})
        else:
            y = self.s2b.run_from_device({k: self.s1.d_out[k] for k in self.B_KEYS})
        return y, g, "A" if use_a else "B"


def predict_router_trt(sysm):
    def fn(im):
        preds, infos = [], []
        for i in range(im.shape[0]):
            y, g, br = sysm.router(np.ascontiguousarray(im[i:i + 1].cpu().numpy(), dtype=np.float32))
            preds.append(torch.from_numpy(y.copy()).to(im.device)); infos.append({"gate": g, "branch": br})
        return preds, infos
    return fn


def predict_static_trt(eng):
    def fn(im):
        preds = [torch.from_numpy(eng.infer(np.ascontiguousarray(im[i:i + 1].cpu().numpy(), dtype=np.float32)).copy()).to(im.device)
                 for i in range(im.shape[0])]
        return preds, [{} for _ in range(im.shape[0])]
    return fn


def lat_stats(a):
    a = np.asarray(a, float)
    return dict(mean=a.mean(), std=a.std(), p50=np.percentile(a, 50), p95=np.percentile(a, 95),
                p99=np.percentile(a, 99), min=a.min(), max=a.max(), n=int(a.size))


def benchmark_latency(sysm, loader, n_warmup=50, n_reps=3):
    """
    Interleaved (urutan dibalik tiap putaran), gambar test asli yang sama, cakupan: H2D gambar + inferensi
    (+ baca 1 skalar gate) + D2H output. Router dipaksa A / B diukur untuk kurva latensi t(r) = r·T_A + (1−r)·T_B.
    """
    as_np = lambda b: np.ascontiguousarray(b["img"].numpy().astype(np.float32) / 255.0)
    for k, batch in enumerate(loader):
        if k >= n_warmup:
            break
        im = as_np(batch)
        sysm.vanilla.infer(im); sysm.p2static.infer(im); sysm.router(im, "A"); sysm.router(im, "B")
    fns = {"Vanilla": lambda im: sysm.vanilla.infer(im), "P2-Static": lambda im: sysm.p2static.infer(im),
           "Router": lambda im: sysm.router(im), "Router paksa A": lambda im: sysm.router(im, "A"),
           "Router paksa B": lambda im: sysm.router(im, "B")}
    t = {k: [] for k in fns}
    branches, by_branch = [], {"A": [], "B": []}
    for rep in range(n_reps):
        for batch in loader:
            im = as_np(batch)
            order = list(fns) if rep % 2 == 0 else list(fns)[::-1]
            for name in order:
                t0 = time.perf_counter(); out = fns[name](im); dt = (time.perf_counter() - t0) * 1e3
                t[name].append(dt)
                if name == "Router":
                    by_branch[out[2]].append(dt)
                    if rep == 0:
                        branches.append(out[2])
    LAT = {k: lat_stats(v) for k, v in t.items()}
    for b in "AB":
        LAT[f"Router (saat Branch {b})"] = lat_stats(by_branch[b]) if by_branch[b] else None
    return LAT, float(np.mean([b == "A" for b in branches]))


# =============================================================================
# JOB TENSORRT DI PROSES TERPISAH  (python -m tahap2_eval <trt_job.json>)
# =============================================================================
def _rss_gb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1e9
    except Exception:
        return float("nan")


def trt_job_main(job_path):
    """
    Build engine + mAP dari output TensorRT + latensi, di proses TERPISAH dari kernel notebook:
      - kernel notebook memegang cache besar (output Branch A/B, jdict, COCOeval) -> RAM kernel tidak ikut
        terbebani builder TensorRT;
      - proses ini TIDAK pernah membuat context CUDA PyTorch (validator & NMS di CPU), sehingga hanya ada satu
        context (pycuda) -> tidak ada konflik dua context di satu proses;
      - bila TensorRT/pycuda crash (segfault) atau proses dibunuh OOM, kernel notebook tetap hidup dan
        returncode memberi tahu penyebabnya.
    Masukan: JSON (onnx, data_yaml, split, imgsz, conf, iou, max_det, gt_json, out_json, n_warmup, n_reps).
    """
    global COCO_MAX_DETS
    job = json.loads(Path(job_path).read_text())
    COCO_MAX_DETS = list(job.get("coco_max_dets", COCO_MAX_DETS))
    t0 = time.perf_counter()
    log = lambda *a: print(f"[trt-job {time.perf_counter() - t0:6.0f}s | RSS {_rss_gb():.1f} GB]", *a, flush=True)
    from ultralytics.data.utils import check_det_dataset
    dev = torch.device("cpu")
    DATA = check_det_dataset(job["data_yaml"])
    # pin_memory=False: pinned memory membuat context CUDA PyTorch -> dua context (torch + pycuda) di satu
    # proses -> "CUDA error: invalid resource handle" (penyebab kernel mati di Kaggle, 10 Okt).
    loader = make_loader(DATA, job["split"], 1, job["imgsz"], workers=2, pin_memory=False)
    EV = {k: job[k] for k in ("data_yaml", "conf", "iou", "max_det", "imgsz")}; EV["report_split"] = job["split"]
    NS = init_trt()
    assert not torch.cuda.is_initialized(), "Context CUDA PyTorch terbentuk di proses job TRT — harus tidak ada"
    if NS is None:
        Path(job["out_json"]).write_text(json.dumps({"trt_available": False}))
        sys.exit(3)
    log("TensorRT", NS.trt.__version__)
    eng = {}
    for k, p in job["onnx"].items():
        eng[k] = str(Path(p).with_suffix(".engine"))
        NS.build_engine(p, eng[k]); log("engine", k, "OK")
    SYS = TRTSystem(NS, eng)
    from pycocotools.coco import COCO
    import contextlib, io
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(job["gt_json"])
    res = {"trt_available": True, "trt_version": NS.trt.__version__, "acc": {}}
    for name, fn in (("Router (TRT)", predict_router_trt(SYS)), ("Vanilla (TRT)", predict_static_trt(SYS.vanilla)),
                     ("P2-Static (TRT)", predict_static_trt(SYS.p2static))):
        r = run_eval(name, fn, loader, DATA, EV, dev, save_json=True, verbose=False)
        res["acc"][name] = {**summarize_acc(r), **{f"coco_{m}": v for m, v in coco_eval(gt, r["jdict"])[0].items()}}
        if name.startswith("Router"):
            res["files"] = [x["file"] for x in r["records"]]
            res["router_branch"] = [x["branch"] for x in r["records"]]
            res["router_gate"] = [x["gate"] for x in r["records"]]
        log(name, "mAP50", round(res["acc"][name]["mAP50"], 4)); del r
    lat, act = benchmark_latency(SYS, loader, job["n_warmup"], job["n_reps"])
    res.update(latency=lat, activation_bench=act)
    Path(job["out_json"]).write_text(json.dumps(res, default=float))
    assert not torch.cuda.is_initialized(), "Context CUDA PyTorch terbentuk di proses job TRT"
    log("selesai")


if __name__ == "__main__":
    import sys
    trt_job_main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/tahap2b_plafon.py
# [8b] modul (hapus sel ini setelah file di-commit ke repo)
"""
tahap2b_plafon.py — Uji plafon routing + gate post-hoc (Opsi A) untuk Router-P2 Dual-Branch.

Pertanyaan:
  (1) PLAFON: adakah routing per gambar (A vs B) yang mengalahkan routing acak secara berarti pada aktivasi
      yang sama, bila target diukur dari HASIL DETEKSI (bukan loss)? Oracle memakai GT test -> batas atas,
      termasuk optimisme seleksi; bukan klaim deployment.
  (2) DAPAT DIPELAJARI: gate baru di atas detektor BEKU (fitur pooled dari keluaran Stage 1 — sudah tersedia
      di engine Stage 1, jadi deployable), dilatih di TRAIN, dipilih di VAL dengan aturan tetap, dinilai
      SEKALI di TEST.

Desain:
  - Satu forward per gambar (letterbox persegi 640, format deployment): z gate lama, fitur pooled
    (GAP+GMP dari p3_neck, p2_backbone, p4_neck, p5_sppf), deteksi pasca-NMS Branch A & B (conf 0.001,
    IoU 0.7, max_det 300 — sama dengan Tahap 2), dikonversi ke koordinat asli PER SUMBU.
  - COCOeval dijalankan SEKALI untuk A dan SEKALI untuk B per split. Karena evaluateImg COCO independen per
    gambar, metrik routing apa pun = gabungan evalImgs A/B per gambar (CocoMix) -> ratusan routing murah.
    Diverifikasi sama persis dengan COCOeval pada jdict ter-route (verify_mix).
  - Target per gambar dari evalImgs (IoU 0.5): soft-recall objek kecil (jumlah skor deteksi TP pada GT kecil),
    soft-recall semua objek, dan TP−FP semua objek pada conf ≥ 0.25. Δ = nilai A − nilai B.
"""
import copy
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import tahap2_eval as E
from router_train_utils import find_router, _raw_gate_logit

TARGETS = ("soft_small", "soft_all", "net25_all")


# =============================================================================
# CACHE: deteksi pasca-NMS A/B (koordinat asli), z, fitur pooled
# =============================================================================
def _scale_xyxy(b, ratio_pad, ori_shape):
    (gh, gw), (px, py) = ratio_pad
    b = b.clone()
    b[:, [0, 2]] = ((b[:, [0, 2]] - px) / gw).clamp(0, ori_shape[1])
    b[:, [1, 3]] = ((b[:, [1, 3]] - py) / gh).clamp(0, ori_shape[0])
    return b


def _to_dets(y, ratio_pad, ori_shape, conf, iou, max_det):
    """y (1,4+nc,N) -> array (n,6): x, y, w, h (COCO, piksel asli), skor, category_id (=cls+1)."""
    from ultralytics.utils.nms import non_max_suppression
    x = non_max_suppression(y.clone(), conf, iou, nc=0, multi_label=True, agnostic=False, max_det=max_det)[0]
    if not len(x):
        return np.zeros((0, 6), np.float32)
    b = _scale_xyxy(x[:, :4], ratio_pad, ori_shape)
    out = torch.stack([b[:, 0], b[:, 1], b[:, 2] - b[:, 0], b[:, 3] - b[:, 1], x[:, 4], x[:, 5] + 1], 1)
    return out.float().cpu().numpy()


def _pool(t):
    return torch.cat([t.mean((2, 3)), t.amax((2, 3))], 1)


@torch.no_grad()
def cache_dets(model, stages, loader, device, conf=0.001, iou=0.7, max_det=300, max_images=None, log_every=1000):
    """Loader batch 1, urutan tetap. Returns dict file, image_id, z, feat (N,D), detA, detB (list array (n,6))."""
    s1, s2a, s2b = stages
    rt = find_router(model)
    out = dict(file=[], image_id=[], z=[], feat=[], detA=[], detB=[])
    t0 = time.perf_counter()
    for i, b in enumerate(loader):
        if max_images and i >= max_images:
            break
        assert b["img"].shape[0] == 1
        im = b["img"].to(device).float() / 255.0
        _, p3, p2, p4, p5 = s1(im)
        rp, osh = b["ratio_pad"][0], b["ori_shape"][0]
        out["detA"].append(_to_dets(s2a(p3, p2, p4, p5), rp, osh, conf, iou, max_det))
        out["detB"].append(_to_dets(s2b(p3, p4, p5), rp, osh, conf, iou, max_det))
        out["z"].append(float(_raw_gate_logit(rt, p3, p2)[0]))
        out["feat"].append(torch.cat([_pool(p3), _pool(p2), _pool(p4), _pool(p5)], 1)[0].float().cpu().numpy())
        stem = Path(b["im_file"][0]).stem
        out["file"].append(Path(b["im_file"][0]).name)
        out["image_id"].append(int(stem) if stem.isnumeric() else stem)
        if log_every and (i + 1) % log_every == 0:
            print(f"  {i + 1} gambar | {time.perf_counter() - t0:.0f}s", flush=True)
    out["z"] = np.asarray(out["z"]); out["feat"] = np.stack(out["feat"])
    return out


def jdict(dets, image_ids, use_a=None):
    """dets: dict detA/detB; use_a None -> pakai detA semua. Returns list COCO results."""
    js = []
    for i, iid in enumerate(image_ids):
        d = dets["detA"][i] if (use_a is None or use_a[i]) else dets["detB"][i]
        for x, y, w, h, s, c in d:
            js.append({"image_id": iid, "category_id": int(c), "bbox": [float(x), float(y), float(w), float(h)], "score": float(s)})
    return js


def subset_coco_gt(dataset, names, path, n):
    """GT COCO untuk n gambar pertama dataset (sesuai cache_dets dengan max_images)."""
    ds = copy.copy(dataset); ds.labels = dataset.labels[:n]
    return E.build_coco_gt(ds, names, path)


# =============================================================================
# CAMPURAN evalImgs A/B -> metrik routing apa pun
# =============================================================================
class CocoMix(E.CocoBootstrap):
    """Metrik COCO untuk routing per gambar dari evalImgs A & B (indeks = urutan E.params.imgIds)."""
    def __init__(self, bA, bB, use_a_sorted):
        for k in ("T", "R", "I", "A", "K", "maxDet", "iou50"):
            setattr(self, k, getattr(bA, k))
        assert bA.I == bB.I == len(use_a_sorted)
        self.cells = {key: [ca if u else cb for ca, cb, u in zip(bA.cells[key], bB.cells[key], use_a_sorted)]
                      for key in bA.cells}


class Router:
    """Mengelola urutan: array per gambar (urutan cache/loader) <-> urutan imgIds COCO (sorted)."""
    def __init__(self, E_A, E_B, image_ids):
        self.bA, self.bB = E.CocoBootstrap(E_A), E.CocoBootstrap(E_B)
        ids_sorted = list(E_A.params.imgIds)
        assert ids_sorted == list(E_B.params.imgIds)
        pos = {iid: j for j, iid in enumerate(ids_sorted)}
        self.perm = np.array([pos[i] for i in image_ids])          # cache i -> posisi COCO
        self.n = len(image_ids)

    @classmethod
    def from_boots(cls, bA, bB, ids_sorted, image_ids):
        """Dari dua CocoBootstrap yang sudah ada (urutan = ids_sorted = E.params.imgIds)."""
        r = cls.__new__(cls)
        r.bA, r.bB = bA, bB
        pos = {iid: j for j, iid in enumerate(ids_sorted)}
        r.perm = np.array([pos[i] for i in image_ids]); r.n = len(image_ids)
        return r

    def sorted_mask(self, use_a):
        m = np.zeros(self.n, bool); m[self.perm] = np.asarray(use_a, bool)
        return m

    def mix(self, use_a):
        return CocoMix(self.bA, self.bB, self.sorted_mask(use_a))

    def metrics(self, use_a, idx=None):
        return self.mix(use_a).metrics(np.arange(self.n) if idx is None else idx)

    def targets(self, conf_thr=0.25):
        """Δ per gambar (A − B), urutan cache. Juga jumlah GT kecil & semua (non-ignored)."""
        def per_image(b):
            res = {t: np.zeros(self.n) for t in TARGETS}
            res["n_gt_small"] = np.zeros(self.n); res["n_gt_all"] = np.zeros(self.n)
            t50 = b.iou50
            for (k, area), cells in b.cells.items():
                if area not in ("all", "small"):
                    continue
                for j, c in enumerate(cells):
                    if c is None:
                        continue
                    sc, dtm, dtig, npig = c
                    tp = dtm[t50] & ~dtig[t50]
                    if area == "small":
                        res["soft_small"][j] += sc[tp].sum(); res["n_gt_small"][j] += npig
                    else:
                        res["soft_all"][j] += sc[tp].sum(); res["n_gt_all"][j] += npig
                        fp = ~dtm[t50] & ~dtig[t50]
                        res["net25_all"][j] += (tp & (sc >= conf_thr)).sum() - (fp & (sc >= conf_thr)).sum()
            return res
        rA, rB = per_image(self.bA), per_image(self.bB)
        out = {t: (rA[t] - rB[t])[self.perm] for t in TARGETS}
        out["n_gt_small"] = rA["n_gt_small"][self.perm]; out["n_gt_all"] = rA["n_gt_all"][self.perm]
        return out


def verify_mix(router, coco_gt, dets, image_ids, use_a, atol=1e-9):
    """Metrik CocoMix == COCOeval langsung pada jdict ter-route (bukti campuran evalImgs sah)."""
    st, _ = E.coco_eval(coco_gt, jdict(dets, image_ids, use_a))
    m = router.metrics(use_a)
    for k, kk in (("mAP50-95", "mAP50-95"), ("mAP50", "mAP50"), ("mAP_small", "mAP_small")):
        assert abs(m[k] - st[kk]) <= atol, f"CocoMix {k} {m[k]} != COCOeval {st[kk]}"
    return m


# =============================================================================
# ROUTING DARI SKOR, ACAK, KURVA
# =============================================================================
def top_rho(score, rho, rng=None):
    """use_a = ρ·N gambar dengan skor tertinggi (seri dipecah acak tetapi deterministik)."""
    n = len(score); k = int(round(rho * n))
    rng = rng or np.random.default_rng(0)
    order = np.lexsort((rng.random(n), -np.asarray(score, float)))
    m = np.zeros(n, bool); m[order[:k]] = True
    return m


def random_masks(n, rho, n_draw, seed):
    rng = np.random.default_rng(seed)
    k = int(round(rho * n))
    out = []
    for _ in range(n_draw):
        m = np.zeros(n, bool); m[rng.choice(n, k, replace=False)] = True; out.append(m)
    return out


def curve(router, scores, rhos, n_random=20, seed=0):
    """DataFrame: per ρ dan per skor -> metrik COCO + selisih terhadap rata-rata acak (n_random undian)."""
    rows = []
    for rho in rhos:
        rnd = [router.metrics(m) for m in random_masks(router.n, rho, n_random, seed + int(rho * 1000))]
        base = {k: float(np.mean([r[k] for r in rnd])) for k in E.BOOT_METRICS}
        sd = {k: float(np.std([r[k] for r in rnd])) for k in E.BOOT_METRICS}
        rows.append(dict(rho=rho, routing="acak (rata-rata)", **base, **{f"sd_{k}": v for k, v in sd.items()}))
        for name, sc in scores.items():
            m = router.metrics(top_rho(sc, rho))
            rows.append(dict(rho=rho, routing=name, **m, **{f"gain_{k}": m[k] - base[k] for k in E.BOOT_METRICS}))
    return pd.DataFrame(rows)


def paired_bootstrap_routing(router, use_a, rho, n_random=5, n_boot=1000, seed=0):
    """Bootstrap berpasangan per gambar: routing − rata-rata acak (n_random undian), pada resample yang sama."""
    mix_a = router.mix(use_a)
    mix_r = [router.mix(m) for m in random_masks(router.n, rho, n_random, seed + 7)]
    return E.paired_bootstrap(mix_a, mix_r, n_boot=n_boot, seed=seed)


# =============================================================================
# GATE POST-HOC (detektor beku)
# =============================================================================
def candidate_models():
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import Ridge, LogisticRegression
    from sklearn.ensemble import HistGradientBoostingRegressor
    c = {}
    for a in (1.0, 10.0, 100.0, 1000.0):
        c[f"ridge_a{a:g}"] = ("reg", lambda a=a: make_pipeline(StandardScaler(), Ridge(alpha=a)))
    for C in (0.001, 0.01, 0.1):
        c[f"logreg_C{C:g}"] = ("clf", lambda C=C: make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=2000)))
    c["hgb"] = ("reg", lambda: HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                                              min_samples_leaf=40, l2_regularization=1.0, random_state=0))
    return c


def fit_score(kind, make, Xtr, ytr, X):
    m = make()
    if kind == "clf":
        yb = (ytr > 0).astype(int)
        if yb.min() == yb.max():
            return None, None
        m.fit(Xtr, yb)
        return m, m.decision_function(X)
    m.fit(Xtr, ytr)
    return m, m.predict(X)


def features(cache, with_z=True):
    X = cache["feat"]
    return np.concatenate([X, cache["z"][:, None]], 1) if with_z else X


def select_gate(cache_tr, tgt_tr, cache_va, router_va, rhos, n_random=20, seed=0, select_metric="mAP_small"):
    """
    Latih semua kandidat (model x target) di TRAIN, nilai di VAL: rata-rata gain metrik vs acak pada rhos.
    Aturan pemilihan TETAP: argmax rata-rata gain `select_metric` di val. Returns (tabel, nama terbaik, objek).
    """
    Xtr, Xva = features(cache_tr), features(cache_va)
    rnd = {rho: np.mean([router_va.metrics(m)[select_metric]
                         for m in random_masks(router_va.n, rho, n_random, seed + int(rho * 1000))]) for rho in rhos}
    rows, fitted = [], {}
    for tname in TARGETS:
        for mname, (kind, make) in candidate_models().items():
            m, s = fit_score(kind, make, Xtr, tgt_tr[tname], Xva)
            if m is None:
                continue
            gains = [router_va.metrics(top_rho(s, rho))[select_metric] - rnd[rho] for rho in rhos]
            key = f"{tname}|{mname}"
            fitted[key] = (kind, m)
            rows.append(dict(kandidat=key, target=tname, model=mname, mean_gain_val=float(np.mean(gains)),
                             **{f"gain_rho{rho:g}": g for rho, g in zip(rhos, gains)}))
    tab = pd.DataFrame(rows).sort_values("mean_gain_val", ascending=False).reset_index(drop=True)
    best = tab.kandidat.iloc[0]
    return tab, best, fitted[best]


def gate_score(fitted, cache):
    kind, m = fitted
    X = features(cache)
    return m.decision_function(X) if kind == "clf" else m.predict(X)

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/tahap2c_gate.py
# [8c] modul (hapus sel ini setelah file di-commit ke repo)
"""
tahap2c_gate.py — Gate post-hoc terkunci (Tahap 2c): dari sklearn ke modul PyTorch/ONNX/TensorRT.

Gate (dikunci dari Tahap 2b, dipilih di val di 2 dari 3 seed):
    target  : y = 1[Δsoft_small > 0]   (Δ = A − B jumlah skor deteksi TP pada GT kecil, IoU 0.5, gambar TRAIN)
    model   : StandardScaler + LogisticRegression(C=0.001)
    fitur   : [GAP‖GMP](p3_neck) ‖ [GAP‖GMP](p2_backbone) ‖ [GAP‖GMP](p4_neck) ‖ [GAP‖GMP](p5_sppf) ‖ z_lama   (961)
    keputusan: skor > thr, thr = kuantil (1 − ρ) skor VAL, ρ = 0.5

Scaler + logistic dilipat jadi satu vektor bobot: s = f·w + b, w = coef/scale, b = intercept − Σ coef·mean/scale.
Di engine Stage 1 ini hanya pooling global + 1 Gemm + perbandingan -> output `gate` 0/1 dengan nama & bentuk
yang sama seperti Stage 1 lama, sehingga runtime 3-engine (TRTSystem) tidak berubah.
"""
import json
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn

import tahap2_eval as E
from router_train_utils import find_router, _raw_gate_logit


def _pool(t):
    return torch.cat([t.mean((2, 3)), t.amax((2, 3))], 1)


class PostHocGate(nn.Module):
    def __init__(self, router, w, b, thr):
        super().__init__()
        self.router = router                                   # hanya dipakai untuk z lama (fitur ke-961)
        self.register_buffer("w", torch.as_tensor(np.asarray(w), dtype=torch.float32))
        self.register_buffer("b", torch.tensor(float(b), dtype=torch.float32))
        self.register_buffer("thr", torch.tensor(float(thr), dtype=torch.float32))

    def features(self, p3, p2, p4, p5):
        z = _raw_gate_logit(self.router, p3, p2).float()
        return torch.cat([_pool(p3), _pool(p2), _pool(p4), _pool(p5), z[:, None]], 1).float()

    def score(self, p3, p2, p4, p5):
        return self.features(p3, p2, p4, p5) @ self.w + self.b

    def forward(self, p3, p2, p4, p5):
        return (self.score(p3, p2, p4, p5) > self.thr).float()


class Stage1PostHoc(nn.Module):
    """Layer 0–15 + gate post-hoc. Output identik bentuk/namanya dengan Stage1 lama: (gate, p3, p2, p4, p5)."""
    def __init__(self, full_model, gate):
        super().__init__()
        ml = list(full_model.model)
        self.pre_router_layers = nn.ModuleList(ml[0:16])
        self.gate = gate

    def trunk(self, x):
        y = []
        for m in self.pre_router_layers:
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y.append(x)
        return x, y[2], y[12], y[9]

    def forward(self, x):
        p3, p2, p4, p5 = self.trunk(x)
        return self.gate(p3, p2, p4, p5), p3, p2, p4, p5


def fit_locked_gate(feat_train, delta_train, C=0.001):
    """Returns (w, b, info). feat_train sudah termasuk z di kolom terakhir (P.features)."""
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    y = (np.asarray(delta_train) > 0).astype(int)
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(feat_train, y)
    sc, lr = pipe[0], pipe[1]
    scale = np.where(sc.scale_ == 0, 1.0, sc.scale_)
    w = lr.coef_[0] / scale
    b = float(lr.intercept_[0] - np.sum(lr.coef_[0] * sc.mean_ / scale))
    info = dict(C=C, n_train=int(len(y)), pos_rate_train=float(y.mean()), n_features=int(feat_train.shape[1]),
                n_iter=int(np.max(lr.n_iter_)))
    return w, b, pipe, info


@torch.no_grad()
def gate_scores(stage1ph, loader, device):
    """Skor & keputusan gate per gambar (urutan loader) lewat modul yang SAMA dengan yang diekspor."""
    s, d, files = [], [], []
    for b in loader:
        im = b["img"].to(device).float() / 255.0
        p3, p2, p4, p5 = stage1ph.trunk(im)
        sc = stage1ph.gate.score(p3, p2, p4, p5)
        s.append(sc.cpu().numpy()); d.append((sc > stage1ph.gate.thr).cpu().numpy())
        files += [Path(f).name for f in b["im_file"]]
    return np.concatenate(s), np.concatenate(d), files


def export_onnx_posthoc(model, gate, vanilla, p2static, x0, out_dir, tag):
    """ONNX untuk job TensorRT (nama file/IO sama dengan Tahap 2). Returns paths, salinan CPU, statics."""
    m_cpu = E.cpu_copy(model)
    g_cpu = PostHocGate(find_router(m_cpu), gate.w.cpu().numpy(), float(gate.b), float(gate.thr)).eval()
    s1c = Stage1PostHoc(m_cpu, g_cpu).eval()
    _, s2ac, s2bc = E.make_stages(m_cpu)
    with torch.no_grad():
        _, p3, p2, p4, p5 = s1c(x0)
    P = {k: Path(out_dir) / f"{k}_{tag}.onnx" for k in ("stage1", "stage2a", "stage2b")}
    P["vanilla"], P["p2static"] = Path(out_dir) / "vanilla.onnx", Path(out_dir) / "p2static.onnx"
    E.export_onnx(E._Stage1Out(s1c), (x0,), P["stage1"], ["image"], ["gate", "p3_neck", "p2_backbone", "p4_neck", "p5_sppf"])
    E.export_onnx(s2ac, (p3, p2, p4, p5), P["stage2a"], ["p3_neck", "p2_backbone", "p4_neck", "p5_sppf"], ["y_A"])
    E.export_onnx(s2bc, (p3, p4, p5), P["stage2b"], ["p3_neck", "p4_neck", "p5_sppf"], ["y_B"])
    statics = {"vanilla": E._StaticOut(E.cpu_copy(vanilla)).eval(), "p2static": E._StaticOut(E.cpu_copy(p2static)).eval()}
    for k, mod in statics.items():
        E.export_onnx(mod, (x0,), P[k], ["image"], ["y"])
    return P, (s1c, s2ac, s2bc), statics


def verify_onnx_posthoc(P, cpu_stages, statics, loader, n=5):
    """Seperti E.verify_onnx; keputusan gate boleh beda HANYA bila |skor − thr| < 1e-4 (batas numerik)."""
    import onnxruntime as ort
    sess = ort.InferenceSession(str(P["stage1"]), providers=["CPUExecutionProvider"])
    s1c = cpu_stages[0]
    with torch.no_grad():
        for k, b in enumerate(loader):
            if k >= n:
                break
            x = b["img"].float() / 255.0
            g, p3, p2, p4, p5 = s1c(x)
            o = sess.run(None, {"image": x.numpy()})
            if float(o[0].flatten()[0]) != float(g.flatten()[0]):
                margin = float((s1c.gate.score(p3, p2, p4, p5) - s1c.gate.thr).abs())
                assert margin < 1e-4, f"Gate ONNX != PyTorch pada {b['im_file'][0]} (margin {margin:.2e})"
    return _verify_rest(P, cpu_stages, statics, loader, n)


def _verify_rest(P, cpu_stages, statics, loader, n):
    import onnxruntime as ort
    rel = lambda a, b: float(np.abs(a - b).max() / (np.abs(b).max() + 1e-12))
    sess = {k: ort.InferenceSession(str(p), providers=["CPUExecutionProvider"]) for k, p in P.items()}
    run = lambda k, feed: sess[k].run(None, {i.name: feed[i.name] for i in sess[k].get_inputs()})
    s1c, s2ac, s2bc = cpu_stages
    worst = {}
    with torch.no_grad():
        for k, b in enumerate(loader):
            if k >= n:
                break
            x = b["img"].float() / 255.0
            _, p3, p2, p4, p5 = s1c(x)
            o1 = run("stage1", {"image": x.numpy()})
            worst["stage1_feats"] = max(worst.get("stage1_feats", 0), *[rel(a, t.numpy()) for a, t in zip(o1[1:], (p3, p2, p4, p5))])
            f = {"p3_neck": p3.numpy(), "p2_backbone": p2.numpy(), "p4_neck": p4.numpy(), "p5_sppf": p5.numpy()}
            worst["stage2a"] = max(worst.get("stage2a", 0), rel(run("stage2a", f)[0], s2ac(p3, p2, p4, p5).numpy()))
            worst["stage2b"] = max(worst.get("stage2b", 0), rel(run("stage2b", f)[0], s2bc(p3, p4, p5).numpy()))
            for nm, mod in statics.items():
                worst[nm] = max(worst.get(nm, 0), rel(run(nm, {"image": x.numpy()})[0], mod(x).numpy()))
    assert all(v < 1e-4 for v in worst.values()), f"Selisih ONNX vs PyTorch > 1e-4: {worst}"
    return worst

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/tahap3a.py
# [8d] modul (hapus sel ini setelah file di-commit ke repo)
"""
tahap3a.py — Tahap 3a (tanpa training): D1 ablasi fitur gate, B kendali anggaran aktivasi, A1 profil latensi + GFLOPs.

D1  Fitur gate post-hoc (target & model TERKUNCI: soft_small biner, StandardScaler + LogReg C=0.001):
    varian blok fitur dari [GAP‖GMP](p3) ‖ [GAP‖GMP](p2) ‖ [GAP‖GMP](p4) ‖ [GAP‖GMP](p5) ‖ z  (961 kolom).
    Dinilai pada urutan skor tepat ρ (top-ρ) agar perbandingan antar-varian tidak terganggu pergeseran threshold.
B   Strategi threshold untuk anggaran aktivasi ρ pada aliran gambar test:
    B1 kuantil val tetap | B2 kuantil bergulir N gambar terakhir | B3 kuantil 10% awal aliran (tanpa label).
    Dievaluasi di 90% sisa aliran; urutan asli (nama file) + beberapa urutan acak.
A1  Profil TensorRT per segmen (CUDA event, satu stream) + GFLOPs per modul (FlopCounterMode).
"""
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

import tahap2_eval as E
import tahap2b_plafon as P
import tahap2c_gate as G
from router_train_utils import find_router

# indeks kolom fitur (urutan P.cache_dets: p3, p2, p4, p5 masing-masing [GAP | GMP], lalu z)
def feature_blocks(c_p3, c_p2, c_p4, c_p5):
    idx, o = {}, 0
    for nm, c in (("p3", c_p3), ("p2", c_p2), ("p4", c_p4), ("p5", c_p5)):
        idx[f"{nm}_gap"] = np.arange(o, o + c); idx[f"{nm}_gmp"] = np.arange(o + c, o + 2 * c); o += 2 * c
    idx["z"] = np.array([o])
    return idx, o + 1


def variant_columns(blocks):
    cat = lambda *ks: np.concatenate([blocks[k] for k in ks])
    allf = [k for k in blocks if k != "z"]
    return {
        "lengkap (961)": cat(*allf, "z"),
        "tanpa z": cat(*allf),
        "hanya z": cat("z"),
        "p3+p2": cat("p3_gap", "p3_gmp", "p2_gap", "p2_gmp"),
        "hanya p3": cat("p3_gap", "p3_gmp"),
        "hanya p2": cat("p2_gap", "p2_gmp"),
        "hanya p4": cat("p4_gap", "p4_gmp"),
        "hanya p5": cat("p5_gap", "p5_gmp"),
        "hanya GAP": cat(*[k for k in allf if k.endswith("gap")]),
        "hanya GMP": cat(*[k for k in allf if k.endswith("gmp")]),
    }


def fit_logreg(X, delta, C=0.001):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    y = (np.asarray(delta) > 0).astype(int)
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000)).fit(X, y)


def d1_feature_ablation(Ctr, Tgt_tr, Cva, Rva, Cte, Rte, rho, C=0.001, n_random=20, seed=0, n_boot=1000):
    """Returns (tabel, skor per varian {split: array}, model per varian)."""
    blocks, D = feature_blocks(*_channel_split(Cte))
    cols = variant_columns(blocks)
    Xtr, Xva, Xte = P.features(Ctr), P.features(Cva), P.features(Cte)
    assert Xtr.shape[1] == D, f"jumlah kolom fitur {Xtr.shape[1]} != {D}"
    rnd = {}
    for nm, R, n in (("val", Rva, len(Cva["z"])), ("test", Rte, len(Cte["z"]))):
        rnd[nm] = [R.metrics(m) for m in P.random_masks(n, rho, n_random, seed + 3)]
    base = {s: {k: float(np.mean([r[k] for r in rnd[s]])) for k in E.BOOT_METRICS} for s in rnd}
    rows, scores, models = [], {}, {}
    for vname, cidx in cols.items():
        pipe = fit_logreg(Xtr[:, cidx], Tgt_tr["soft_small"], C)
        sv, st = pipe.decision_function(Xva[:, cidx]), pipe.decision_function(Xte[:, cidx])
        scores[vname] = {"val": sv, "test": st}; models[vname] = (pipe, cidx)
        mv, mt = Rva.metrics(P.top_rho(sv, rho)), Rte.metrics(P.top_rho(st, rho))
        rows.append(dict(varian=vname, n_fitur=len(cidx),
                         **{f"val_gain_{k}": mv[k] - base["val"][k] for k in E.BOOT_METRICS},
                         **{f"test_gain_{k}": mt[k] - base["test"][k] for k in E.BOOT_METRICS}))
    tab = pd.DataFrame(rows)
    # uji berpasangan di test: lengkap vs tanpa z; tanpa z vs acak
    m_full = Rte.mix(P.top_rho(scores["lengkap (961)"]["test"], rho))
    m_noz = Rte.mix(P.top_rho(scores["tanpa z"]["test"], rho))
    m_rnd = [Rte.mix(m) for m in P.random_masks(len(Cte["z"]), rho, 5, seed + 7)]
    bs = pd.concat([E.paired_bootstrap(m_full, [m_noz], n_boot, seed).assign(uji="lengkap − tanpa z (test, top-ρ)"),
                    E.paired_bootstrap(m_noz, m_rnd, n_boot, seed).assign(uji="tanpa z − acak (test, top-ρ)")])
    return tab, bs, scores, models


def _channel_split(C):
    """Jumlah kanal p3, p2, p4, p5 dari panjang fitur (YOLOv8n-P2: 64, 32, 128, 256). Diverifikasi via 2*sum+1."""
    D = P.features(C).shape[1]                                          # termasuk kolom z
    for ch in ((64, 32, 128, 256), (128, 64, 256, 512), (192, 96, 384, 576)):
        if 2 * sum(ch) + 1 == D:
            return ch
    raise ValueError(f"Tidak bisa menentukan kanal dari {D} fitur")


# =============================================================================
# B — kendali anggaran aktivasi
# =============================================================================
def threshold_stream(scores_stream, rho, calib_n, strategy, thr_val=None, window=None):
    """Keputusan use_a untuk setiap posisi aliran. Hanya memakai skor MASA LALU (tanpa label)."""
    s = np.asarray(scores_stream, float); n = len(s)
    if strategy == "B1 kuantil val":
        return s > thr_val
    if strategy == "B3 kalibrasi 10% awal":
        return s > np.quantile(s[:calib_n], 1 - rho)
    if strategy.startswith("B2"):
        out = np.zeros(n, bool)
        for t in range(n):
            lo = max(0, t - window)
            hist = s[lo:t]
            thr = np.quantile(hist, 1 - rho) if len(hist) >= 20 else thr_val
            out[t] = s[t] > thr
        return out
    raise ValueError(strategy)


def b_budget(Rte, scores_test, s_val, rho, windows=(50, 200, 500), n_orders=5, calib_frac=0.10, n_random=20, seed=0):
    """Returns DataFrame per (urutan, strategi): aktivasi terealisasi di 90% evaluasi, metrik, gain vs acak (aktivasi sama)."""
    n = len(scores_test); thr_val = float(np.quantile(s_val, 1 - rho))
    rng = np.random.default_rng(seed)
    orders = {"asli (nama file)": np.arange(n), **{f"acak #{j + 1}": rng.permutation(n) for j in range(n_orders)}}
    strategies = ["B1 kuantil val", "B3 kalibrasi 10% awal"] + [f"B2 bergulir N={w}" for w in windows]
    rows = []
    for oname, order in orders.items():
        calib_n = int(round(calib_frac * n))
        eval_idx = order[calib_n:]                                   # indeks cache yang dievaluasi
        idx_sorted = Rte.perm[eval_idx]                              # posisi COCO
        for strat in strategies:
            w = int(strat.split("=")[1]) if strat.startswith("B2") else None
            dec_stream = threshold_stream(scores_test[order], rho, calib_n, strat, thr_val, w)
            use_a = np.zeros(n, bool); use_a[order] = dec_stream
            act = float(use_a[eval_idx].mean())
            m = Rte.metrics(use_a, idx=idx_sorted)
            k = int(use_a[eval_idx].sum()); ne = len(eval_idx)
            rnd = []
            for d in P.random_masks(ne, k / ne, n_random, seed + 11):
                ua = np.zeros(n, bool); ua[eval_idx[d]] = True
                rnd.append(Rte.metrics(ua, idx=idx_sorted))
            base = {kk: float(np.mean([r[kk] for r in rnd])) for kk in E.BOOT_METRICS}
            rows.append(dict(urutan=oname, strategi=strat, aktivasi=act, abs_dev=abs(act - rho),
                             **{kk: m[kk] for kk in E.BOOT_METRICS},
                             **{f"gain_{kk}": m[kk] - base[kk] for kk in E.BOOT_METRICS}))
    return pd.DataFrame(rows)


# =============================================================================
# A1 — GFLOPs & modul ekspor untuk profil
# =============================================================================
class PostHocGateNoZ(nn.Module):
    """Gate post-hoc TANPA z lama: hanya pooling global + Gemm (router lama tidak dihitung)."""
    def __init__(self, w, b, thr):
        super().__init__()
        self.register_buffer("w", torch.as_tensor(np.asarray(w), dtype=torch.float32))
        self.register_buffer("b", torch.tensor(float(b), dtype=torch.float32))
        self.register_buffer("thr", torch.tensor(float(thr), dtype=torch.float32))

    def score(self, p3, p2, p4, p5):
        f = torch.cat([G._pool(p3), G._pool(p2), G._pool(p4), G._pool(p5)], 1).float()
        return f @ self.w + self.b

    def forward(self, p3, p2, p4, p5):
        return (self.score(p3, p2, p4, p5) > self.thr).float()


class TrunkOnly(nn.Module):
    """Layer 0–15 tanpa gate: keluaran p3, p2, p4, p5 (untuk mengisolasi biaya gate)."""
    def __init__(self, full_model):
        super().__init__()
        self.s = G.Stage1PostHoc(full_model, nn.Identity())

    def forward(self, x):
        return self.s.trunk(x)


def folded_weights(pipe):
    sc, lr = pipe[0], pipe[1]
    scale = np.where(sc.scale_ == 0, 1.0, sc.scale_)
    w = lr.coef_[0] / scale
    return w, float(lr.intercept_[0] - np.sum(lr.coef_[0] * sc.mean_ / scale))


def gflops(module, args):
    from torch.utils.flop_counter import FlopCounterMode
    with torch.no_grad(), FlopCounterMode(display=False) as fc:
        module(*args)
    return fc.get_total_flops() / 1e9


def export_profile_onnx(model, vanilla, p2static, gate_full, gate_noz, x0, out_dir):
    """ONNX untuk profil + GFLOPs per modul (CPU, input 1x3x640x640)."""
    m = E.cpu_copy(model)
    rt = find_router(m)
    g_full = G.PostHocGate(rt, gate_full["w"], gate_full["b"], gate_full["thr"]).eval()
    g_noz = PostHocGateNoZ(gate_noz["w"], gate_noz["b"], gate_noz["thr"]).eval()
    s1_full, s1_noz, trunk = G.Stage1PostHoc(m, g_full).eval(), G.Stage1PostHoc(m, g_noz).eval(), TrunkOnly(m).eval()
    _, s2a, s2b = E.make_stages(m)
    van, p2s = E._StaticOut(E.cpu_copy(vanilla)).eval(), E._StaticOut(E.cpu_copy(p2static)).eval()
    with torch.no_grad():
        p3, p2, p4, p5 = trunk(x0)
    out = Path(out_dir)
    P_ = {k: out / f"prof_{k}.onnx" for k in ("trunk", "s1_full", "s1_noz", "s2a", "s2b", "vanilla", "p2static")}
    feats = ["p3_neck", "p2_backbone", "p4_neck", "p5_sppf"]
    E.export_onnx(trunk, (x0,), P_["trunk"], ["image"], feats)
    E.export_onnx(E._Stage1Out(s1_full), (x0,), P_["s1_full"], ["image"], ["gate"] + feats)
    E.export_onnx(E._Stage1Out(s1_noz), (x0,), P_["s1_noz"], ["image"], ["gate"] + feats)
    E.export_onnx(s2a, (p3, p2, p4, p5), P_["s2a"], feats, ["y_A"])
    E.export_onnx(s2b, (p3, p4, p5), P_["s2b"], ["p3_neck", "p4_neck", "p5_sppf"], ["y_B"])
    E.export_onnx(van, (x0,), P_["vanilla"], ["image"], ["y"])
    E.export_onnx(p2s, (x0,), P_["p2static"], ["image"], ["y"])
    fl = {"trunk 0–15": gflops(trunk, (x0,)),
          "gate lama (z)": gflops(lambda a, b: G._raw_gate_logit(rt, a, b), (p3, p2)),
          "gate post-hoc tanpa z (pool+Gemm)": gflops(g_noz.score, (p3, p2, p4, p5)),
          "Stage 2A (P2 + head A)": gflops(s2a, (p3, p2, p4, p5)),
          "Stage 2B (head B)": gflops(s2b, (p3, p4, p5)),
          "Vanilla utuh": gflops(van, (x0,)), "P2-Static utuh": gflops(p2s, (x0,))}
    # cek numerik ONNX stage1 tanpa z vs PyTorch (gate & fitur)
    import onnxruntime as ort
    sess = ort.InferenceSession(str(P_["s1_noz"]), providers=["CPUExecutionProvider"])
    o = sess.run(None, {"image": x0.numpy()})
    with torch.no_grad():
        ref = s1_noz(x0)
    margin = float((g_noz.score(p3, p2, p4, p5) - g_noz.thr).abs())
    assert float(o[0].flatten()[0]) == float(ref[0].flatten()[0]) or margin < 1e-4, "gate ONNX tanpa z != PyTorch"
    return P_, fl


# =============================================================================
# A1 — profil TensorRT per segmen (dijalankan di proses terpisah: python -m tahap3a job.json)
# =============================================================================
def profile_job_main(job_path):
    import sys
    job = json.loads(Path(job_path).read_text())
    log = lambda *a: print("[profil]", *a, flush=True)
    NS = E.init_trt()
    if NS is None:
        Path(job["out_json"]).write_text(json.dumps({"trt_available": False})); sys.exit(3)
    trt, cuda = NS.trt, NS.cuda
    assert not torch.cuda.is_initialized()

    class Eng:
        def __init__(self, path):
            with open(path, "rb") as f, trt.Runtime(NS.logger) as rt_:
                self.engine = rt_.deserialize_cuda_engine(f.read())
            self.ctx = self.engine.create_execution_context()
            self.inp, self.out, self.d, self.h = [], [], {}, {}
            for i in range(self.engine.num_io_tensors):
                nme = self.engine.get_tensor_name(i)
                (self.inp if self.engine.get_tensor_mode(nme) == trt.TensorIOMode.INPUT else self.out).append(nme)
            for nme in self.inp + self.out:
                shape, dt = tuple(self.engine.get_tensor_shape(nme)), trt.nptype(self.engine.get_tensor_dtype(nme))
                if nme in self.inp:
                    self.ctx.set_input_shape(nme, shape)
                self.h[nme] = cuda.pagelocked_empty(shape, dt)
                self.d[nme] = cuda.mem_alloc(self.h[nme].nbytes)
                self.ctx.set_tensor_address(nme, int(self.d[nme]))

        def bind(self, nme, ptr):
            self.ctx.set_tensor_address(nme, int(ptr))

        def run(self, stream):
            self.ctx.execute_async_v3(stream_handle=stream.handle)

    eng = {}
    for k, p in job["onnx"].items():
        ep = str(Path(p).with_suffix(".engine")); NS.build_engine(p, ep); eng[k] = Eng(ep); log("engine", k)
    stream = cuda.Stream()
    arr = np.load(job["images_npy"])                                   # uint8 (N,3,H,W); konversi di luar timer
    imgs = [np.ascontiguousarray(a[None].astype(np.float32) / 255.0) for a in arr]
    ev = [cuda.Event() for _ in range(8)]

    def t(a, b):
        return ev[b].time_since(ev[a])

    def static(name, im):
        e = eng[name]; w0 = time.perf_counter()
        ev[0].record(stream); cuda.memcpy_htod_async(e.d["image"], im, stream)
        ev[1].record(stream); e.run(stream)
        ev[2].record(stream); cuda.memcpy_dtoh_async(e.h[e.out[0]], e.d[e.out[0]], stream)
        ev[3].record(stream); stream.synchronize()
        return dict(h2d=t(0, 1), exec=t(1, 2), d2h=t(2, 3), gpu_total=t(0, 3), wall=(time.perf_counter() - w0) * 1e3)

    feats_a, feats_b = ("p3_neck", "p2_backbone", "p4_neck", "p5_sppf"), ("p3_neck", "p4_neck", "p5_sppf")

    def routed(s1name, branch, gated):
        s1 = eng[s1name]; s2 = eng["s2a" if branch == "A" else "s2b"]
        for f in (feats_a if branch == "A" else feats_b):
            s2.bind(f, s1.d[f])

        def fn(im):
            w0 = time.perf_counter()
            ev[0].record(stream); cuda.memcpy_htod_async(s1.d["image"], im, stream)
            ev[1].record(stream); s1.run(stream)
            ev[2].record(stream)
            if gated:
                cuda.memcpy_dtoh_async(s1.h["gate"], s1.d["gate"], stream)
            ev[3].record(stream)
            if gated:
                stream.synchronize(); _ = float(s1.h["gate"].flatten()[0])   # titik sinkronisasi host (seperti runtime)
            w_sync = time.perf_counter()
            ev[4].record(stream); s2.run(stream)
            ev[5].record(stream); cuda.memcpy_dtoh_async(s2.h[s2.out[0]], s2.d[s2.out[0]], stream)
            ev[6].record(stream); stream.synchronize()
            return dict(h2d=t(0, 1), stage1=t(1, 2), gate_d2h=t(2, 3), stage2=t(4, 5), d2h=t(5, 6),
                        host_gap=t(3, 4), gpu_total=t(0, 6), wall=(time.perf_counter() - w0) * 1e3,
                        wall_after_sync=(time.perf_counter() - w_sync) * 1e3)
        return fn

    exps = {"Vanilla": lambda im: static("vanilla", im), "P2-Static": lambda im: static("p2static", im)}
    for s1n, lab, gated in (("trunk", "trunk tanpa gate", False), ("s1_full", "gate lengkap (z)", True),
                            ("s1_noz", "gate tanpa z", True)):
        for br in "AB":
            exps[f"rute {br} | {lab}"] = routed(s1n, br, gated)
    for im in imgs[:job["n_warmup"]]:
        for f in exps.values():
            f(im)
    rec = {k: [] for k in exps}
    names = list(exps)
    for rep in range(job["n_reps"]):
        for im in imgs:
            for k in (names if rep % 2 == 0 else names[::-1]):
                rec[k].append(exps[k](im))
    res = {k: pd.DataFrame(v).agg(["mean", "std", "median"]).to_dict() for k, v in rec.items()}
    Path(job["out_json"]).write_text(json.dumps({"trt_available": True, "trt_version": trt.__version__,
                                                 "n": len(imgs) * job["n_reps"], "profil": res}, default=float))
    log("selesai")


if __name__ == "__main__":
    import sys
    profile_job_main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/tahap3a2.py
# [8e] modul (hapus sel ini setelah file di-commit ke repo)
"""
tahap3a2.py — A2 (CUDA Graph per rute), A3 (satu engine dengan kondisional ONNX `If`), A4 (FP16) untuk M0 +
gate post-hoc tanpa z.

Ringkas:
  - Ekspor: Stage 1 (trunk + gate tanpa z), Stage 2A, Stage 2B, Vanilla, P2-Static, dan A3 = satu graf ONNX:
      image, mode -> Stage 1 -> If(cond) { Stage 2A } else { Stage 2B + padding nol ke jumlah anchor A }
      mode < 0: ikut gate; mode = 1: paksa A; mode = 0: paksa B (untuk profil rute paksa di engine yang sama).
    Padding nol (skor kelas 0) dibuang oleh ambang confidence NMS, jadi keluaran setara dengan Stage 2B.
  - Job TensorRT (proses terpisah) memakai PyTorch untuk memori/stream/event/CUDA Graph — TANPA pycuda, sehingga
    hanya ada satu context CUDA (primary) dan CUDA Graph bisa di-capture di stream PyTorch.
    Untuk FP32 dan FP16: latensi (eager vs CUDA Graph) semua varian, paritas A3 vs 3-engine, dan mAP test.
"""
from __future__ import annotations

import copy
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

import tahap2_eval as E
import tahap2c_gate as G
import tahap3a as A3

FEATS_A = ("p3_neck", "p2_backbone", "p4_neck", "p5_sppf")
FEATS_B = ("p3_neck", "p4_neck", "p5_sppf")


# =============================================================================
# EKSPOR ONNX
# =============================================================================
def export_all(model, vanilla, p2static, gate, x0, out_dir):
    """gate: dict w, b, thr (tanpa z). Returns (paths, modul CPU untuk verifikasi)."""
    m = E.cpu_copy(model)
    g = A3.PostHocGateNoZ(gate["w"], gate["b"], gate["thr"]).eval()
    s1 = G.Stage1PostHoc(m, g).eval()
    _, s2a, s2b = E.make_stages(m)
    with torch.no_grad():
        _, p3, p2, p4, p5 = s1(x0)
    out = Path(out_dir)
    P = {k: out / f"{k}.onnx" for k in ("s1", "s2a", "s2b", "vanilla", "p2static")}
    E.export_onnx(E._Stage1Out(s1), (x0,), P["s1"], ["image"], ["gate", *FEATS_A])
    E.export_onnx(s2a, (p3, p2, p4, p5), P["s2a"], list(FEATS_A), ["y_A"])
    E.export_onnx(s2b, (p3, p4, p5), P["s2b"], list(FEATS_B), ["y_B"])
    statics = {"vanilla": E._StaticOut(E.cpu_copy(vanilla)).eval(), "p2static": E._StaticOut(E.cpu_copy(p2static)).eval()}
    for k, mod in statics.items():
        E.export_onnx(mod, (x0,), P[k], ["image"], ["y"])
    with torch.no_grad():
        n_a, n_b = s2a(p3, p2, p4, p5).shape[-1], s2b(p3, p4, p5).shape[-1]
    P["a3"] = out / "a3_if.onnx"
    build_if_onnx(P["s1"], P["s2a"], P["s2b"], P["a3"], n_a, n_b)
    return P, dict(s1=s1, s2a=s2a, s2b=s2b, **statics, n_a=int(n_a), n_b=int(n_b))


class _IOCast(nn.Module):
    """Bobot FP16 di dalam, I/O tetap FP32 (sama dengan engine FP32 -> pipeline host tidak berubah)."""
    def __init__(self, m):
        super().__init__(); self.m = m

    def forward(self, *xs):
        out = self.m(*[x.half() for x in xs])
        return tuple(o.float() for o in out) if isinstance(out, (tuple, list)) else out.float()


def _half_copy(m, device):
    c = copy.deepcopy(m).to(device).half().eval()
    for mod in c.modules():
        if hasattr(mod, "anchors") and hasattr(mod, "shape"):
            mod.shape = None
            mod.anchors, mod.strides = torch.empty(0), torch.empty(0)
    return c


def export_fp16(model, vanilla, p2static, gate, x0, out_dir, device):
    """
    ONNX FP16 (I/O FP32) untuk A4. TensorRT 11 menghapus BuilderFlag.FP16 (jaringan bertipe kuat), sehingga presisi
    ditentukan oleh tipe di ONNX. Gate (pooling + Gemm 960) DIBIARKAN FP32 agar keputusan routing tidak berubah karena
    pembulatan bobot gate; trunk, Stage 2 dan baseline FP16. Ekspor di GPU (operator half di CPU tidak lengkap).
    """
    m = _half_copy(model, device)
    g = A3.PostHocGateNoZ(gate["w"], gate["b"], gate["thr"]).to(device).float().eval()

    class S1(nn.Module):                                   # trunk FP16 -> fitur FP32 -> gate FP32
        def __init__(self):
            super().__init__(); self.s = G.Stage1PostHoc(m, nn.Identity())
            self.gate = g
        def forward(self, x):
            p3, p2, p4, p5 = (t.float() for t in self.s.trunk(x.half()))
            return self.gate(p3, p2, p4, p5), p3, p2, p4, p5
    s1 = S1().eval()
    _, s2a, s2b = E.make_stages(m)
    x = x0.to(device)
    with torch.no_grad():
        _, p3, p2, p4, p5 = s1(x)
        h = [t.half() for t in (p3, p2, p4, p5)]
        s2a(*h); s2b(h[0], h[2], h[3])                    # hitung anchors Detect (half) di luar tracing
    out = Path(out_dir)
    P = {k: out / f"{k}_fp16.onnx" for k in ("s1", "s2a", "s2b", "vanilla", "p2static")}
    E.export_onnx(s1, (x,), P["s1"], ["image"], ["gate", *FEATS_A])
    E.export_onnx(_IOCast(s2a), (p3, p2, p4, p5), P["s2a"], list(FEATS_A), ["y_A"])
    E.export_onnx(_IOCast(s2b), (p3, p4, p5), P["s2b"], list(FEATS_B), ["y_B"])
    for k, src in (("vanilla", vanilla), ("p2static", p2static)):
        mod = _IOCast(E._StaticOut(_half_copy(src, device))).eval()
        with torch.no_grad():
            mod(x)                                        # anchors di luar tracing
        E.export_onnx(mod, (x,), P[k], ["image"], ["y"])
    with torch.no_grad():
        n_a, n_b = s2a(p3.half(), p2.half(), p4.half(), p5.half()).shape[-1], s2b(p3.half(), p4.half(), p5.half()).shape[-1]
    P["a3"] = out / "a3_if_fp16.onnx"
    build_if_onnx(P["s1"], P["s2a"], P["s2b"], P["a3"], n_a, n_b)
    return P


def _prefixed_graph(model, prefix, keep):
    """Salin graph dengan semua nama tensor/node diberi prefix, kecuali nama di `keep` (input dari scope luar)."""
    import onnx
    g = model.graph
    ren = lambda n: n if (n == "" or n in keep) else prefix + n
    nodes = []
    for nd in g.node:
        new = onnx.NodeProto(); new.CopyFrom(nd)
        new.name = prefix + (nd.name or nd.op_type)
        del new.input[:]; new.input.extend([ren(i) for i in nd.input])
        del new.output[:]; new.output.extend([ren(o) for o in nd.output])
        nodes.append(new)
    inits = []
    for t in g.initializer:
        new = onnx.TensorProto(); new.CopyFrom(t); new.name = ren(t.name); inits.append(new)
    return nodes, inits, [ren(o.name) for o in g.output]


def build_if_onnx(p_s1, p_s2a, p_s2b, out_path, n_a, n_b):
    import onnx
    from onnx import TensorProto, helper, numpy_helper
    m1, ma, mb = onnx.load(str(p_s1)), onnx.load(str(p_s2a)), onnx.load(str(p_s2b))
    no = int(ma.graph.output[0].type.tensor_type.shape.dim[1].dim_value)
    keep = set(FEATS_A)
    na, ia, (ya,) = _prefixed_graph(ma, "A__", keep)
    nb, ib, (yb,) = _prefixed_graph(mb, "B__", keep)
    pad = numpy_helper.from_array(np.zeros((1, no, n_a - n_b), np.float32), "B__pad_zeros")
    nb.append(helper.make_node("Concat", [yb, "B__pad_zeros"], ["B__y_pad"], axis=2, name="B__pad_concat"))
    vi = lambda n: helper.make_tensor_value_info(n, TensorProto.FLOAT, [1, no, n_a])
    then_g = helper.make_graph(na, "rute_A", [], [vi(ya)], initializer=ia)
    else_g = helper.make_graph(nb, "rute_B", [], [vi("B__y_pad")], initializer=ib + [pad])
    g1 = m1.graph
    nodes = list(g1.node) + [
        helper.make_node("Constant", [], ["c_zero"], value=numpy_helper.from_array(np.zeros(1, np.float32))),
        helper.make_node("Constant", [], ["c_half"], value=numpy_helper.from_array(np.full(1, 0.5, np.float32))),
        helper.make_node("Constant", [], ["c_ax0"], value=numpy_helper.from_array(np.zeros(1, np.int64))),
        helper.make_node("Less", ["mode", "c_zero"], ["is_auto"]),
        helper.make_node("Reshape", ["gate", "c_shape1"], ["gate1"]),
        helper.make_node("Where", ["is_auto", "gate1", "mode"], ["sel"]),
        helper.make_node("Greater", ["sel", "c_half"], ["cond1"]),
        helper.make_node("Squeeze", ["cond1", "c_ax0"], ["cond"]),
        helper.make_node("If", ["cond"], ["y"], then_branch=then_g, else_branch=else_g, name="rute_if"),
        helper.make_node("Cast", ["cond1"], ["use_a"], to=TensorProto.FLOAT),
    ]
    inits = list(g1.initializer) + [numpy_helper.from_array(np.ones(1, np.int64), "c_shape1")]
    graph = helper.make_graph(
        nodes, "router_a3",
        [g1.input[0], helper.make_tensor_value_info("mode", TensorProto.FLOAT, [1])],
        [vi("y"), helper.make_tensor_value_info("use_a", TensorProto.FLOAT, [1])],
        initializer=inits)
    model = helper.make_model(graph, opset_imports=list(m1.opset_import), producer_name="tahap3a2")
    model.ir_version = m1.ir_version
    onnx.checker.check_model(model)
    onnx.save(model, str(out_path))


def verify_onnx_all(P, mods, loader, n=5, tol=1e-4):
    """ORT vs PyTorch: semua stage + A3 (mode -1/0/1) termasuk keputusan gate; A3 harus = 3-engine."""
    import onnxruntime as ort
    sess = {k: ort.InferenceSession(str(p), providers=["CPUExecutionProvider"]) for k, p in P.items()}
    rel = lambda a, b: float(np.abs(a - b).max() / (np.abs(b).max() + 1e-12))
    worst, n_a, n_b = {}, mods["n_a"], mods["n_b"]
    up = lambda k, v: worst.__setitem__(k, max(worst.get(k, 0.0), v))
    with torch.no_grad():
        for k, b in enumerate(loader):
            if k >= n:
                break
            x = b["img"].float() / 255.0
            g, p3, p2, p4, p5 = mods["s1"](x)
            o1 = sess["s1"].run(None, {"image": x.numpy()})
            if float(o1[0].ravel()[0]) != float(g.ravel()[0]):
                margin = float((mods["s1"].gate.score(p3, p2, p4, p5) - mods["s1"].gate.thr).abs())
                assert margin < 1e-4, f"gate ONNX != PyTorch (margin {margin:.2e})"
            up("s1", max(rel(a, t.numpy()) for a, t in zip(o1[1:], (p3, p2, p4, p5))))
            f = dict(zip(FEATS_A, (o1[1], o1[2], o1[3], o1[4])))
            ya = sess["s2a"].run(None, f)[0]; yb = sess["s2b"].run(None, {k_: f[k_] for k_ in FEATS_B})[0]
            up("s2a", rel(ya, mods["s2a"](p3, p2, p4, p5).numpy())); up("s2b", rel(yb, mods["s2b"](p3, p4, p5).numpy()))
            for nm in ("vanilla", "p2static"):
                up(nm, rel(sess[nm].run(None, {"image": x.numpy()})[0], mods[nm](x).numpy()))
            for mode in (-1.0, 0.0, 1.0):
                y, use_a = sess["a3"].run(None, {"image": x.numpy(), "mode": np.array([mode], np.float32)})
                want_a = (float(o1[0].ravel()[0]) > 0.5) if mode < 0 else (mode == 1.0)
                assert bool(use_a[0] > 0.5) == want_a, f"A3 memilih rute salah (mode {mode})"
                ref = ya if want_a else np.concatenate([yb, np.zeros((1, yb.shape[1], n_a - n_b), np.float32)], 2)
                up(f"a3_mode{int(mode)}", float(np.abs(y - ref).max()))
    assert all(v < tol for v in worst.values()), f"Selisih ONNX > {tol}: {worst}"
    return worst


# =============================================================================
# JOB TENSORRT (python -m tahap3a2 job.json) — PyTorch untuk memori/stream/CUDA Graph, tanpa pycuda
# =============================================================================
def job_main(job_path):
    import sys
    import tensorrt as trt
    job = json.loads(Path(job_path).read_text())
    T0 = time.perf_counter()
    log = lambda *a: print(f"[trt-job {time.perf_counter() - T0:6.0f}s]", *a, flush=True)
    E.COCO_MAX_DETS = list(job.get("coco_max_dets", E.COCO_MAX_DETS))
    logger = trt.Logger(trt.Logger.WARNING)
    dev = torch.device("cuda:0")
    torch.cuda.init()
    T2T = {trt.float32: torch.float32, trt.float16: torch.float16, trt.int32: torch.int32,
           trt.int64: torch.int64, trt.bool: torch.bool}

    def build(onnx_path, engine_path, fp16):
        builder = trt.Builder(logger)
        flags = 0
        if hasattr(trt.NetworkDefinitionCreationFlag, "STRONGLY_TYPED"):
            flags |= 1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED)   # presisi = tipe di ONNX
        net = builder.create_network(flags); parser = trt.OnnxParser(net, logger)
        with open(onnx_path, "rb") as f:
            if not parser.parse(f.read()):
                raise RuntimeError("\n".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
        cfg = builder.create_builder_config()
        cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
        if fp16 and not flags and hasattr(trt.BuilderFlag, "FP16"):   # TensorRT lama tanpa jaringan bertipe kuat
            cfg.set_flag(trt.BuilderFlag.FP16)
        t = time.perf_counter(); eb = builder.build_serialized_network(net, cfg)
        assert eb is not None, f"gagal build {onnx_path} fp16={fp16}"
        Path(engine_path).write_bytes(eb)
        return time.perf_counter() - t

    class Eng:
        def __init__(self, path):
            self.rt = trt.Runtime(logger)
            self.engine = self.rt.deserialize_cuda_engine(Path(path).read_bytes())
            self.ctx = self.engine.create_execution_context()
            self.inp, self.out, self.buf = [], [], {}
            for i in range(self.engine.num_io_tensors):
                n = self.engine.get_tensor_name(i)
                (self.inp if self.engine.get_tensor_mode(n) == trt.TensorIOMode.INPUT else self.out).append(n)
            for n in self.inp + self.out:
                shape = tuple(self.engine.get_tensor_shape(n))
                if n in self.inp:
                    self.ctx.set_input_shape(n, shape)
                self.buf[n] = torch.zeros(shape, dtype=T2T[self.engine.get_tensor_dtype(n)], device=dev)
                self.ctx.set_tensor_address(n, self.buf[n].data_ptr())

        def bind(self, n, t):
            self.ctx.set_tensor_address(n, t.data_ptr())

        def run(self, s):
            assert self.ctx.execute_async_v3(s.cuda_stream), "execute_async_v3 gagal"

    s = torch.cuda.Stream()
    pin = lambda t: torch.zeros(t.shape, dtype=t.dtype).pin_memory()
    res = {"trt_version": trt.__version__, "gpu": torch.cuda.get_device_name(0), "build_s": {}, "latency": {},
           "graph_capture": {}, "acc": {}, "paritas": {}, "keputusan": {}}

    # ---------- runner: eager & graph untuk satu presisi ----------
    def make_runners(eng):
        R = {}
        h_img = pin(eng["vanilla"].buf["image"])

        def static_runner(e):
            ho = pin(e.buf[e.out[0]])
            def ops():
                e.buf["image"].copy_(h_img, non_blocking=True); e.run(s); ho.copy_(e.buf[e.out[0]], non_blocking=True)
            return ops, ho
        for nm in ("vanilla", "p2static"):
            R[nm] = static_runner(eng[nm])

        s1, s2a, s2b = eng["s1"], eng["s2a"], eng["s2b"]
        for f in FEATS_A:
            s2a.bind(f, s1.buf[f])
        for f in FEATS_B:
            s2b.bind(f, s1.buf[f])
        h_gate, hya, hyb = pin(s1.buf["gate"]), pin(s2a.buf["y_A"]), pin(s2b.buf["y_B"])

        def ops1():
            s1.buf["image"].copy_(h_img, non_blocking=True); s1.run(s); h_gate.copy_(s1.buf["gate"], non_blocking=True)
        def ops2a():
            s2a.run(s); hya.copy_(s2a.buf["y_A"], non_blocking=True)
        def ops2b():
            s2b.run(s); hyb.copy_(s2b.buf["y_B"], non_blocking=True)
        R["chain"] = (ops1, ops2a, ops2b, h_gate, hya, hyb)

        if "a3" not in eng:
            return R, h_img
        a3 = eng["a3"]; h_y3, h_use = pin(a3.buf["y"]), pin(a3.buf["use_a"])
        modes = {mv: torch.full((1,), mv, dtype=torch.float32, device=dev) for mv in (-1.0, 0.0, 1.0)}
        def ops3_for(mv):
            def ops3():
                a3.bind("mode", modes[mv])                # alamat input mode tetap per varian (ikut ter-capture)
                a3.buf["image"].copy_(h_img, non_blocking=True); a3.run(s)
                h_y3.copy_(a3.buf["y"], non_blocking=True); h_use.copy_(a3.buf["use_a"], non_blocking=True)
            return ops3
        R["a3"] = (ops3_for, h_y3, h_use, modes)
        return R, h_img

    def capture(ops, tag):
        with torch.cuda.stream(s):
            ops(); ops()                                  # pemanasan sebelum capture (wajib untuk TensorRT)
        s.synchronize()
        g = torch.cuda.CUDAGraph()
        try:
            with torch.cuda.graph(g, stream=s):
                ops()
            s.synchronize()
            return g, "ok"
        except Exception as ex:                           # mis. engine dengan kondisional tidak bisa di-capture
            torch.cuda.synchronize()
            return None, f"{type(ex).__name__}: {str(ex)[:200]}"

    def variants(R, mode_graph):
        """dict nama -> fungsi(img_np) -> (keluaran host, info). Semua cakupan: H2D gambar .. D2H keluaran + sync."""
        V = {}
        def launcher(ops, key):
            if not mode_graph:
                return ops
            if key not in graphs:
                graphs[key] = capture(ops, key)
                res["graph_capture"][key] = graphs[key][1]
            g = graphs[key][0]
            return None if g is None else g.replay
        for nm in ("vanilla", "p2static"):
            ops, ho = R[nm]; L = launcher(ops, f"{prec}|{nm}")
            if L is not None:
                V[{"vanilla": "Vanilla", "p2static": "P2-Static"}[nm]] = (lambda L=L, ho=ho: (L(), s.synchronize(), ho)[2])
        ops1, ops2a, ops2b, h_gate, hya, hyb = R["chain"]
        L1, LA, LB = launcher(ops1, f"{prec}|s1"), launcher(ops2a, f"{prec}|s2a"), launcher(ops2b, f"{prec}|s2b")
        if None not in (L1, LA, LB):
            def chain(force, L1=L1, LA=LA, LB=LB):
                L1(); s.synchronize()                     # titik sinkronisasi host: baca skalar gate
                use_a = (float(h_gate.reshape(-1)[0]) > 0.5) if force is None else force == "A"
                (LA if use_a else LB)(); s.synchronize()
                return hya if use_a else hyb, use_a
            V["3-engine rute A"] = lambda: chain("A")[0]
            V["3-engine rute B"] = lambda: chain("B")[0]
            V["3-engine router"] = lambda: chain(None)[0]
        if "a3" not in R:
            return V
        ops3_for, h_y3, h_use, _ = R["a3"]
        for lab, mv in (("A3 rute A", 1.0), ("A3 rute B", 0.0), ("A3 router", -1.0)):
            L3 = launcher(ops3_for(mv), f"{prec}|a3|{mv}")
            if L3 is not None:
                V[lab] = (lambda L3=L3: (L3(), s.synchronize(), h_y3)[2])
        return V

    # ---------- data ----------
    from ultralytics.data.utils import check_det_dataset
    DATA = check_det_dataset(job["data_yaml"])
    loader = E.make_loader(DATA, job["split"], 1, job["imgsz"], workers=2, pin_memory=False)
    prof_imgs = []
    for k, b in enumerate(loader):
        if k >= job["n_prof_images"]:
            break
        prof_imgs.append((b["img"].float() / 255.0).numpy())

    graphs = {}
    n_builds = int(job.get("n_builds", 1))                 # uji variasi build: engine dibangun ulang n kali
    for prec, bi in [(p_, b_) for p_ in job["precisions"] for b_ in range(n_builds)]:
        fp16 = prec == "fp16"
        eng = {}
        for k, p in job["onnx"][prec].items():
            ep = str(Path(p).with_name(f"{Path(p).stem}_{prec}_b{bi}.engine"))
            try:
                res["build_s"][f"{prec}|b{bi}|{k}"] = build(p, ep, fp16); eng[k] = Eng(ep); log("engine", prec, f"build {bi}", k)
            except Exception as ex:
                if k != "a3":
                    raise
                res.setdefault("build_error", {})[f"{prec}|a3"] = f"{type(ex).__name__}: {str(ex)[:500]}"
                log("A3 GAGAL dibangun:", res["build_error"][f"{prec}|a3"])
        R, h_img = make_runners(eng)
        V = {}
        for mode_graph in ((False, True) if job.get("graph", True) else (False,)):
            for nm, fn in variants(R, mode_graph).items():
                V[f"{nm} | {'graph' if mode_graph else 'eager'}"] = fn

        def run_fn(fn, im):
            h_img.copy_(torch.from_numpy(im))            # di luar timer
            torch.cuda.synchronize()
            w0 = time.perf_counter()
            with torch.cuda.stream(s):
                out = fn()
            return out, (time.perf_counter() - w0) * 1e3

        # sanity: graph == eager (keluaran) pada 3 gambar
        for nm in [k for k in V if k.endswith("| graph")]:
            base = nm.replace("| graph", "| eager")
            d = 0.0
            for im in prof_imgs[:3]:
                a, _ = run_fn(V[base], im); a = a.clone()
                g_, _ = run_fn(V[nm], im)
                d = max(d, float((a.float() - g_.float()).abs().max()))
            res["paritas"][f"{prec}|graph vs eager|{nm}"] = d
        for im in prof_imgs[:job["n_warmup"]]:
            for fn in V.values():
                run_fn(fn, im)
        rec = {k: [] for k in V}; names = list(V)
        for rep in range(job["n_reps"]):
            for im in prof_imgs:
                for k in (names if rep % 2 == 0 else names[::-1]):
                    rec[k].append(run_fn(V[k], im)[1])
        for k, v in rec.items():
            a = np.asarray(v)
            st_ = dict(mean=a.mean(), std=a.std(), p50=np.percentile(a, 50), p95=np.percentile(a, 95), n=int(a.size))
            res.setdefault("latency_builds", {})[f"{prec}|b{bi}|{k}"] = st_
            if bi == 0:
                res["latency"][f"{prec}|{k}"] = st_
        log("latensi", prec, f"build {bi}", "selesai")
        if bi > 0 or not job.get("acc_eval", True):        # akurasi hanya pada build pertama
            Path(job["out_json"]).write_text(json.dumps(res, default=float, indent=1))
            del eng, R, V; graphs.clear(); torch.cuda.synchronize(); torch.cuda.empty_cache()
            continue

        # ---------- akurasi (test penuh) + paritas A3 vs 3-engine ----------
        ev = {k: job[k] for k in ("data_yaml", "conf", "iou", "max_det", "imgsz")}; ev["report_split"] = job["split"]
        from pycocotools.coco import COCO
        import contextlib, io
        with contextlib.redirect_stdout(io.StringIO()):
            gt = COCO(job["gt_json"])
        dec_chain, dec_a3 = [], []
        n_a, n_b = job["n_a"], job["n_b"]

        def pred(fn_key, track=None):
            fn = V[f"{fn_key} | eager"]
            def f(im):
                out, _ = run_fn(fn, np.ascontiguousarray(im.cpu().numpy(), dtype=np.float32))
                y = out.float().cpu().clone()
                info = {}
                if track == "chain":
                    use_a = y.shape[-1] == n_a
                    info["branch"] = "A" if use_a else "B"; dec_chain.append(use_a)
                elif track == "a3":
                    use_a = bool(float(R["a3"][2].reshape(-1)[0]) > 0.5); dec_a3.append(use_a); info["branch"] = "A" if use_a else "B"
                return [y], [info]
            return f
        acc = {}
        for lab, key, tr in (("Vanilla", "Vanilla", None), ("P2-Static", "P2-Static", None),
                             ("Router 3-engine", "3-engine router", "chain"), ("Router A3", "A3 router", "a3")):
            if f"{key} | eager" not in V:
                continue
            r = E.run_eval(lab, pred(key, tr), loader, DATA, ev, torch.device("cpu"), save_json=True, verbose=False)
            acc[lab] = {**E.summarize_acc(r), **{f"coco_{m}": v for m, v in E.coco_eval(gt, r["jdict"])[0].items()}}
            log(prec, lab, "mAP50", round(acc[lab]["mAP50"], 4), "| mAP_small", round(acc[lab].get("coco_mAP_small", float("nan")), 4))
        res["acc"][prec] = acc
        res["keputusan"][prec] = dict(aktivasi_3engine=float(np.mean(dec_chain)),
                                      aktivasi_a3=float(np.mean(dec_a3)) if dec_a3 else None,
                                      setuju_a3_vs_3engine=float(np.mean(np.asarray(dec_chain) == np.asarray(dec_a3))) if dec_a3 else None,
                                      keputusan_3engine=[int(x) for x in dec_chain])
        Path(job["out_json"]).write_text(json.dumps(res, default=float, indent=1))      # simpan sementara
        log("hasil", prec, "tersimpan")
        del eng, R, V; graphs.clear(); torch.cuda.synchronize(); torch.cuda.empty_cache()

    if len(job["precisions"]) == 2:
        a, b = np.asarray(res["keputusan"]["fp32"]["keputusan_3engine"]), np.asarray(res["keputusan"]["fp16"]["keputusan_3engine"])
        res["keputusan"]["setuju_fp16_vs_fp32"] = float((a == b).mean())
    Path(job["out_json"]).write_text(json.dumps(res, default=float, indent=1))
    log("selesai")


def latency_table(res):
    rows = []
    for k, v in res["latency"].items():
        prec, rest = k.split("|", 1)
        name, mode = rest.rsplit("|", 1)
        rows.append(dict(presisi=prec, varian=name.strip(), mode=mode.strip(), mean=v["mean"], p50=v["p50"], p95=v["p95"], std=v["std"]))
    return pd.DataFrame(rows).pivot_table(index=["presisi", "varian"], columns="mode", values=["mean", "p50"]).round(3)


if __name__ == "__main__":
    import sys
    job_main(sys.argv[1])

In [ ]:
%%writefile /kaggle/working/thesis-Difficulty-Aware-Router/tahap4b.py
# [8f] modul Tahap 4B (dibangkitkan oleh buat_tahap4b.py dari tahap4_final.py)
"""
tahap4b.py — Tahap 4B: SENSITIVITAS pasca-hasil atas Tahap 4 (dibangkitkan oleh buat_tahap4b.py dari tahap4_final.py).
Tambahan: latensi ujung-ke-ujung B2 + A3 (K-6) dan pilihan checkpoint baseline (K-7, di notebook). Sisanya identik.

Isi asli tahap4_final.py:
tahap4_final.py — Evaluasi final sistem router (protokol dikunci 16 Okt 2026, sebelum dijalankan).

Sistem final (tidak ada pemilihan apa pun di notebook ini):
  detektor dual-branch M0 (beku, last.pt)
  + gate post-hoc tanpa z: [GAP‖GMP](p3, p2, p4, p5) = 960 fitur, StandardScaler + LogReg C = 0.001 (target soft_small
    biner dari train), dilipat menjadi satu Gemm
  + kendali anggaran B2: threshold = kuantil (1 − ρ) skor N = 200 gambar sebelumnya (ρ = 0.5); < 20 riwayat -> kuantil val
  + deployment A3 FP16: SATU engine TensorRT dengan ONNX `If`; threshold = INPUT engine, skor = OUTPUT engine,
    sehingga B2 berjalan di host tanpa engine kedua.

Pembanding:
  Vanilla, P2-Static, Branch A / B paksa, routing acak (aktivasi sama), campuran acak Vanilla/P2-Static pada latensi
  sama (H6), heuristik kepadatan tanpa label di Stage 1 (head P3 Detect_B, B2 yang sama), kaskade bergate
  Vanilla -> P2-Static (eksploratif; latensi dirakit dari komponen terukur).

Akurasi utama = keluaran TensorRT FP16 (sistem yang di-deploy). PyTorch FP32 = pemeriksaan silang.
Semua metrik routing dirakit dari evalImgs COCO per gambar (CocoMix, Tahap 2b), sehingga satu COCOeval per keluaran
cukup untuk ratusan routing. Bootstrap berpasangan per gambar memakai indeks resample yang SAMA untuk semua model
(boot_seed tetap) -> selisih antar-seed KITTI (test sama) bisa dirata-rata per resample di ringkasan gabungan.

Job TensorRT berjalan di proses terpisah (python -m tahap4_final job.json), memakai PyTorch untuk memori/stream.
Backend "ort" (ONNX Runtime CPU, graf FP32) memakai alur yang sama persis; dipakai untuk uji lokal tanpa GPU.
"""
from __future__ import annotations

import collections
import contextlib
import gc
import glob
import io
import json
import math
import multiprocessing as mp
import pickle
import subprocess
import sys
import threading
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

import tahap2_eval as E
import tahap2b_plafon as P
import tahap2c_gate as G
import tahap3a as A3
import tahap3a2 as X

FEATS_A, FEATS_B = X.FEATS_A, X.FEATS_B
SMALL_PX = 32.0          # batas "kecil" COCO (piksel asli)
HEUR_CONF = 0.25         # ambang skor kelas maks anchor P3 (dikunci; sama dengan proksi kepadatan D2b)
HEUR_EPS = 0.01          # bobot pemecah seri: skor = log(1 + n_kecil) + HEUR_EPS · log(1 + n_semua)
METRICS = E.BOOT_METRICS  # ("mAP50-95", "mAP50", "mAP_small")
H6_SENS = "[sensitivitas, pasca-hasil] H6 | router (B2 efisien) − campuran acak V/P2S (latensi sama)"


def _pool(t):
    return torch.cat([t.mean((2, 3)), t.amax((2, 3))], 1)


def pool4(p3, p2, p4, p5):
    return torch.cat([_pool(p3), _pool(p2), _pool(p4), _pool(p5)], 1).float()


# =============================================================================
# MODUL STAGE 1: gate post-hoc (skor) dan heuristik kepadatan P3 (tanpa label)
# =============================================================================
class P3Density(nn.Module):
    """
    Heuristik kepadatan objek kecil TANPA label dan TANPA fit, dihitung di Stage 1 dari head P3 Detect_B
    (layer 15, stride 8): n_semua = jumlah anchor P3 dengan skor kelas maks ≥ HEUR_CONF; n_kecil = yang luas kotak
    hasil decode-nya < small_area (piksel letterbox², diisi host = 32² · gain_h · gain_w = "kecil" COCO di piksel asli).
    skor = log(1 + n_kecil) + HEUR_EPS · log(1 + n_semua). Tanpa NMS: jumlah anchor ≠ jumlah objek, tetapi untuk
    PENGURUTAN yang penting monotonisitasnya (diperiksa terhadap versi deteksi lengkap D2b).
    """
    def __init__(self, det, conf=HEUR_CONF, eps=HEUR_EPS):
        super().__init__()
        self.cv2, self.cv3, self.dfl = det.cv2[0], det.cv3[0], det.dfl
        self.reg_max, self.stride = int(det.reg_max), float(det.stride[0])
        assert self.stride == 8.0, f"head P3 Detect_B harus stride 8, bukan {self.stride}"
        self.logit_thr, self.eps = float(math.log(conf / (1 - conf))), float(eps)

    def counts(self, p3, small_area):
        box, cls = self.cv2(p3), self.cv3(p3)
        b, _, h, w = cls.shape
        hit = cls.amax(1).reshape(b, h * w) > self.logit_thr                      # sigmoid(c) > conf  <=>  c > logit(conf)
        d = self.dfl(box.reshape(b, 4 * self.reg_max, h * w)).float() * self.stride   # jarak l, t, r, b (piksel letterbox)
        area = (d[:, 0] + d[:, 2]) * (d[:, 1] + d[:, 3])
        small = area < small_area.float().reshape(-1, 1)
        return hit.float().sum(1), (hit & small).float().sum(1)

    def combine(self, n_all, n_small):
        return torch.log(1.0 + n_small) + self.eps * torch.log(1.0 + n_all)

    def forward(self, p3, small_area):
        return self.combine(*self.counts(p3, small_area))


class S1Gate(nn.Module):
    """image -> (score, p3, p2, p4, p5). Trunk ikut dtype model (FP32/FP16); fitur, Gemm gate dan skor FP32."""
    def __init__(self, model, w, b):
        super().__init__()
        self.s = G.Stage1PostHoc(model, nn.Identity())
        self.register_buffer("w", torch.as_tensor(np.asarray(w), dtype=torch.float32))
        self.register_buffer("b", torch.tensor([float(b)], dtype=torch.float32))
        self._dt = next(model.parameters()).dtype

    def forward(self, x):
        p = [t.float() for t in self.s.trunk(x.to(self._dt))]
        return (pool4(*p) @ self.w + self.b, *p)


class S1Heur(nn.Module):
    """(image, small_area) -> (score, p3, p2, p4, p5); skor = P3Density (head P3 Detect_B di Stage 1)."""
    def __init__(self, model):
        super().__init__()
        self.s = G.Stage1PostHoc(model, nn.Identity())
        self.dens = P3Density(list(model.model)[33])
        self._dt = next(model.parameters()).dtype

    def forward(self, x, small_area):
        p = self.s.trunk(x.to(self._dt))
        return (self.dens(p[0], small_area), *[t.float() for t in p])


@torch.no_grad()
def static_with_feats(m, x):
    """Model statis (Vanilla/P2-Static) -> (y, (p3 = layer 15, p2 = 2, p4 = 12, p5 = 9)) untuk gate kaskade."""
    y = []
    for mod in m.model:
        if mod.f != -1:
            x = y[mod.f] if isinstance(mod.f, int) else [x if j == -1 else y[j] for j in mod.f]
        x = mod(x)
        y.append(x)
    out = x[0] if isinstance(x, (tuple, list)) else x
    return out, (y[15], y[2], y[12], y[9])


def small_area_of(ratio_pad):
    (gh, gw), _ = ratio_pad
    return SMALL_PX ** 2 * float(gh) * float(gw)


# =============================================================================
# CACHE PyTorch (train / val / test)
# =============================================================================
@torch.no_grad()
def cache_split(model, loader, device, conf, iou, max_det, dets=True, heur=True, statics=None, max_images=None,
                log_every=1000, log=print):
    """Satu forward per gambar (letterbox persegi, batch 1, urutan loader). Returns dict list/array per gambar."""
    trunk = G.Stage1PostHoc(model, nn.Identity()).eval()
    _, s2a, s2b = E.make_stages(model)
    dens = P3Density(list(model.model)[33]).eval() if heur else None
    van, p2s = statics if statics else (None, None)
    keys = ("file", "image_id", "feat", "detA", "detB", "heur", "heur_n_all", "heur_n_small", "small_area",
            "featV", "detV", "detP")
    out = {k: [] for k in keys}
    t0 = time.perf_counter()
    for i, b in enumerate(loader):
        if max_images and i >= max_images:
            break
        im = b["img"].to(device).float() / 255.0
        rp, osh = b["ratio_pad"][0], b["ori_shape"][0]
        sa = small_area_of(rp)
        p3, p2, p4, p5 = trunk.trunk(im)
        out["feat"].append(pool4(p3, p2, p4, p5)[0].cpu().numpy())
        if dets:
            out["detA"].append(P._to_dets(s2a(p3, p2, p4, p5), rp, osh, conf, iou, max_det))
            out["detB"].append(P._to_dets(s2b(p3, p4, p5), rp, osh, conf, iou, max_det))
        if dens is not None:
            na, ns = dens.counts(p3, torch.tensor([sa], device=device))
            out["heur"].append(float(dens.combine(na, ns)[0])); out["heur_n_all"].append(float(na[0]))
            out["heur_n_small"].append(float(ns[0]))
        out["small_area"].append(sa)
        if van is not None:
            yV, fV = static_with_feats(van, im)
            out["featV"].append(pool4(*fV)[0].cpu().numpy())
            out["detV"].append(P._to_dets(yV, rp, osh, conf, iou, max_det))
            yP = p2s(im)
            out["detP"].append(P._to_dets(yP[0] if isinstance(yP, (tuple, list)) else yP, rp, osh, conf, iou, max_det))
        stem = Path(b["im_file"][0]).stem
        out["file"].append(Path(b["im_file"][0]).name)
        out["image_id"].append(int(stem) if stem.isnumeric() else stem)
        if log_every and (i + 1) % log_every == 0:
            log(f"  {i + 1} gambar | {time.perf_counter() - t0:.0f} s")
    for k in ("feat", "featV"):
        out[k] = np.stack(out[k]) if out[k] else None
    for k in ("heur", "heur_n_all", "heur_n_small", "small_area"):
        out[k] = np.asarray(out[k], float)
    return out


def jdict_list(dets, image_ids):
    js = []
    for d, iid in zip(dets, image_ids):
        for x, y, w, h, s, c in d:
            js.append({"image_id": iid, "category_id": int(c), "bbox": [float(x), float(y), float(w), float(h)], "score": float(s)})
    return js


def coco_of(gt, dets, image_ids):
    with contextlib.redirect_stdout(io.StringIO()):
        return E.coco_eval(gt, jdict_list(dets, image_ids))


def router_of(gt, dets_a, dets_b, image_ids):
    """P.Router: use_a True -> dets_a. Returns (router, stats_a, stats_b)."""
    sa, ea = coco_of(gt, dets_a, image_ids)
    sb, eb = coco_of(gt, dets_b, image_ids)
    return P.Router(ea, eb, image_ids), sa, sb


def fit_gate(X_, delta, C):
    X_ = np.asarray(X_, np.float64)                          # fit & cek lipatan di float64 (fitur cache float32)
    pipe = A3.fit_logreg(X_, delta, C)
    w, b = A3.folded_weights(pipe)
    ref = pipe.decision_function(X_[:256])
    assert np.allclose(X_[:256] @ w + b, ref, atol=1e-6 * max(1.0, np.abs(ref).max())), "lipatan scaler + logreg salah"
    y = (np.asarray(delta) > 0)
    return dict(w=np.asarray(w, np.float64), b=float(b), info=dict(C=C, n_train=int(len(y)), pos_rate=float(y.mean()),
                                                                   n_fitur=int(X_.shape[1])))


# =============================================================================
# EKSPOR ONNX: A3 v2 (threshold = input, skor = output)
# =============================================================================
def build_if_v2(p_s1, p_s2a, p_s2b, out_path, n_a, n_b):
    """
    Satu graf: (input Stage 1 [image, (small_area)], mode, thr) -> Stage 1 -> If(cond) {Stage 2A} else {Stage 2B + pad 0}.
      cond = mode < 0 ? (score > thr) : (mode > 0.5)       (mode −1 ikut skor, 1 paksa A, 0 paksa B)
    Keluaran: y (1, 4+nc, n_a), use_a (1,), score (1,). Padding nol (skor kelas 0) dibuang ambang conf NMS.
    """
    import onnx
    from onnx import TensorProto, helper, numpy_helper
    m1, ma, mb = onnx.load(str(p_s1)), onnx.load(str(p_s2a)), onnx.load(str(p_s2b))
    g1 = m1.graph
    assert [o.name for o in g1.output] == ["score", *FEATS_A], [o.name for o in g1.output]
    no = int(ma.graph.output[0].type.tensor_type.shape.dim[1].dim_value)
    keep = set(FEATS_A)
    na, ia, (ya,) = X._prefixed_graph(ma, "A__", keep)
    nb, ib, (yb,) = X._prefixed_graph(mb, "B__", keep)
    pad = numpy_helper.from_array(np.zeros((1, no, n_a - n_b), np.float32), "B__pad_zeros")
    nb.append(helper.make_node("Concat", [yb, "B__pad_zeros"], ["B__y_pad"], axis=2, name="B__pad_concat"))
    vi = lambda n: helper.make_tensor_value_info(n, TensorProto.FLOAT, [1, no, n_a])
    then_g = helper.make_graph(na, "rute_A", [], [vi(ya)], initializer=ia)
    else_g = helper.make_graph(nb, "rute_B", [], [vi("B__y_pad")], initializer=ib + [pad])
    nodes = list(g1.node) + [
        helper.make_node("Constant", [], ["c_zero"], value=numpy_helper.from_array(np.zeros(1, np.float32))),
        helper.make_node("Constant", [], ["c_half"], value=numpy_helper.from_array(np.full(1, 0.5, np.float32))),
        helper.make_node("Constant", [], ["c_ax0"], value=numpy_helper.from_array(np.zeros(1, np.int64))),
        helper.make_node("Less", ["mode", "c_zero"], ["is_auto"]),
        helper.make_node("Greater", ["score", "thr"], ["gate_on"]),
        helper.make_node("Cast", ["gate_on"], ["gate_f"], to=TensorProto.FLOAT),
        helper.make_node("Where", ["is_auto", "gate_f", "mode"], ["sel_f"]),        # Where float (pola 3a2 yang lolos TRT)
        helper.make_node("Greater", ["sel_f", "c_half"], ["sel"]),
        helper.make_node("Squeeze", ["sel", "c_ax0"], ["cond"]),
        helper.make_node("If", ["cond"], ["y"], then_branch=then_g, else_branch=else_g, name="rute_if"),
        helper.make_node("Cast", ["sel"], ["use_a"], to=TensorProto.FLOAT),
    ]
    graph = helper.make_graph(
        nodes, "router_a3_v2",
        list(g1.input) + [helper.make_tensor_value_info("mode", TensorProto.FLOAT, [1]),
                          helper.make_tensor_value_info("thr", TensorProto.FLOAT, [1])],
        [vi("y"), helper.make_tensor_value_info("use_a", TensorProto.FLOAT, [1]),
         helper.make_tensor_value_info("score", TensorProto.FLOAT, [1])],
        initializer=list(g1.initializer))
    model = helper.make_model(graph, opset_imports=list(m1.opset_import), producer_name="tahap4_final")
    model.ir_version = m1.ir_version
    onnx.checker.check_model(model)
    onnx.save(model, str(out_path))


def export_final(model, van, p2s, gate, x0, out_dir, device, precisions=("fp16",)):
    """
    FP32 (CPU) selalu diekspor (verifikasi ONNX Runtime + backend ORT). FP16 (GPU) bila diminta: trunk, Stage 2 dan
    baseline FP16 dengan I/O FP32; Gemm gate & skor heuristik FP32 (pola tahap3a2.export_fp16 yang sudah lolos di
    TensorRT 11). Returns ({presisi: {nama: path}}, modul FP32 CPU untuk verifikasi).
    """
    out = Path(out_dir)
    sa0 = torch.tensor([SMALL_PX ** 2 * 0.25])
    m = E.cpu_copy(model)
    s1g, s1h = S1Gate(m, gate["w"], gate["b"]).eval(), S1Heur(m).eval()
    _, s2a, s2b = E.make_stages(m)
    with torch.no_grad():
        _, p3, p2, p4, p5 = s1g(x0)
        n_a, n_b = int(s2a(p3, p2, p4, p5).shape[-1]), int(s2b(p3, p4, p5).shape[-1])
    statics = {"vanilla": E._StaticOut(E.cpu_copy(van)).eval(), "p2static": E._StaticOut(E.cpu_copy(p2s)).eval()}
    P32 = {k: out / f"{k}_fp32.onnx" for k in ("s1g", "s1h", "s2a", "s2b", "vanilla", "p2static", "a3g", "a3h")}
    E.export_onnx(s1g, (x0,), P32["s1g"], ["image"], ["score", *FEATS_A])
    E.export_onnx(s1h, (x0, sa0), P32["s1h"], ["image", "small_area"], ["score", *FEATS_A])
    E.export_onnx(s2a, (p3, p2, p4, p5), P32["s2a"], list(FEATS_A), ["y_A"])
    E.export_onnx(s2b, (p3, p4, p5), P32["s2b"], list(FEATS_B), ["y_B"])
    for k, mod in statics.items():
        E.export_onnx(mod, (x0,), P32[k], ["image"], ["y"])
    build_if_v2(P32["s1g"], P32["s2a"], P32["s2b"], P32["a3g"], n_a, n_b)
    build_if_v2(P32["s1h"], P32["s2a"], P32["s2b"], P32["a3h"], n_a, n_b)
    paths = {"fp32": P32}
    mods = dict(s1g=s1g, s1h=s1h, s2a=s2a, s2b=s2b, **statics, n_a=n_a, n_b=n_b)
    if "fp16" in precisions:
        m16 = X._half_copy(model, device)
        s1g16, s1h16 = S1Gate(m16, gate["w"], gate["b"]).to(device).eval(), S1Heur(m16).to(device).eval()
        _, s2a16, s2b16 = E.make_stages(m16)
        x, sa = x0.to(device), sa0.to(device)
        with torch.no_grad():
            _, q3, q2, q4, q5 = s1g16(x); s1h16(x, sa)
            h = [t.half() for t in (q3, q2, q4, q5)]
            s2a16(*h); s2b16(h[0], h[2], h[3])                      # anchors Detect (half) dihitung di luar tracing
        P16 = {k: out / f"{k}_fp16.onnx" for k in ("s1g", "s1h", "s2a", "s2b", "vanilla", "p2static", "a3g", "a3h")}
        E.export_onnx(s1g16, (x,), P16["s1g"], ["image"], ["score", *FEATS_A])
        E.export_onnx(s1h16, (x, sa), P16["s1h"], ["image", "small_area"], ["score", *FEATS_A])
        E.export_onnx(X._IOCast(s2a16), (q3, q2, q4, q5), P16["s2a"], list(FEATS_A), ["y_A"])
        E.export_onnx(X._IOCast(s2b16), (q3, q4, q5), P16["s2b"], list(FEATS_B), ["y_B"])
        for k, src in (("vanilla", van), ("p2static", p2s)):
            mod = X._IOCast(E._StaticOut(X._half_copy(src, device))).eval()
            with torch.no_grad():
                mod(x)
            E.export_onnx(mod, (x,), P16[k], ["image"], ["y"])
        build_if_v2(P16["s1g"], P16["s2a"], P16["s2b"], P16["a3g"], n_a, n_b)
        build_if_v2(P16["s1h"], P16["s2a"], P16["s2b"], P16["a3h"], n_a, n_b)
        paths["fp16"] = P16
        del m16, s1g16, s1h16, s2a16, s2b16
    return paths, mods


def verify_final(P32, mods, loader, n=4, tol=1e-4):
    """ORT (graf FP32) vs PyTorch: skor gate, fitur, Stage 2, statis, A3 v2 untuk semua mode & arah threshold."""
    import onnxruntime as ort
    sess = {k: ort.InferenceSession(str(P32[k]), providers=["CPUExecutionProvider"])
            for k in ("s1g", "s1h", "vanilla", "p2static", "a3g", "a3h")}
    rel = lambda a, b: float(np.abs(a - b).max() / (np.abs(b).max() + 1e-12))
    worst, n_a, n_b = {}, mods["n_a"], mods["n_b"]
    up = lambda k, v: worst.__setitem__(k, max(worst.get(k, 0.0), float(v)))
    with torch.no_grad():
        for k, b in enumerate(loader):
            if k >= n:
                break
            x = b["img"].float() / 255.0
            sa = np.array([small_area_of(b["ratio_pad"][0])], np.float32)
            sg, p3, p2, p4, p5 = mods["s1g"](x)
            sh = float(mods["s1h"](x, torch.from_numpy(sa))[0][0])
            o = sess["s1g"].run(None, {"image": x.numpy()})
            up("s1g_skor", abs(float(o[0][0]) - float(sg[0])) / (1.0 + abs(float(sg[0]))))
            up("s1g_fitur", max(rel(a, t.numpy()) for a, t in zip(o[1:], (p3, p2, p4, p5))))
            oh = sess["s1h"].run(None, {"image": x.numpy(), "small_area": sa})
            up("s1h_skor (diskret, informatif)", abs(float(oh[0][0]) - sh))
            ya, yb = mods["s2a"](p3, p2, p4, p5).numpy(), mods["s2b"](p3, p4, p5).numpy()
            ybp = np.concatenate([yb, np.zeros((1, yb.shape[1], n_a - n_b), np.float32)], 2)
            for nm, extra, s_ref in (("a3g", {}, float(o[0][0])), ("a3h", {"small_area": sa}, float(oh[0][0]))):
                for mode, thr, want in ((1.0, 1e9, True), (0.0, -1e9, False), (-1.0, s_ref - 0.5, True), (-1.0, s_ref + 0.5, False)):
                    y, use_a, score = sess[nm].run(None, {"image": x.numpy(), "mode": np.array([mode], np.float32),
                                                          "thr": np.array([thr], np.float32), **extra})
                    assert bool(use_a[0] > 0.5) == want, f"{nm}: rute salah (mode {mode}, thr {thr})"
                    up(f"{nm}_y", rel(y, ya if want else ybp))
                    up(f"{nm}_skor", abs(float(score[0]) - s_ref))
            for nm in ("vanilla", "p2static"):
                up(nm, rel(sess[nm].run(None, {"image": x.numpy()})[0], mods[nm](x).numpy()))
    bad = {k: v for k, v in worst.items() if "informatif" not in k and v > tol}
    assert not bad, f"Selisih ONNX vs PyTorch > {tol}: {bad}"
    assert worst["s1h_skor (diskret, informatif)"] < 1.0, f"skor heuristik ONNX jauh dari PyTorch: {worst}"
    return worst


# =============================================================================
# JOB INFERENSI (TensorRT FP16 / ORT): keluaran per gambar test + latensi per build
# =============================================================================
class _OrtEng:
    """Antarmuka sama dengan engine TensorRT di job (buf, inp, out, bind, run) — untuk uji lokal di CPU."""
    def __init__(self, path):
        import onnxruntime as ort
        self.sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
        self.inp = [i.name for i in self.sess.get_inputs()]
        self.out = [o.name for o in self.sess.get_outputs()]
        self.buf = {i.name: torch.zeros([d if isinstance(d, int) else 1 for d in i.shape], dtype=torch.float32)
                    for i in self.sess.get_inputs()}
        for nme, r in zip(self.out, self.sess.run(None, {k: self.buf[k].numpy() for k in self.inp})):
            self.buf[nme] = torch.from_numpy(np.array(r, np.float32))

    def bind(self, nme, t):
        self.buf[nme] = t

    def run(self, s=None):
        for nme, r in zip(self.out, self.sess.run(None, {k: self.buf[k].numpy() for k in self.inp})):
            self.buf[nme].copy_(torch.from_numpy(np.asarray(r, np.float32)))


def b2_thresholds(scores, rho, window, min_hist, thr0):
    """Threshold B2 untuk tiap posisi aliran (hanya skor MASA LALU). float32 agar sama dengan input engine."""
    s = np.asarray(scores, np.float64); out = np.empty(len(s), np.float32)
    for t in range(len(s)):
        hist = s[max(0, t - window):t]
        out[t] = np.float32(np.quantile(hist, 1 - rho)) if len(hist) >= min_hist else np.float32(thr0)
    return out


class SortedWindowQuantile:
    """Kuantil jendela bergulir dengan daftar terurut (bisect): nilai IDENTIK dengan np.quantile (interpolasi linear)
    pada jendela yang sama, tanpa membuat array baru per gambar."""
    def __init__(self, window):
        import bisect
        self._b, self.window, self.h, self.s = bisect, window, collections.deque(), []

    def push(self, x):
        if len(self.h) == self.window:
            old = self.h.popleft(); del self.s[self._b.bisect_left(self.s, old)]
        self.h.append(x); self._b.insort(self.s, x)

    def quantile(self, q):
        n = len(self.s); pos = q * (n - 1); lo = int(pos); fr = pos - lo
        return self.s[lo] + (self.s[min(lo + 1, n - 1)] - self.s[lo]) * fr


def b2_host_overhead(window, rho, n=4000):
    """ms per gambar: (np.quantile atas deque = implementasi job terkunci) vs jendela terurut (nilai identik)."""
    xs = np.random.default_rng(0).standard_normal(n + window)
    h = collections.deque(xs[:window], maxlen=window)
    t0 = time.perf_counter()
    for i in range(n):
        np.quantile(np.asarray(h, np.float64), 1 - rho); h.append(xs[window + i])
    t_np = (time.perf_counter() - t0) / n * 1e3
    sw = SortedWindowQuantile(window)
    for x in xs[:window]:
        sw.push(float(x))
    h = collections.deque(xs[:window], maxlen=window)
    for i in range(200):                                       # cek kesamaan nilai
        assert abs(sw.quantile(1 - rho) - np.quantile(np.asarray(h, np.float64), 1 - rho)) < 1e-12
        sw.push(float(xs[window + i])); h.append(xs[window + i])
    t0 = time.perf_counter()
    for i in range(n):
        sw.quantile(1 - rho); sw.push(float(xs[i]))
    return t_np, (time.perf_counter() - t0) / n * 1e3


def b2_decide(scores, rho, window, min_hist, thr0):
    thr = b2_thresholds(scores, rho, window, min_hist, thr0)
    return np.asarray(scores, np.float32) > thr, thr


E2E_VARIANTS = ("Vanilla", "P2-Static", "A3 gate rute A", "A3 gate rute B",
                "Router B2 terkunci (e2e)", "Router B2 efisien (e2e)")
H6_E2E = "[4B, pasca-hasil] H6 | router (B2) − campuran acak V/P2S (latensi ujung-ke-ujung)"


def e2e_pass(ops, H, batches, rho, window, min_hist, thr0, n_warmup, n_reps, ctx, dsync):
    """[TAHAP 4B] Latensi ujung-ke-ujung (ms/gambar) pada aliran test berurutan.

    batches: list (img_uint8, small_area) dalam urutan aliran (= urutan acc_pass).
    Timer mencakup: hitung threshold B2 di host -> salin gambar & threshold ke GPU -> engine A3 -> salin y, use_a,
    skor ke host. Persiapan gambar (uint8 -> float, ke buffer pinned) di luar timer, sama dengan lat_pass Tahap 4.
    Riwayat B2 dimulai ulang di setiap pengulangan, sehingga keputusan setiap pengulangan identik dengan acc_pass.
    Kembali: (lat: varian -> list ms, use: varian -> list keputusan pengulangan pertama).
    """
    def b2_router(efisien):
        hist = SortedWindowQuantile(window) if efisien else collections.deque(maxlen=window)

        def f():
            n_h = len(hist.h) if efisien else len(hist)
            if n_h >= min_hist:
                thr = np.float32(hist.quantile(1 - rho) if efisien else np.quantile(np.asarray(hist, np.float64), 1 - rho))
            else:
                thr = np.float32(thr0)
            H["thr"]["a3g"].fill_(float(thr))
            ops[("a3g", -1.0)]()
            s_ = float(H["a3g"][2][0])
            if efisien:
                hist.push(s_)
            else:
                hist.append(s_)
            return bool(float(H["a3g"][1][0]) > 0.5)
        return f

    def fixed(op):
        def f():
            H["thr"]["a3g"].fill_(float(thr0))
            op()
            return None
        return f

    def fns_fresh():
        return {"Vanilla": ops["vanilla"], "P2-Static": ops["p2static"],
                "A3 gate rute A": fixed(ops[("a3g", 1.0)]), "A3 gate rute B": fixed(ops[("a3g", 0.0)]),
                "Router B2 terkunci (e2e)": b2_router(False), "Router B2 efisien (e2e)": b2_router(True)}

    def timed(fn, im, sa):
        H["img"].copy_(im.float() / 255.0); H["sa"].fill_(sa); dsync()       # konversi di luar timer
        w0 = time.perf_counter()
        with ctx():
            u = fn()
        return (time.perf_counter() - w0) * 1e3, u

    warm = fns_fresh()
    for im, sa in batches[:n_warmup]:
        for fn in warm.values():
            timed(fn, im, sa)
    lat = {k: [] for k in E2E_VARIANTS}
    use = {k: [] for k in E2E_VARIANTS if k.startswith("Router")}
    for rep in range(n_reps):
        fns = fns_fresh()
        order = list(fns)
        for i, (im, sa) in enumerate(batches):
            for k in (order if (rep + i) % 2 == 0 else order[::-1]):
                ms, u = timed(fns[k], im, sa)
                lat[k].append(ms)
                if rep == 0 and k in use:
                    use[k].append(u)
    return lat, use


def job_main(job_path):
    job = json.loads(Path(job_path).read_text())
    T0 = time.perf_counter()
    log = lambda *a: print(f"[job {time.perf_counter() - T0:6.0f}s]", *a, flush=True)
    E.COCO_MAX_DETS = list(job["coco_max_dets"])
    use_trt = job["backend"] == "trt"
    res = {"backend": job["backend"], "build_s": {}, "latency_builds": {}, "parity": {}, "prof_activation": {}}
    if use_trt:
        import tensorrt as trt
        logger = trt.Logger(trt.Logger.WARNING)
        dev = torch.device("cuda:0"); torch.cuda.init()
        T2T = {trt.float32: torch.float32, trt.float16: torch.float16, trt.int32: torch.int32,
               trt.int64: torch.int64, trt.bool: torch.bool}

        def build(onnx_path, engine_path):
            builder = trt.Builder(logger)
            flags = 0
            if hasattr(trt.NetworkDefinitionCreationFlag, "STRONGLY_TYPED"):
                flags |= 1 << int(trt.NetworkDefinitionCreationFlag.STRONGLY_TYPED)   # presisi = tipe di ONNX
            net = builder.create_network(flags); parser = trt.OnnxParser(net, logger)
            with open(onnx_path, "rb") as f:
                if not parser.parse(f.read()):
                    raise RuntimeError("\n".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
            cfg = builder.create_builder_config()
            cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, 1 << 30)
            if job["precision"] == "fp16" and not flags and hasattr(trt.BuilderFlag, "FP16"):
                cfg.set_flag(trt.BuilderFlag.FP16)
            t = time.perf_counter(); eb = builder.build_serialized_network(net, cfg)
            assert eb is not None, f"gagal build {onnx_path}"
            Path(engine_path).write_bytes(eb)
            return engine_path, time.perf_counter() - t

        class Eng:
            def __init__(self, path):
                self.rt = trt.Runtime(logger)
                self.engine = self.rt.deserialize_cuda_engine(Path(path).read_bytes())
                self.ctx = self.engine.create_execution_context()
                self.inp, self.out, self.buf = [], [], {}
                for i in range(self.engine.num_io_tensors):
                    nme = self.engine.get_tensor_name(i)
                    (self.inp if self.engine.get_tensor_mode(nme) == trt.TensorIOMode.INPUT else self.out).append(nme)
                for nme in self.inp + self.out:
                    shape = tuple(self.engine.get_tensor_shape(nme))
                    if nme in self.inp:
                        self.ctx.set_input_shape(nme, shape)
                    self.buf[nme] = torch.zeros(shape, dtype=T2T[self.engine.get_tensor_dtype(nme)], device=dev)
                    self.ctx.set_tensor_address(nme, self.buf[nme].data_ptr())

            def bind(self, nme, t):
                self.buf[nme] = t; self.ctx.set_tensor_address(nme, t.data_ptr())

            def run(self, s):
                assert self.ctx.execute_async_v3(s.cuda_stream), "execute_async_v3 gagal"

        s = torch.cuda.Stream()
        ctx = lambda: torch.cuda.stream(s)
        sync, dsync = s.synchronize, torch.cuda.synchronize
        pin = lambda t: torch.zeros(t.shape, dtype=t.dtype).pin_memory()
        res.update(trt_version=trt.__version__, gpu=torch.cuda.get_device_name(0))
    else:
        dev = torch.device("cpu"); s = None
        build = lambda o, e: (o, 0.0)
        Eng = _OrtEng
        ctx = contextlib.nullcontext
        sync = dsync = (lambda: None)
        pin = lambda t: torch.zeros(t.shape, dtype=t.dtype)
        res.update(trt_version=None, gpu="cpu (ONNX Runtime, uji lokal)")

    from ultralytics.data.utils import check_det_dataset
    DATA = check_det_dataset(job["data_yaml"])
    loader = E.make_loader(DATA, job["split"], 1, job["imgsz"], workers=job.get("workers", 2), pin_memory=False)
    rho, W, MINH = job["rho"], job["b2_window"], job["b2_min_hist"]
    thr_g0, thr_h0 = np.float32(job["thr_val_gate"]), np.float32(job["thr_val_heur"])
    names = list(job["onnx"])
    Path(job["engine_dir"]).mkdir(parents=True, exist_ok=True)

    def load_engines(bi):
        eng = {}
        for k in names:
            p = job["onnx"][k]
            try:
                path, secs = build(p, str(Path(job["engine_dir"]) / f"{Path(p).stem}_b{bi}.engine"))
                e_ = Eng(path)
            except Exception as ex:                       # heuristik bersifat pembanding: gagal build tidak menghentikan job
                if k != "a3h":
                    raise
                res.setdefault("build_error", {})[f"b{bi}|{k}"] = f"{type(ex).__name__}: {str(ex)[:500]}"
                log("A3 heuristik GAGAL dibangun:", res["build_error"][f"b{bi}|{k}"]); continue
            res["build_s"][f"b{bi}|{k}"] = secs; eng[k] = e_; log("engine", f"build {bi}", k, f"{secs:.0f} s")
        return eng

    def make_ops(eng):
        H = {"img": pin(eng["vanilla"].buf["image"]), "sa": pin(torch.zeros(1)),
             "thr": {k: pin(torch.zeros(1)) for k in ("a3g", "a3h")}}
        modes = {mv: torch.full((1,), mv, dtype=torch.float32, device=dev) for mv in (-1.0, 0.0, 1.0)}
        ops = {}
        for k in ("vanilla", "p2static"):
            e = eng[k]; ho = pin(e.buf[e.out[0]]); H[k] = ho

            def f(e=e, ho=ho):
                e.buf["image"].copy_(H["img"], non_blocking=True); e.run(s)
                ho.copy_(e.buf[e.out[0]], non_blocking=True); sync()
            ops[k] = f
        for k in [k_ for k_ in ("a3g", "a3h") if k_ in eng]:
            e = eng[k]; hy, hu, hs = pin(e.buf["y"]), pin(e.buf["use_a"]), pin(e.buf["score"]); H[k] = (hy, hu, hs)
            has_sa = "small_area" in e.inp
            for mv in (-1.0, 0.0, 1.0):
                def f(e=e, mv=mv, hy=hy, hu=hu, hs=hs, has_sa=has_sa, ht=H["thr"][k]):
                    e.bind("mode", modes[mv])
                    e.buf["image"].copy_(H["img"], non_blocking=True); e.buf["thr"].copy_(ht, non_blocking=True)
                    if has_sa:
                        e.buf["small_area"].copy_(H["sa"], non_blocking=True)
                    e.run(s)
                    hy.copy_(e.buf["y"], non_blocking=True); hu.copy_(e.buf["use_a"], non_blocking=True)
                    hs.copy_(e.buf["score"], non_blocking=True); sync()
                ops[(k, mv)] = f
        d_img = torch.zeros_like(eng["vanilla"].buf["image"])

        def fh():
            d_img.copy_(H["img"], non_blocking=True); sync()
        ops["h2d"] = fh                                              # untuk kaskade: H2D gambar tidak dihitung dua kali
        e = eng["s1g"]; hs1 = pin(e.buf["score"]); H["s1g"] = hs1

        def f1(e=e):
            e.buf["image"].copy_(H["img"], non_blocking=True); e.run(s); hs1.copy_(e.buf["score"], non_blocking=True); sync()
        ops["s1g"] = f1
        return ops, H

    def call(op):
        dsync()
        with ctx():
            op()

    nms = lambda y, rp, osh: P._to_dets(y, rp, osh, job["conf"], job["iou"], job["max_det"])

    def acc_pass(eng, ops, H):
        D = {k: [] for k in ("V", "P2S", "A", "B")}
        rec = collections.defaultdict(list); hist = collections.deque(maxlen=W)
        par = {"y_router_vs_paksa_maks": 0.0, "skor_antar_mode_maks": 0.0, "use_a_vs_host_beda": 0}
        for i, b in enumerate(loader):
            H["img"].copy_(b["img"].float() / 255.0)
            rp, osh = b["ratio_pad"][0], b["ori_shape"][0]
            H["sa"].fill_(small_area_of(rp))
            call(ops["vanilla"]); D["V"].append(nms(eng["vanilla"].buf[eng["vanilla"].out[0]], rp, osh))
            call(ops["p2static"]); D["P2S"].append(nms(eng["p2static"].buf[eng["p2static"].out[0]], rp, osh))
            H["thr"]["a3g"].fill_(float(thr_g0))
            call(ops[("a3g", 1.0)]); yA = eng["a3g"].buf["y"].clone(); sA = float(H["a3g"][2][0])
            call(ops[("a3g", 0.0)]); yB = eng["a3g"].buf["y"].clone(); sB = float(H["a3g"][2][0])
            D["A"].append(nms(yA, rp, osh)); D["B"].append(nms(yB, rp, osh))
            thr = np.float32(np.quantile(np.asarray(hist, np.float64), 1 - rho)) if len(hist) >= MINH else thr_g0
            H["thr"]["a3g"].fill_(float(thr))
            call(ops[("a3g", -1.0)])                                   # sistem yang di-deploy: B2 + A3 (thr = input)
            s_auto, use = float(H["a3g"][2][0]), bool(float(H["a3g"][1][0]) > 0.5)
            par["y_router_vs_paksa_maks"] = max(par["y_router_vs_paksa_maks"], float((eng["a3g"].buf["y"] - (yA if use else yB)).abs().max()))
            par["skor_antar_mode_maks"] = max(par["skor_antar_mode_maks"], abs(s_auto - sA), abs(s_auto - sB))
            par["use_a_vs_host_beda"] += int(use != bool(np.float32(s_auto) > thr))
            hist.append(s_auto)
            if "a3h" in eng:
                H["thr"]["a3h"].fill_(float(thr_h0))
                call(ops[("a3h", 0.0)])                                # skor heuristik (rute B: murah); keputusan B2 offline
                rec["score_h"].append(float(H["a3h"][2][0]))
            rec["file"].append(Path(b["im_file"][0]).name); rec["score_g"].append(s_auto); rec["thr_g"].append(float(thr))
            rec["use_g"].append(use)
            if (i + 1) % 250 == 0:
                log(f"akurasi: {i + 1} gambar | aktivasi sejauh ini {np.mean(rec['use_g']):.3f}")
        par["n"] = len(rec["file"])
        return D, dict(rec), par

    def lat_pass(ops, H, imgs, bi):
        H["thr"]["a3g"].fill_(float(thr_g0)); H["thr"]["a3h"].fill_(float(thr_h0))
        V = {"Vanilla": ops["vanilla"], "P2-Static": ops["p2static"], "Stage 1 + gate (s1g)": ops["s1g"], "H2D gambar": ops["h2d"],
             "A3 gate rute A": ops[("a3g", 1.0)], "A3 gate rute B": ops[("a3g", 0.0)],
             "A3 gate router (thr val)": ops[("a3g", -1.0)]}
        if ("a3h", 1.0) in ops:
            V.update({"A3 heuristik rute A": ops[("a3h", 1.0)], "A3 heuristik rute B": ops[("a3h", 0.0)],
                      "A3 heuristik router (thr val)": ops[("a3h", -1.0)]})

        def run_fn(fn, im, sa):
            H["img"].copy_(im.float() / 255.0); H["sa"].fill_(sa); dsync()          # konversi di luar timer
            w0 = time.perf_counter()
            with ctx():
                fn()
            return (time.perf_counter() - w0) * 1e3
        for im, sa in imgs[:job["n_warmup"]]:
            for fn in V.values():
                run_fn(fn, im, sa)
        acts = {k: [] for k in ("A3 gate router (thr val)", "A3 heuristik router (thr val)") if k in V}
        rec = {k: [] for k in V}; order = list(V)
        for rep in range(job["n_reps"]):
            for im, sa in imgs:
                for k in (order if rep % 2 == 0 else order[::-1]):
                    rec[k].append(run_fn(V[k], im, sa))
                    if k in acts and rep == 0:
                        acts[k].append(float(H["a3g" if "gate" in k else "a3h"][1][0]) > 0.5)
        for k, v in rec.items():
            a = np.asarray(v)
            res["latency_builds"][f"b{bi}|{k}"] = dict(mean=a.mean(), std=a.std(), p50=np.percentile(a, 50),
                                                         p95=np.percentile(a, 95), n=int(a.size))
        res["prof_activation"][f"b{bi}"] = {k: float(np.mean(v)) for k, v in acts.items()}

    imgs = []
    for k, b in enumerate(loader):
        if k >= job["n_prof_images"]:
            break
        imgs.append((b["img"].clone(), small_area_of(b["ratio_pad"][0])))            # uint8 (hemat RAM); float di luar timer
    n_builds = int(job["n_builds"]) if use_trt else 1
    for bi in range(n_builds):
        eng = load_engines(bi); ops, H = make_ops(eng)
        if bi == 0:
            D, rec, par = acc_pass(eng, ops, H)
            res["parity"] = par; res["files"] = rec["file"]
            with open(job["dets_out"], "wb") as f:
                pickle.dump({"dets": D, "rec": rec}, f)
            Path(job["out_json"]).write_text(json.dumps(res, default=float, indent=1))
            log("akurasi selesai | paritas", par)
        lat_pass(ops, H, imgs, bi)
        Path(job["out_json"]).write_text(json.dumps(res, default=float, indent=1))
        log(f"latensi build {bi} selesai")
        stream = [(b["img"].clone(), small_area_of(b["ratio_pad"][0])) for b in loader]   # [4B] seluruh test, urutan aliran
        L_e, U_e = e2e_pass(ops, H, stream, rho, W, MINH, thr_g0, job["n_warmup"], job["n_reps"], ctx, dsync)
        del stream
        for k, v in L_e.items():
            a = np.asarray(v)
            res.setdefault("latency_e2e_builds", {})[f"b{bi}|{k}"] = dict(mean=a.mean(), std=a.std(), p50=np.percentile(a, 50),
                                                                         p95=np.percentile(a, 95), n=int(a.size))
        res.setdefault("e2e_activation", {}).update({f"b{bi}|{k}": float(np.mean(v)) for k, v in U_e.items()})
        if bi == 0:
            ref = list(map(bool, rec["use_g"]))
            res["e2e_parity"] = {k: dict(n=len(v), beda_vs_akurasi=int(sum(a_ != b_ for a_, b_ in zip(v, ref))))
                                 for k, v in U_e.items()}
        Path(job["out_json"]).write_text(json.dumps(res, default=float, indent=1))
        log(f"[4B] latensi ujung-ke-ujung build {bi} selesai | paritas {res.get('e2e_parity')}")
        del eng, ops, H; gc.collect()
        if use_trt:
            torch.cuda.synchronize(); torch.cuda.empty_cache()
    res["b2_host_ms"], res["b2_host_ms_efisien"] = b2_host_overhead(W, rho)
    res["engine_bytes"] = {Path(f).name: Path(f).stat().st_size for f in sorted(Path(job["engine_dir"]).glob("*_b0.engine"))}
    Path(job["out_json"]).write_text(json.dumps(res, default=float, indent=1))
    log("selesai")


def run_subprocess(cmd, cwd, timeout_s, tail_path, log=print):
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    killed = []
    timer = threading.Timer(timeout_s, lambda: (killed.append(1), proc.kill())); timer.start()
    tail = []
    for line in proc.stdout:
        print(line, end="", flush=True); tail = (tail + [line])[-80:]
    rc = proc.wait(); timer.cancel()
    Path(tail_path).write_text("".join(tail))
    assert not killed, f"subprocess TIMEOUT {timeout_s} s; lihat {tail_path}"
    assert rc == 0, f"subprocess gagal (returncode {rc}); lihat {tail_path}"


# =============================================================================
# BOOTSTRAP BERPASANGAN (indeks resample sama untuk semua routing; paralel lewat fork)
# =============================================================================
class UnionBoot:
    """
    Bootstrap cepat untuk routing per gambar antara dua sumber deteksi (A/B) — pengganti CocoMix.metrics di bootstrap.
    Deteksi kedua sumber diurutkan SEKALI per (kategori, area) dengan kunci (−skor, gambar, sumber, posisi) = urutan
    COCOeval.accumulate untuk routing apa pun. Untuk resample dengan multiplisitas c_i, setiap deteksi aktif menjadi satu
    titik berbobot c_i: presisi terinterpolasi di akhir kelompok salinan identik sama dengan ekspansi penuh (TP: presisi
    naik di dalam kelompok; FP: tidak pernah menjadi maksimum), sehingga argsort/concatenate per resample tidak perlu.
    Hasil pada himpunan penuh == COCOeval persis; pada resample berulang hanya urutan seri skor antar-gambar yang bisa
    berbeda dari CocoBootstrap (urutan indeks resample), efeknya ~1e-6.
    """
    AREAS = ("all", "small")

    def __init__(self, bA, bB):
        self.I, self.T, self.R, self.K, self.iou50 = bA.I, bA.T, np.asarray(bA.R), bA.K, bA.iou50
        assert bB.I == self.I and bB.K == self.K
        self.cells = {}
        for k in range(self.K):
            for area in self.AREAS:
                sc, tp, fp, img, src, pos = [], [], [], [], [], []
                npig = np.zeros(self.I, np.int64)
                for s_i, b in enumerate((bA, bB)):
                    for i, c in enumerate(b.cells[(k, area)]):
                        if c is None:
                            continue
                        s_, dtm, dtig, n_ = c
                        npig[i] = n_                                   # GT sama untuk kedua sumber
                        if len(s_):
                            sc.append(s_); tp.append(dtm & ~dtig); fp.append(~dtm & ~dtig)
                            img.append(np.full(len(s_), i, np.int64)); src.append(np.full(len(s_), s_i, np.int8))
                            pos.append(np.arange(len(s_)))
                if sc:
                    sc, img, src, pos = map(np.concatenate, (sc, img, src, pos))
                    tp, fp = np.concatenate(tp, 1), np.concatenate(fp, 1)
                    o = np.lexsort((pos, src, img, -sc))
                    o = o[(tp[:, o] | fp[:, o]).any(0)]                # diabaikan di semua IoU -> tanpa pengaruh (eksak)
                    self.cells[(k, area)] = (tp[:, o], fp[:, o], img[o], src[o], npig)
                else:
                    self.cells[(k, area)] = (np.zeros((self.T, 0), bool), np.zeros((self.T, 0), bool),
                                             np.zeros(0, np.int64), np.zeros(0, np.int8), npig)

    def metrics(self, use_a_sorted, idx):
        c = np.bincount(idx, minlength=self.I)
        P_ = {"all": [], "small": []}
        for (k, area), (tp, fp, img, src, npig) in self.cells.items():
            npg = int((c * npig).sum())
            if npg == 0:
                continue
            q = np.zeros((self.T, len(self.R)))
            if len(img):
                w = c[img] * ((src == 0) == use_a_sorted[img])
                sel = np.flatnonzero(w)
                if len(sel):
                    ww = w[sel].astype(np.int32)
                    tpc = np.cumsum(tp[:, sel] * ww, axis=1, dtype=np.int32).astype(float)   # bilangan bulat: eksak
                    fpc = np.cumsum(fp[:, sel] * ww, axis=1, dtype=np.int32).astype(float)
                    rc, pr = tpc / npg, tpc / (fpc + tpc + np.spacing(1))
                    pr = np.maximum.accumulate(pr[:, ::-1], axis=1)[:, ::-1]
                    nd = len(sel)
                    for t in range(self.T):
                        ii = np.searchsorted(rc[t], self.R, side="left"); ok = ii < nd
                        q[t, ok] = pr[t, ii[ok]]
            P_[area].append(q)
        m = {}
        if P_["all"]:
            Pa = np.stack(P_["all"]); m["mAP50-95"] = float(Pa.mean()); m["mAP50"] = float(Pa[:, self.iou50].mean())
        else:
            m["mAP50-95"] = m["mAP50"] = np.nan
        m["mAP_small"] = float(np.stack(P_["small"]).mean()) if P_["small"] else np.nan
        return m


_BCTX = {}


def _boot_rows(rows):
    mixes, idx = _BCTX["mixes"], _BCTX["idx"]
    return [{k: U.metrics(u, idx[r]) for k, (U, u) in mixes.items()} for r in rows]


def boot_engine(mixes, n_boot, seed, n_jobs=1):
    """
    mixes: {nama: (UnionBoot, use_a dalam urutan imgIds COCO)}.
    Returns (titik {nama: metrik} pada himpunan penuh, sampel {nama: {metrik: array n_boot}}).
    """
    U0 = next(iter(mixes.values()))[0]
    I = U0.I
    assert all(U.I == I for U, _ in mixes.values())
    idx = np.random.default_rng(seed).integers(0, I, (n_boot, I))
    t1 = time.perf_counter()
    point = {k: U.metrics(u, np.arange(I)) for k, (U, u) in mixes.items()}
    t1 = time.perf_counter() - t1                                     # ≈ waktu satu resample untuk semua routing
    _BCTX.update(mixes=mixes, idx=idx)
    chunks = [c for c in np.array_split(np.arange(n_boot), max(1, 4 * n_jobs)) if len(c)]
    parts = None
    try:
        if n_jobs > 1:
            timeout = max(300.0, 4.0 * t1 * n_boot / n_jobs)            # jaga-jaga deadlock fork -> fallback serial
            pool = mp.get_context("fork").Pool(n_jobs)
            try:
                parts = pool.map_async(_boot_rows, chunks).get(timeout=timeout)
                pool.close()
            except mp.TimeoutError:
                print(f"⚠ bootstrap paralel melewati {timeout:.0f} s -> dihentikan, diulang serial", flush=True)
            finally:
                pool.terminate(); pool.join()
        if parts is None:
            parts = [_boot_rows(c) for c in chunks]
    finally:
        _BCTX.clear()
    flat = [d for p_ in parts for d in p_]
    samples = {k: {m: np.array([d[k][m] for d in flat], float) for m in METRICS} for k in mixes}
    return point, samples


def _pval(s):
    return float(min(1.0, max(2 * min((s <= 0).mean(), (s >= 0).mean()), 1.0 / len(s))))


def boot_tests(point, samples, tests):
    """tests: [(label, nama_a, [nama_b...])] -> DataFrame selisih a − rata-rata(b) + CI persentil 95%."""
    rows = []
    for label, a, bs in tests:
        if a not in point or not bs or any(b not in point for b in bs):
            continue
        for m in METRICS:
            pt = point[a][m] - np.nanmean([point[b][m] for b in bs])
            sm = samples[a][m] - np.nanmean(np.stack([samples[b][m] for b in bs]), 0)
            sm = sm[np.isfinite(sm)]
            lo, hi = np.percentile(sm, [2.5, 97.5])
            rows.append(dict(uji=label, metric=m, diff=pt, ci_lo=lo, ci_hi=hi, p_two_sided=_pval(sm), n_boot=len(sm)))
    return pd.DataFrame(rows)


def partial_spearman_p(x, y, controls):
    """ρ Spearman parsial (rank kontrol diregresikan keluar dari rank x dan y) + p dua sisi (uji t, df = n − 2 − k)."""
    from scipy import stats
    x, y = np.asarray(x, float), np.asarray(y, float)
    R = lambda v: pd.Series(np.asarray(v, float)).rank().values
    Xc = np.c_[np.ones(len(x)), *[R(c) for c in controls]]
    res = lambda v: R(v) - Xc @ np.linalg.lstsq(Xc, R(v), rcond=None)[0]
    rx, ry = res(x), res(y)
    if rx.std() == 0 or ry.std() == 0:
        return float("nan"), float("nan"), len(x)
    r = float(np.corrcoef(rx, ry)[0, 1]); df = len(x) - 2 - len(controls)
    t = r * math.sqrt(df / max(1e-12, 1 - r * r))
    return r, float(2 * stats.t.sf(abs(t), df)), len(x)


# =============================================================================
# TAHAP A (PyTorch): cache, fit gate (train), threshold (val), cache test, ekspor + verifikasi ONNX
# =============================================================================
def find_ckpt(archive, ckpt_name, log=print):
    arch = Path(archive)
    if not arch.exists():                                   # nama pemilik dataset bisa beda (agungmaugi / arungmaugi)
        alt = sorted(glob.glob(f"/kaggle/input/datasets/*/{arch.name}")) + sorted(glob.glob(f"/kaggle/input/{arch.name}"))
        assert len(alt) == 1, f"arsip {arch} tidak ada; kandidat dengan nama sama: {alt}"
        log(f"arsip dipindah ke {alt[0]}"); arch = Path(alt[0])
    hits = sorted(glob.glob(str(arch / "**" / "weights" / ckpt_name), recursive=True))
    assert len(hits) == 1, f"{ckpt_name} ditemukan {len(hits)}x di {arch}: {hits}"
    return hits[0]


def check_static(path, nc, log=print):
    p = Path(path)
    assert p.exists(), (f"checkpoint baseline tidak ada: {p}. Kandidat best.pt di /kaggle/input: "
                        f"{sorted(glob.glob('/kaggle/input/**/best.pt', recursive=True))[:20]}")
    return str(p)


def stage_a(spec, cfg, out, device, log=print):
    from ultralytics.data.utils import check_det_dataset
    t0 = time.perf_counter()
    E.COCO_MAX_DETS = list(spec["coco_max_dets"])
    DATA = check_det_dataset(spec["data_yaml"]); NAMES = DATA["names"]
    L = {s: E.make_loader(DATA, s, 1, cfg["imgsz"]) for s in ("train", "val", "test")}
    N = {s: len(ld.dataset) for s, ld in L.items()}
    if cfg.get("n_train"):
        N["train"] = min(N["train"], cfg["n_train"])
    ckpt = find_ckpt(spec["archive"], cfg["ckpt_name"], log)
    model, minfo = E.load_dualbranch(ckpt, cfg["model_yaml"], DATA["nc"], NAMES, device)
    assert minfo["train_args_seed"] in (None, spec["seed"]), f"seed checkpoint {minfo['train_args_seed']} != {spec['seed']}"
    E.check_stage_equivalence(model, E.make_stages(model), L["test"], device)
    van = E.load_static(check_static(spec["vanilla_ckpt"], DATA["nc"]), device)
    p2s = E.load_static(check_static(spec["p2static_ckpt"], DATA["nc"]), device)
    for nm, m_ in (("Vanilla", van), ("P2-Static", p2s)):
        assert int(m_.model[-1].nc) == int(DATA["nc"]), f"{nm}: nc {m_.model[-1].nc} != dataset {DATA['nc']} (baseline dataset lain?)"
    statics = (van, p2s)
    kw = dict(conf=cfg["conf"], iou=cfg["iou"], max_det=spec["max_det"], log=log)
    log(f"train: cache M0{' + Vanilla + P2-Static' if cfg['cascade'] else ''} ({N['train']} gambar) ...")
    Ctr = cache_split(model, L["train"], device, dets=True, heur=False, statics=statics if cfg["cascade"] else None,
                      max_images=N["train"], **kw)
    gt_tr = P.subset_coco_gt(L["train"].dataset, NAMES, out / "gt_train.json", N["train"])
    R_tr, _, _ = router_of(gt_tr, Ctr["detA"], Ctr["detB"], Ctr["image_id"])
    TG_tr = R_tr.targets(); del R_tr; gc.collect()
    gate = fit_gate(Ctr["feat"], TG_tr["soft_small"], cfg["gate_C"])
    label_cost = dict(n_gambar_train=int(N["train"]), n_kotak_gt_train=int(TG_tr["n_gt_all"].sum()),
                      n_kotak_gt_kecil_train=int(TG_tr["n_gt_small"].sum()),
                      frac_gambar_berobjek_kecil=float((TG_tr["n_gt_small"] > 0).mean()))
    log(f"gate: {gate['info']} | biaya label: {label_cost}")
    casc = None
    if cfg["cascade"]:
        R_c, _, _ = router_of(gt_tr, Ctr["detP"], Ctr["detV"], Ctr["image_id"])
        TG_c = R_c.targets(); del R_c; gc.collect()
        casc = fit_gate(Ctr["featV"], TG_c["soft_small"], cfg["gate_C"])
        log(f"gate kaskade (fitur trunk Vanilla, target Δ P2-Static − Vanilla): {casc['info']}")
    del Ctr, gt_tr; gc.collect()

    log(f"val: skor gate & heuristik ({N['val']} gambar, tanpa NMS) ...")
    Cva = cache_split(model, L["val"], device, dets=False, heur=True, statics=None, **kw)
    sg_va = Cva["feat"] @ gate["w"] + gate["b"]
    thr_g = float(np.quantile(sg_va, 1 - cfg["rho"])); thr_h = float(np.quantile(Cva["heur"], 1 - cfg["rho"]))
    log(f"thr val: gate {thr_g:+.4f} | heuristik {thr_h:.4f} | nilai heuristik unik di val {len(np.unique(Cva['heur']))}/{N['val']}")

    log(f"test: cache M0 + Vanilla + P2-Static ({N['test']} gambar) ...")
    Cte = cache_split(model, L["test"], device, dets=True, heur=True, statics=statics, **kw)
    gt_te_path = out / "gt_test.json"; E.build_coco_gt(L["test"].dataset, NAMES, gt_te_path)
    test = {k: Cte[k] for k in ("file", "image_id", "detA", "detB", "detV", "detP", "heur", "heur_n_all", "heur_n_small", "small_area")}
    test["score_g32"] = Cte["feat"] @ gate["w"] + gate["b"]
    test["score_c32"] = Cte["featV"] @ casc["w"] + casc["b"] if casc else None

    log("GFLOPs + ekspor ONNX (FP32 untuk verifikasi, FP16 untuk TensorRT) ...")
    x0 = next(iter(L["test"]))["img"].float() / 255.0
    mc = E.cpu_copy(model); trunk = G.Stage1PostHoc(mc, nn.Identity()).eval(); _, s2a, s2b = E.make_stages(mc)
    with torch.no_grad():
        p3, p2, p4, p5 = trunk.trunk(x0)
    dens = P3Density(list(mc.model)[33]).eval(); sa0 = torch.tensor([SMALL_PX ** 2 * 0.25])
    gw_, gb_ = torch.as_tensor(gate["w"], dtype=torch.float32), float(gate["b"])
    fl = {"trunk 0–15": A3.gflops(trunk.trunk, (x0,)),
          "gate post-hoc (pooling + Gemm)": A3.gflops(lambda *a: pool4(*a) @ gw_ + gb_, (p3, p2, p4, p5)),
          "heuristik P3 (head P3 Detect_B + hitung)": A3.gflops(dens, (p3, sa0)),
          "Stage 2A": A3.gflops(s2a, (p3, p2, p4, p5)), "Stage 2B": A3.gflops(s2b, (p3, p4, p5)),
          "Vanilla": A3.gflops(E._StaticOut(E.cpu_copy(van)).eval(), (x0,)),
          "P2-Static": A3.gflops(E._StaticOut(E.cpu_copy(p2s)).eval(), (x0,))}
    log("GFLOPs:", {k: round(v, 3) for k, v in fl.items()})
    del mc, trunk, s2a, s2b, dens
    precs = ("fp16",) if (cfg["precision"] == "fp16" and (torch.cuda.is_available() or cfg.get("allow_cpu_fp16"))) else ()
    paths, mods = export_final(model, van, p2s, gate, x0, out, device, precs)
    ver = verify_final(paths["fp32"], mods, L["test"], n=cfg["n_onnx_check"])
    log("verifikasi ONNX Runtime (selisih maks):", {k: f"{v:.1e}" for k, v in ver.items()})
    A = dict(spec=spec, minfo=minfo, gate=gate, casc=casc, thr_val_gate=thr_g, thr_val_heur=thr_h,
             val=dict(score_g32=sg_va, score_h32=Cva["heur"], file=Cva["file"]), test=test, label_cost=label_cost,
             gflops=fl, onnx={p: {k: str(v) for k, v in d.items()} for p, d in paths.items()}, onnx_verif=ver,
             n_a=mods["n_a"], n_b=mods["n_b"], gt_test=str(gt_te_path), stage_a_min=(time.perf_counter() - t0) / 60)
    del model, van, p2s, mods, Cva, Cte; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return A


# =============================================================================
# TAHAP C: analisis (semua dari keluaran job + cache test)
# =============================================================================
def latency_summary(res):
    rows = []
    for k, v in res["latency_builds"].items():
        b, name = k.split("|", 1)
        rows.append(dict(build=b, varian=name, mean=v["mean"], p50=v["p50"], p95=v["p95"], std=v["std"]))
    L = pd.DataFrame(rows)
    T = L.groupby("varian")["mean"].agg(["mean", "std", "min", "max", "count"]).rename(columns={"count": "n_build"})
    T["p50 (rata-rata build)"] = L.groupby("varian")["p50"].mean()
    T["p95 (rata-rata build)"] = L.groupby("varian")["p95"].mean()
    return L, T


def analyze(A, res, J, cfg, out, log=print):
    from pycocotools.coco import COCO
    spec, rho, n_draws, seed = A["spec"], cfg["rho"], cfg["n_draws"], int(spec_seed(A))
    E.COCO_MAX_DETS = list(spec["coco_max_dets"])
    te, rec, D = A["test"], J["rec"], J["dets"]
    assert rec["file"] == te["file"], "urutan gambar job != cache test"
    ids = te["image_id"]; n = len(ids)
    gt_path = A["gt_test"] if Path(A["gt_test"]).exists() else Path(out) / "gt_test.json"   # sesi lain: dari reuse_dir
    with contextlib.redirect_stdout(io.StringIO()):
        gt = COCO(str(gt_path))

    # ---- COCO per keluaran (FP16 deployment) + router CocoMix
    R_AB, st_A, st_B = router_of(gt, D["A"], D["B"], ids)
    R_VP, st_P, st_V = router_of(gt, D["P2S"], D["V"], ids)
    TG = R_AB.targets()

    # ---- latensi
    LB, LT = latency_summary(res)
    T = LT["mean"].to_dict(); b2ms = float(res.get("b2_host_ms", 0.0))
    TA, TB, TV, TP = T["A3 gate rute A"], T["A3 gate rute B"], T["Vanilla"], T["P2-Static"]
    THA, THB = T.get("A3 heuristik rute A", np.nan), T.get("A3 heuristik rute B", np.nan)
    TS1, TH2D = T["Stage 1 + gate (s1g)"], T.get("H2D gambar", 0.0)

    # ---- keputusan
    heur_src = "TensorRT" if rec.get("score_h") else "PyTorch FP32 (engine A3 heuristik gagal dibangun)"
    sg = np.asarray(rec["score_g"], float)
    sh = np.asarray(rec["score_h"], float) if rec.get("score_h") else np.asarray(te["heur"], float)
    d_dep = np.asarray(rec["use_g"], bool)
    d_off, _ = b2_decide(sg, rho, cfg["b2_window"], cfg["b2_min_hist"], A["thr_val_gate"])
    d_heur, _ = b2_decide(sh, rho, cfg["b2_window"], cfg["b2_min_hist"], A["thr_val_heur"])
    d_top_g, d_top_h = P.top_rho(sg, rho), P.top_rho(sh, rho)
    act, act_h = float(d_dep.mean()), float(d_heur.mean())
    T_router = act * TA + (1 - act) * TB + b2ms
    T_heur = act_h * THA + (1 - act_h) * THB + b2ms
    f_mix = float(np.clip((T_router - TV) / (TP - TV), 0, 1))
    f_mix_h = float(np.clip((T_heur - TV) / (TP - TV), 0, 1)) if np.isfinite(T_heur) else np.nan
    TcA, TcB = TS1 + TP - TH2D, TV                   # kaskade: rute A = trunk Vanilla + gate + P2-Static utuh (gambar sudah di GPU)
    rho_c = float(np.clip((T_router - TcB) / (TcA - TcB), 0, 1))
    # sensitivitas (pasca-hasil s42): B2 dengan jendela terurut -> keputusan IDENTIK, hanya overhead host berbeda
    if "b2_host_ms_efisien" in res:
        b2e, b2e_src = float(res["b2_host_ms_efisien"]), "job"
    else:                                            # job lama: ukur ulang kedua implementasi di proses ini, skala ke angka job
        t_np, t_eff = b2_host_overhead(cfg["b2_window"], rho)
        b2e, b2e_src = b2ms * t_eff / t_np, f"diukur ulang di analisis (np {t_np * 1e3:.1f} µs, efisien {t_eff * 1e3:.1f} µs), diskalakan"
    T_router_e = act * TA + (1 - act) * TB + b2e
    f_mix_e = float(np.clip((T_router_e - TV) / (TP - TV), 0, 1))
    rho_c_e = float(np.clip((T_router_e - TcB) / (TcA - TcB), 0, 1))
    lat = dict(T_router=T_router, T_heur=T_heur, T_V=TV, T_P2S=TP, T_A3_A=TA, T_A3_B=TB, T_A3h_A=THA, T_A3h_B=THB,
               T_s1g=TS1, T_h2d=TH2D, b2_host_ms=b2ms, f_mix=f_mix, f_mix_heur=f_mix_h, rho_kaskade=rho_c,
               T_kaskade_A=TcA, T_kaskade_B=TcB, b2_host_ms_efisien=b2e, b2_efisien_sumber=b2e_src,
               T_router_b2_efisien=T_router_e, f_mix_b2_efisien=f_mix_e, rho_kaskade_b2_efisien=rho_c_e,
               H5_rute_A=TA - TP, H5_rute_B=TB - TV, heur_tambahan_rute_A=THA - TA, heur_tambahan_rute_B=THB - TB)

    # ---- [4B] latensi ujung-ke-ujung (pass terpisah, sesi yang sama)
    LE = None
    if res.get("latency_e2e_builds"):
        LE = pd.DataFrame([dict(build=k.split("|", 1)[0], varian=k.split("|", 1)[1], **v)
                           for k, v in res["latency_e2e_builds"].items()])
        te_ = LE.groupby("varian")["mean"].mean().to_dict()
        TrE, TVE, TPE = te_["Router B2 terkunci (e2e)"], te_["Vanilla"], te_["P2-Static"]
        TAE, TBE = te_["A3 gate rute A"], te_["A3 gate rute B"]
        act_e = float(np.mean([v for k, v in res["e2e_activation"].items() if k.endswith("terkunci (e2e)")]))
        lat.update(T_router_e2e=TrE, T_router_e2e_efisien=te_["Router B2 efisien (e2e)"], T_V_e2e=TVE, T_P2S_e2e=TPE,
                   T_A_e2e=TAE, T_B_e2e=TBE, aktivasi_e2e=act_e,
                   T_router_komposit_sepass=act_e * TAE + (1 - act_e) * TBE + b2ms,
                   f_mix_e2e=float(np.clip((TrE - TVE) / (TPE - TVE), 0, 1)),
                   e2e_parity=res.get("e2e_parity"))
        lat["selisih_e2e_minus_komposit_sepass"] = TrE - lat["T_router_komposit_sepass"]

    # ---- campuran untuk bootstrap
    k_dep, k_h = int(d_dep.sum()), int(d_heur.sum())
    U_AB, U_VP = UnionBoot(R_AB.bA, R_AB.bB), UnionBoot(R_VP.bA, R_VP.bB)
    ab = lambda m_: (U_AB, R_AB.sorted_mask(m_)); vp = lambda m_: (U_VP, R_VP.sorted_mask(m_))
    mixes = {"A": ab(np.ones(n, bool)), "B": ab(np.zeros(n, bool)), "V": vp(np.zeros(n, bool)), "P2S": vp(np.ones(n, bool)),
             "router": ab(d_dep), "router_top": ab(d_top_g), "heur": ab(d_heur), "heur_top": ab(d_top_h)}
    for j, m_ in enumerate(P.random_masks(n, k_dep / n, n_draws, seed + 11)):
        mixes[f"acak_router_{j}"] = ab(m_)
    n2 = cfg.get("n_draws_sekunder", 3)                       # pembanding sekunder: undian lebih sedikit (waktu)
    for j, m_ in enumerate(P.random_masks(n, rho, n2, seed + 13)):
        mixes[f"acak_top_{j}"] = ab(m_)
    for j, m_ in enumerate(P.random_masks(n, k_h / n, n2, seed + 17)):
        mixes[f"acak_heur_{j}"] = ab(m_)
    for j, m_ in enumerate(P.random_masks(n, f_mix, n_draws, seed + 19)):
        mixes[f"campuran_router_{j}"] = vp(m_)
    if np.isfinite(f_mix_h):
        for j, m_ in enumerate(P.random_masks(n, f_mix_h, n2, seed + 23)):
            mixes[f"campuran_heur_{j}"] = vp(m_)
    for j, m_ in enumerate(P.random_masks(n, f_mix_e, n_draws, seed + 19)):
        mixes[f"campuran_b2ef_{j}"] = vp(m_)
    if "f_mix_e2e" in lat:                                   # [4B]
        for j, m_ in enumerate(P.random_masks(n, lat["f_mix_e2e"], n_draws, seed + 19)):
            mixes[f"campuran_e2e_{j}"] = vp(m_)
    if te.get("score_c32") is not None:
        mixes["kaskade"] = vp(P.top_rho(te["score_c32"], rho_c))
        mixes["kaskade_b2ef"] = vp(P.top_rho(te["score_c32"], rho_c_e))
    grp = lambda p_: [k for k in mixes if k.startswith(p_)]
    tests = [("H2/H7 | router (B2) − acak (aktivasi sama)", "router", grp("acak_router_")),
             ("router (top-ρ) − acak (ρ)", "router_top", grp("acak_top_")),
             ("H6 | router (B2) − campuran acak Vanilla/P2-Static (latensi sama)", "router", grp("campuran_router_")),
             ("router (B2) − Branch A (P2 selalu)", "router", ["A"]),
             ("router (B2) − Branch B (P2 tidak pernah)", "router", ["B"]),
             ("Branch A − P2-Static", "A", ["P2S"]), ("Branch B − Vanilla", "B", ["V"]),
             ("gate − heuristik kepadatan (top-ρ)", "router_top", ["heur_top"]),
             ("gate (B2) − heuristik kepadatan (B2)", "router", ["heur"]),
             ("heuristik (B2) − acak (aktivasi sama)", "heur", grp("acak_heur_")),
             ("heuristik (B2) − campuran Vanilla/P2-Static (latensi heuristik)", "heur", grp("campuran_heur_")),
             ("router (B2) − kaskade bergate Vanilla→P2-Static (latensi sama)", "router", ["kaskade"]),
             (H6_SENS, "router", grp("campuran_b2ef_")),
             ("[sensitivitas, pasca-hasil] router (B2 efisien) − kaskade (latensi sama)", "router", ["kaskade_b2ef"])]
    if "f_mix_e2e" in lat:                                   # [4B]
        tests.append((H6_E2E, "router", grp("campuran_e2e_")))
    tb = time.perf_counter()
    log(f"bootstrap: {len(mixes)} routing × {cfg['n_boot']} resample ({cfg['boot_jobs']} proses) ...")
    point, samples = boot_engine(mixes, cfg["n_boot"], cfg["boot_seed"], cfg["boot_jobs"])
    BS = boot_tests(point, samples, tests)
    log(f"bootstrap selesai ({time.perf_counter() - tb:.0f} s)")
    np.savez_compressed(out / "boot_samples.npz", **{f"{k}|{m}": v[m].astype(np.float32) for k, v in samples.items() for m in METRICS})
    (out / "boot_point.json").write_text(json.dumps(point, indent=1, default=float))
    (out / "boot_tests.json").write_text(json.dumps(tests, indent=1))

    # ---- sensitivitas latensi per build (titik; rata-rata undian campuran pada himpunan penuh)
    PB = []
    for b_, g_ in LB.groupby("build"):
        t_ = g_.set_index("varian")["mean"]
        for lab, b2x in (("B2 terkunci (np.quantile)", b2ms), ("B2 efisien", b2e)):
            Tr = act * t_["A3 gate rute A"] + (1 - act) * t_["A3 gate rute B"] + b2x
            f_ = float(np.clip((Tr - t_["Vanilla"]) / (t_["P2-Static"] - t_["Vanilla"]), 0, 1))
            mm = [R_VP.metrics(m_) for m_ in P.random_masks(n, f_, n_draws, seed + 19)]
            PB.append(dict(build=b_, b2=lab, T_router=Tr, f_campuran=f_,
                           **{f"router − campuran {k}": point["router"][k] - np.mean([x[k] for x in mm]) for k in METRICS}))
    PB = pd.DataFrame(PB)

    # ---- COCO lengkap (12 metrik) untuk sistem utama
    def routed(mask):
        return [a if u else b for a, b, u in zip(D["A"], D["B"], mask)]
    full = {"Vanilla": st_V, "P2-Static": st_P, "Branch A (paksa)": st_A, "Branch B (paksa)": st_B,
            "Router (B2, deploy)": coco_of(gt, routed(d_dep), ids)[0],
            "Heuristik kepadatan (B2)": coco_of(gt, routed(d_heur), ids)[0]}
    for nm, key in (("router", "Router (B2, deploy)"), ("heur", "Heuristik kepadatan (B2)"), ("A", "Branch A (paksa)"),
                    ("V", "Vanilla"), ("P2S", "P2-Static")):
        for k in METRICS:                                  # CocoMix (dasar bootstrap) == COCOeval langsung pada jdict ter-route
            a_, b_ = point[nm][k], full[key][k]
            b_ = np.nan if b_ < 0 else b_
            assert (np.isnan(a_) and np.isnan(b_)) or abs(a_ - b_) < 1e-9, f"CocoMix {nm} {k} {a_} != COCOeval {b_}"
    FULL = pd.DataFrame(full).T
    FULL["latensi_ms"] = [TV, TP, TA, TB, T_router, T_heur]
    FULL["aktivasi_P2"] = [0.0, 1.0, 1.0, 0.0, act, act_h]

    # ---- H3 & perilaku skor
    has = TG["n_gt_small"] > 0; anyg = TG["n_gt_all"] > 0
    ratio = np.where(anyg, TG["n_gt_small"] / np.maximum(TG["n_gt_all"], 1), np.nan)
    H3 = []
    for nm, sc in (("gate", sg), ("heuristik", sh)):
        r, p, nn_ = partial_spearman_p(sc[anyg], ratio[anyg], [TG["n_gt_all"][anyg]])
        H3.append(dict(skor=nm, uji="H3: ρ parsial(skor, proporsi GT kecil | jumlah GT)", rho=r, p=p, n=nn_))
        r, p, nn_ = partial_spearman_p(sc[has], TG["soft_small"][has], [TG["n_gt_small"][has], TG["n_gt_all"][has]])
        H3.append(dict(skor=nm, uji="ρ parsial(skor, Δ_small | GT kecil & semua), gambar berobjek kecil", rho=r, p=p, n=nn_))
    from scipy import stats as st_
    nsm_det = np.array([np.log1p(((d[:, 4] >= HEUR_CONF) & (d[:, 2] * d[:, 3] < SMALL_PX ** 2)).sum()) if len(d) else 0.0
                        for d in te["detB"]])
    SAN = dict(spearman_heur_trt_vs_pytorch=float(st_.spearmanr(sh, te["heur"])[0]),
               spearman_heur_vs_kotak_kecil_deteksi_D2b=float(st_.spearmanr(te["heur"], nsm_det)[0]),
               spearman_gate_vs_heur=float(st_.spearmanr(sg, sh)[0]),
               spearman_gate_trt_vs_pytorch=float(st_.spearmanr(sg, te["score_g32"])[0]),
               setuju_keputusan_router_job_vs_offline=float((d_dep == d_off).mean()), sumber_skor_heuristik=heur_src,
               nilai_heuristik_unik_test=int(len(np.unique(sh))))

    # ---- B2 pada urutan acak (ketahanan; titik saja)
    rng = np.random.default_rng(seed + 29)
    orders = {"asli (nama file)": np.arange(n), **{f"acak #{j + 1}": rng.permutation(n) for j in range(cfg["b2_orders"])}}
    BO = []
    for on, order in orders.items():
        for nm, sc, thr0 in (("gate", sg, A["thr_val_gate"]), ("heuristik", sh, A["thr_val_heur"])):
            dec = np.zeros(n, bool)
            dec[order] = b2_decide(sc[order], rho, cfg["b2_window"], cfg["b2_min_hist"], thr0)[0]
            m_ = R_AB.metrics(dec)
            base = [R_AB.metrics(r_) for r_ in P.random_masks(n, dec.mean(), n_draws, seed + 31)]
            BO.append(dict(urutan=on, skor=nm, aktivasi=float(dec.mean()), abs_dev=abs(float(dec.mean()) - rho),
                           **{f"gain_{k}": m_[k] - np.mean([b_[k] for b_ in base]) for k in METRICS}))
    BO = pd.DataFrame(BO)

    # ---- pemeriksaan silang PyTorch FP32 (titik saja)
    X32 = {}
    if cfg["fp32_check"]:
        R32, _, _ = router_of(gt, te["detA"], te["detB"], ids)
        RVP32, _, _ = router_of(gt, te["detP"], te["detV"], ids)
        d32, _ = b2_decide(te["score_g32"], rho, cfg["b2_window"], cfg["b2_min_hist"], A["thr_val_gate"])
        a32 = float(d32.mean()); f32 = float(np.clip((a32 * TA + (1 - a32) * TB + b2ms - TV) / (TP - TV), 0, 1))
        m_r = R32.metrics(d32)
        rnd = [R32.metrics(r_) for r_ in P.random_masks(n, a32, n_draws, seed + 11)]
        mix = [RVP32.metrics(r_) for r_ in P.random_masks(n, f32, n_draws, seed + 19)]
        mA, mB = R32.metrics(np.ones(n, bool)), R32.metrics(np.zeros(n, bool))
        mV, mP = RVP32.metrics(np.zeros(n, bool)), RVP32.metrics(np.ones(n, bool))
        for k in METRICS:
            X32[k] = {"router − acak": m_r[k] - np.mean([r_[k] for r_ in rnd]),
                      "router − campuran V/P2S": m_r[k] - np.mean([r_[k] for r_ in mix]),
                      "Branch A − P2-Static": mA[k] - mP[k], "Branch B − Vanilla": mB[k] - mV[k],
                      "router": m_r[k], "Vanilla": mV[k], "P2-Static": mP[k]}
        X32 = pd.DataFrame(X32)
        SAN["aktivasi_fp32"] = a32; SAN["setuju_keputusan_fp16_vs_fp32"] = float((d32 == d_dep).mean())

    # ---- kriteria per model
    bs = BS.set_index(["uji", "metric"])
    get = lambda u, m="mAP_small": bs.loc[(u, m)] if (u, m) in bs.index else None
    h2 = get(tests[0][0]); h6 = get(tests[2][0])
    stat = lambda r: "unggul (CI > 0)" if r["ci_lo"] > 0 else ("kalah (CI < 0)" if r["ci_hi"] < 0 else "tidak kalah (CI memuat 0)")
    KR = dict(
        H1_mAP50=bool(st_V["mAP50"] < full["Router (B2, deploy)"]["mAP50"] < st_P["mAP50"]),
        H1_latensi=bool(TV < T_router < TP),
        H2_router_vs_acak=bool(h2["ci_lo"] > 0),
        H5_rute_A=bool(TA - TP <= cfg["h5_ms"]), H5_rute_B=bool(TB - TV <= cfg["h5_ms"]),
        H6_status=stat(h6),
        H7_aktivasi=bool(abs(act - rho) <= cfg["h7_dev"]), H7_gain=bool(h2["ci_lo"] > 0),
        H6_sensitivitas_B2_efisien=stat(get(H6_SENS)) if get(H6_SENS) is not None else None)
    if "f_mix_e2e" in lat:                                   # [4B]
        KR.update(H1_latensi_e2e=bool(lat["T_V_e2e"] < lat["T_router_e2e"] < lat["T_P2S_e2e"]),
                  H5_rute_A_e2e=bool(lat["T_A_e2e"] - lat["T_P2S_e2e"] <= cfg["h5_ms"]),
                  H5_rute_B_e2e=bool(lat["T_B_e2e"] - lat["T_V_e2e"] <= cfg["h5_ms"]),
                  H6_e2e_status=stat(get(H6_E2E)) if get(H6_E2E) is not None else None)
    RES = dict(model=spec["name"], aktivasi_test=act, aktivasi_heur=act_h, latensi=lat, kriteria=KR, sanity=SAN,
               parity=res["parity"], label_cost=A["label_cost"], gflops=A["gflops"], gate_info=A["gate"]["info"],
               thr_val_gate=A["thr_val_gate"], thr_val_heur=A["thr_val_heur"], trt=res.get("trt_version"), gpu=res.get("gpu"),
               backend=res["backend"], n_test=n, prof_activation=res.get("prof_activation"), build_error=res.get("build_error"),
               engine_bytes=res.get("engine_bytes"),
               h3={f"{r['skor']} | {r['uji']}": dict(rho=r["rho"], p=r["p"], n=r["n"]) for r in H3})
    # ---- simpan
    BS.to_csv(out / "bootstrap_test.csv", index=False); FULL.to_csv(out / "coco_lengkap.csv")
    LT.to_csv(out / "latensi.csv"); LB.to_csv(out / "latensi_per_build.csv", index=False)
    pd.DataFrame(H3).to_csv(out / "h3_perilaku_skor.csv", index=False); BO.to_csv(out / "b2_urutan.csv", index=False)
    PB.to_csv(out / "h6_per_build.csv", index=False)
    if LE is not None:                                        # [4B]
        LE.to_csv(out / "latensi_e2e_per_build.csv", index=False)
    if len(X32):
        X32.to_csv(out / "cek_fp32.csv")
    pd.DataFrame({"file": te["file"], "skor_gate": sg, "thr_b2": rec["thr_g"], "pakai_P2": d_dep, "skor_heuristik": sh,
                  "pakai_P2_heuristik": d_heur, "skor_gate_fp32": te["score_g32"], "n_gt_kecil": TG["n_gt_small"],
                  "n_gt": TG["n_gt_all"], "delta_soft_small": TG["soft_small"]}).to_csv(out / "per_gambar_test.csv", index=False)
    (out / "hasil.json").write_text(json.dumps(RES, indent=1, default=float))
    return RES, dict(BS=BS, FULL=FULL, LT=LT, H3=pd.DataFrame(H3), BO=BO, X32=X32, PB=PB)


def spec_seed(A):
    return A["spec"]["seed"]


def print_model(RES, T, log=print):
    pd.set_option("display.width", 220)
    log("\n=== " + RES["model"] + " ===")
    log("COCO (TensorRT " + str(RES["backend"]) + "):\n" + T["FULL"][["mAP50-95", "mAP50", "mAP_small", "mAP_medium", "mAP_large",
                                                                   "AR_small", "latensi_ms", "aktivasi_P2"]].round(4).to_string())
    log("Bootstrap berpasangan (test):\n" + T["BS"][["uji", "metric", "diff", "ci_lo", "ci_hi", "p_two_sided"]].round(4).to_string(index=False))
    log("Latensi (ms; rata-rata antar-build):\n" + T["LT"].round(3).to_string())
    log("Turunan latensi:", {k: (round(v, 4) if isinstance(v, (int, float)) else v) for k, v in RES["latensi"].items()})
    log("H3 / perilaku skor:\n" + T["H3"].round(4).to_string(index=False))
    log("B2 per urutan:\n" + T["BO"].round(4).to_string(index=False))
    log("H6 per build (titik; B2 terkunci vs efisien):\n" + T["PB"].round(4).to_string(index=False))
    if isinstance(T["X32"], pd.DataFrame) and len(T["X32"]):
        log("Cek silang PyTorch FP32 (titik):\n" + T["X32"].round(4).to_string())
    log("Sanity:", RES["sanity"]); log("Paritas job:", RES["parity"]); log("Kriteria:", RES["kriteria"])


# =============================================================================
# ORKESTRASI SATU MODEL
# =============================================================================
INTERMEDIATE = ("stageA.pkl", "job_res.json", "job_dets.pkl", "gt_test.json")


def run_model(spec, cfg, out_root, repo, log=print):
    out = Path(out_root) / spec["name"]; out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    assert torch.cuda.is_available() or cfg["backend"] == "ort", "GPU tidak aktif: cek Accelerator di pengaturan notebook"
    if cfg.get("reuse_dir"):
        cand = sorted(Path(cfg["reuse_dir"]).glob(f"**/{spec['name']}/stageA.pkl"))
        src = cand[0].parent if cand else Path(cfg["reuse_dir"]) / spec["name"]
        for f in INTERMEDIATE:
            if (src / f).exists() and not (out / f).exists():
                (out / f).write_bytes((src / f).read_bytes()); log(f"pakai ulang {src / f}")
    A_path = out / "stageA.pkl"
    if A_path.exists():
        A = pickle.loads(A_path.read_bytes()); log("tahap A dimuat dari", A_path)
        if not all(Path(p).exists() for p in A["onnx"].get(cfg["precision"], {}).values()):
            assert (out / "job_res.json").exists(), "ONNX tidak ada (dipakai ulang dari sesi lain?) dan job belum jalan: hapus stageA.pkl"
    else:
        A = stage_a(spec, cfg, out, device, log)
        A_path.write_bytes(pickle.dumps(A)); log(f"tahap A selesai ({A['stage_a_min']:.1f} mnt)")
    if not ((out / "job_res.json").exists() and (out / "job_dets.pkl").exists()):
        prec = cfg["precision"]
        assert prec in A["onnx"], f"ONNX {prec} tidak tersedia (GPU tidak aktif saat ekspor?)"
        O = A["onnx"][prec]
        job = dict(backend=cfg["backend"], precision=prec, onnx={k: O[k] for k in ("vanilla", "p2static", "a3g", "a3h", "s1g")},
                   data_yaml=spec["data_yaml"], split="test", imgsz=cfg["imgsz"], conf=cfg["conf"], iou=cfg["iou"],
                   max_det=spec["max_det"], coco_max_dets=spec["coco_max_dets"], rho=cfg["rho"], b2_window=cfg["b2_window"],
                   b2_min_hist=cfg["b2_min_hist"], thr_val_gate=A["thr_val_gate"], thr_val_heur=A["thr_val_heur"],
                   n_prof_images=cfg["n_prof_images"], n_warmup=cfg["n_warmup"], n_reps=cfg["n_reps"],
                   n_builds=spec.get("n_builds", cfg["n_builds"]), engine_dir=str(out / "engines"), workers=cfg.get("job_workers", 2),
                   out_json=str(out / "job_res.json"), dets_out=str(out / "job_dets.pkl"))
        (out / "job.json").write_text(json.dumps(job, indent=1))
        log(f"job inferensi ({cfg['backend']}, {prec}; timeout {cfg['trt_timeout_s']} s) ...")
        run_subprocess([sys.executable, "-u", "-m", "tahap4b", str(out / "job.json")], cwd=str(repo),
                       timeout_s=cfg["trt_timeout_s"], tail_path=out / "job_log_tail.txt", log=log)
    res = json.loads((out / "job_res.json").read_text())
    J = pickle.loads((out / "job_dets.pkl").read_bytes())
    log("analisis + bootstrap ...")
    RES, T = analyze(A, res, J, cfg, out, log)
    print_model(RES, T, log)
    (out / "DONE").write_text(time.strftime("%Y-%m-%dT%H:%M:%S"))
    return RES, T


def zip_model(out_root, name, dest_dir, intermediate=False, log=print):
    d = Path(out_root) / name
    arc = Path(dest_dir) / f"tahap4b_{name}.zip"
    skip_suffix = (".onnx", ".engine", ".pkl")
    with zipfile.ZipFile(arc, "w", zipfile.ZIP_DEFLATED) as z:
        for f in d.rglob("*"):
            if f.is_file() and not f.name.startswith("gt_") and f.suffix not in skip_suffix:
                z.write(f, Path(name) / f.relative_to(d))
    log(f"📦 {arc} ({arc.stat().st_size / 1e6:.2f} MB)")
    if intermediate:
        arc2 = Path(dest_dir) / f"tahap4b_{name}_intermediate.zip"
        with zipfile.ZipFile(arc2, "w", zipfile.ZIP_DEFLATED) as z:
            for f in INTERMEDIATE:
                if (d / f).exists():
                    z.write(d / f, Path(name) / f)
        log(f"📦 {arc2} ({arc2.stat().st_size / 1e6:.1f} MB) — untuk menjalankan ulang analisis tanpa GPU (reuse_dir)")


# =============================================================================
# RINGKASAN GABUNGAN (semua model)
# =============================================================================
MAIN_TESTS = ("H2/H7 | router (B2) − acak (aktivasi sama)",
              "H6 | router (B2) − campuran acak Vanilla/P2-Static (latensi sama)", H6_SENS,
              "gate (B2) − heuristik kepadatan (B2)", "gate − heuristik kepadatan (top-ρ)",
              "router (B2) − kaskade bergate Vanilla→P2-Static (latensi sama)",
              "router (B2) − Branch A (P2 selalu)", "Branch A − P2-Static", "Branch B − Vanilla", H6_E2E)


def pooled_tests(dirs, tests_wanted=MAIN_TESTS):
    """Selisih rata-rata antar-model per resample (indeks resample sama -> sah bila test sama, mis. KITTI 3 seed)."""
    S, Pt, Tdef = {}, {}, None
    for d in dirs:
        z = np.load(d / "boot_samples.npz"); S[d.name] = {k: z[k] for k in z.files}
        Pt[d.name] = json.loads((d / "boot_point.json").read_text())
        Tdef = Tdef or json.loads((d / "boot_tests.json").read_text())
    rows = []
    for label, a, bs in Tdef:
        if label not in tests_wanted:
            continue
        ok = [m for m in S if all(f"{x}|mAP_small" in S[m] for x in [a, *bs])]
        if len(ok) != len(S):
            continue
        for met in METRICS:
            diffs = [S[m][f"{a}|{met}"] - np.nanmean(np.stack([S[m][f"{b}|{met}"] for b in bs]), 0) for m in ok]
            pts = [Pt[m][a][met] - np.nanmean([Pt[m][b][met] for b in bs]) for m in ok]
            sm = np.mean(np.stack(diffs), 0); sm = sm[np.isfinite(sm)]
            lo, hi = np.percentile(sm, [2.5, 97.5])
            rows.append(dict(uji=label, metric=met, diff_rata2=float(np.mean(pts)), ci_lo=lo, ci_hi=hi, p_two_sided=_pval(sm),
                             n_model=len(ok)))
    return pd.DataFrame(rows)


def merge(out_root, all_models, merge_glob=None, log=print):
    out_root = Path(out_root)
    if merge_glob:
        hits = sorted(glob.glob(merge_glob, recursive=True)); log("gabung:", hits)
        for hp in map(Path, hits):
            if hp.is_file() and hp.suffix == ".zip":
                with zipfile.ZipFile(hp) as z:
                    z.extractall(out_root)
            elif hp.is_dir():                                   # dataset Kaggle: zip sudah diekstrak otomatis
                import shutil
                for s in all_models:
                    cand = [d.parent for d in hp.glob(f"**/{s['name']}/DONE")]
                    if not cand:
                        continue
                    best = sorted(cand, key=lambda d: ((d / "h6_per_build.csv").exists(), (d / "DONE").stat().st_mtime))[-1]
                    if len(cand) > 1:
                        log(f"{s['name']}: {len(cand)} salinan, dipakai {best}")
                    shutil.copytree(best, out_root / s["name"], dirs_exist_ok=True)
    done = [s for s in all_models if (out_root / s["name"] / "DONE").exists()]
    log("model selesai:", [s["name"] for s in done], f"({len(done)}/{len(all_models)})")
    assert done, "Belum ada model yang selesai."
    R = {s["name"]: json.loads((out_root / s["name"] / "hasil.json").read_text()) for s in done}
    BS = pd.concat([pd.read_csv(out_root / s["name"] / "bootstrap_test.csv").assign(model=s["name"]) for s in done])
    FULL = pd.concat([pd.read_csv(out_root / s["name"] / "coco_lengkap.csv", index_col=0).assign(model=s["name"]) for s in done])
    rows = []
    for nm, r in R.items():
        bm = BS[(BS.model == nm) & (BS.metric == "mAP_small")].set_index("uji")
        row = dict(model=nm, aktivasi=r["aktivasi_test"], T_router=r["latensi"]["T_router"], T_V=r["latensi"]["T_V"],
                   T_P2S=r["latensi"]["T_P2S"], f_campuran=r["latensi"]["f_mix"],
                   **{k: r["latensi"].get(k) for k in ("T_router_e2e", "T_V_e2e", "T_P2S_e2e", "aktivasi_e2e",
                                                       "T_router_komposit_sepass", "selisih_e2e_minus_komposit_sepass",
                                                       "f_mix_e2e")},
                   **{k: v for k, v in r["kriteria"].items()})
        for u in MAIN_TESTS:
            if u in bm.index:
                row[f"{u.split('|')[-1].strip()} [mAP_small]"] = f"{bm.loc[u, 'diff']:+.4f} [{bm.loc[u, 'ci_lo']:+.4f}, {bm.loc[u, 'ci_hi']:+.4f}]"
        rows.append(row)
    TAB = pd.DataFrame(rows).set_index("model")
    H3 = pd.DataFrame([dict(model=nm, kunci=k, **v) for nm, r in R.items() for k, v in r["h3"].items()])
    h3m = H3.kunci == "gate | H3: ρ parsial(skor, proporsi GT kecil | jumlah GT)"
    H3.loc[h3m, "p_holm"] = E.holm(H3.loc[h3m, "p"].fillna(1.0).values)
    H3.loc[h3m, "H3_lolos"] = (H3.loc[h3m, "rho"] > 0) & (H3.loc[h3m, "p_holm"] < 0.05)
    kitti = [out_root / s["name"] for s in done if s["name"].startswith("kitti")]
    POOL = pooled_tests(kitti) if len(kitti) > 1 else pd.DataFrame()
    LAT = pd.concat([pd.read_csv(out_root / s["name"] / "latensi_per_build.csv").assign(model=s["name"]) for s in done])
    LATP = LAT.groupby("varian")["mean"].agg(["mean", "std", "count"]).rename(columns={"count": "n_build_total"})
    return dict(R=R, TAB=TAB, BS=BS, FULL=FULL, H3=H3, POOL=POOL, LATP=LATP, done=[s["name"] for s in done])


def pareto_plot(out_root, R, FULL, path):
    import matplotlib.pyplot as plt
    names = list(R)
    fig, axes = plt.subplots(1, len(names), figsize=(5.2 * len(names), 4.4), squeeze=False)
    for ax, nm in zip(axes[0], names):
        r, F = R[nm], FULL[FULL.model == nm]
        L = r["latensi"]
        bp = json.loads((Path(out_root) / nm / "boot_point.json").read_text())
        ax.plot([L["T_V"], L["T_P2S"]], [F.loc["Vanilla", "mAP_small"], F.loc["P2-Static", "mAP_small"]], "k:", label="garis Vanilla↔P2-Static")
        ax.plot(L["T_V"], F.loc["Vanilla", "mAP_small"], "ks", label="Vanilla"); ax.plot(L["T_P2S"], F.loc["P2-Static", "mAP_small"], "k^", label="P2-Static")
        cm = [v["mAP_small"] for k, v in bp.items() if k.startswith("campuran_router_")]
        if cm:
            ax.plot(L["T_router"], np.mean(cm), "o", mfc="none", ms=9, label="campuran acak V/P2S")
        if "kaskade" in bp:
            ax.plot(L["T_router"], bp["kaskade"]["mAP_small"], "D", mfc="none", ms=8, label="kaskade V→P2S")
        ax.plot(L["T_heur"], F.loc["Heuristik kepadatan (B2)", "mAP_small"], "P", ms=9, label="heuristik (B2)")
        ax.plot(L["T_router"], F.loc["Router (B2, deploy)", "mAP_small"], "*", ms=14, label="router (B2, A3 FP16)")
        ax.plot([L["T_A3_B"], L["T_A3_A"]], [F.loc["Branch B (paksa)", "mAP_small"], F.loc["Branch A (paksa)", "mAP_small"]], "x--", alpha=.5, label="rute paksa B/A")
        ax.set(title=nm, xlabel="latensi TensorRT FP16 (ms)", ylabel="mAP_small (COCO)"); ax.grid(alpha=.3)
    axes[0][0].legend(fontsize=7)
    plt.tight_layout(); fig.savefig(path, dpi=130)
    return fig


if __name__ == "__main__":
    job_main(sys.argv[1])


In [ ]:
# [9] SATU MODEL: tahap A (PyTorch: cache, fit gate di train, threshold val, ekspor + verifikasi ONNX)
#     -> job inferensi (TensorRT FP16, proses terpisah, timeout) -> analisis + bootstrap -> zip
import importlib, numpy as np, pandas as pd, torch
import router_train_utils as U, tahap2_eval as E, tahap2b_plafon as P, tahap2c_gate as G, tahap3a as A3, tahap3a2 as X, tahap4b as F
for m_ in (U, E, P, G, A3, X, F):
    importlib.reload(m_)
torch.manual_seed(0); np.random.seed(0); torch.backends.cudnn.benchmark = False
DEST = Path("/kaggle/working") if "CFG_OVERRIDE" not in globals() else OUT.parent
MODEL_RESULTS = {}
for spec in CFG["models"]:
    if (OUT / spec["name"] / "DONE").exists():
        stamp(f"{spec['name']}: sudah selesai, dilewati"); continue
    stamp(f"mulai {spec['name']} | device {'cuda' if torch.cuda.is_available() else 'cpu'}")
    MODEL_RESULTS[spec["name"]] = F.run_model(spec, CFG, OUT, REPO, log=stamp)
    F.zip_model(OUT, spec["name"], DEST, intermediate=CFG["zip_intermediate"], log=stamp)
if not CFG["models"]:
    stamp("RUN_MODEL = None: hanya ringkasan gabungan (sel [10])")

In [ ]:
# [10] RINGKASAN GABUNGAN: kriteria H1–H7 per model, Holm untuk H3, pooling KITTI 3 seed, latensi, plot Pareto
import matplotlib.pyplot as plt, zipfile
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 60)
mg = MERGE_GLOB if "CFG_OVERRIDE" not in globals() else CFG.get("merge_glob")
M = F.merge(OUT, ALL_MODELS, mg, log=stamp)
if len(M["done"]) < len(ALL_MODELS):
    print(f"⚠ Baru {len(M['done'])}/{len(ALL_MODELS)} model: H3 (Holm 4 model) dan pooling KITTI belum final.")
print("\nKriteria & uji utama per model (mAP_small, selisih [CI 95%]):\n" + M["TAB"].T.to_string())
print("\nCOCO lengkap (TensorRT FP16):\n" + M["FULL"][["model", "mAP50-95", "mAP50", "mAP_small", "mAP_medium", "mAP_large", "AR_small",
                                                    "latensi_ms", "aktivasi_P2"]].round(4).to_string())
print("\nH3 / perilaku skor (Holm hanya untuk baris H3 gate):\n" + M["H3"].round(4).to_string(index=False))
if len(M["POOL"]):
    print("\nKITTI — selisih rata-rata 3 seed per resample (test sama; CI 95%):\n" + M["POOL"].round(4).to_string(index=False))
print("\nLatensi TensorRT FP16 (ms) — semua build semua model:\n" + M["LATP"].round(3).to_string())
print("\n[4B] Latensi ujung-ke-ujung & H6 e2e per model:\n" + M["TAB"][[c for c in M["TAB"].columns if "e2e" in c or "sepass" in c]].T.to_string())
fig = F.pareto_plot(OUT, M["R"], M["FULL"], OUT / "pareto.png"); plt.show()
for k in ("TAB", "FULL", "H3", "POOL", "LATP"):
    if len(M[k]):
        M[k].to_csv(OUT / f"ringkasan_{k.lower()}.csv")
PROV.update(finished_utc=datetime.datetime.utcnow().isoformat(timespec="seconds"), wall_min=round((time.perf_counter() - T_START) / 60, 1),
            models_done=M["done"])
(OUT / "provenance.json").write_text(json.dumps(PROV, indent=1, default=str))
arc = (Path("/kaggle/working") if "CFG_OVERRIDE" not in globals() else OUT.parent) / f"tahap4b_ringkasan_{BASELINE_CKPT[:-3]}.zip"
with zipfile.ZipFile(arc, "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.glob("*"):
        if f.is_file():
            z.write(f, f.name)
stamp(f"📦 {arc} ({arc.stat().st_size / 1e6:.2f} MB)")